# `source_final.ipynb` — finalized pipeline (incremental)

This notebook is the **production** version of the sources pipeline.

Rule: we only copy stages into here once they were validated in `sources_test.ipynb`.

**Status**
- ✅ Stage B (Chapter Blueprint) finalized: `coverage_v1` (wins vs baseline)
- ✅ Stage C scoring finalized: `w_embed_max=0.0`, `w_embed=0.7`, `cite_weight=0.08`
- ✅ Stage C.3 finalized: LLM rerank **within Stage C top-50** + calibrated scoring + deterministic tie-break (model: `gpt-5-nano`)
- ✅ Stage D enabled: TF-IDF MMR selection for diverse top-20 (can be disabled)
- ✅ Stage A/C.2 included: OpenAlex + Semantic Scholar fetch; embeddings + TF-IDF scoring


In [1]:
from __future__ import annotations

import os
import json
import re
import hashlib
import time
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

from pydantic import BaseModel, Field

# OpenAI / Agents SDK (used for schema-validated JSON output)
from openai import OpenAI
try:
    from agents import Agent, Runner, ModelSettings
except ModuleNotFoundError as e:
    raise RuntimeError("Missing dependency: `openai-agents`. Install with: `pip install openai-agents` and restart the kernel.") from e

def require_python_packages() -> None:
    import importlib.util
    required = [
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("sklearn", "scikit-learn"),
        ("requests", "requests"),
    ]
    missing = [(mod, pkg) for mod, pkg in required if importlib.util.find_spec(mod) is None]
    if missing:
        mods = ", ".join([m for m, _ in missing])
        pkgs = " ".join(sorted({p for _, p in missing}))
        raise RuntimeError(f"Missing python packages: {mods}\nInstall with: `pip install {pkgs}`\nThen restart the notebook kernel.")

require_python_packages()


# -----------------------------
# Repo/workspace paths
# -----------------------------
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "package.json").exists() and (parent / "README.md").exists():
            return parent
    return p


REPO_ROOT = find_repo_root()
SOURCES_WORKSPACE_DIR = REPO_ROOT / "sources_workspace"


# -----------------------------
# Minimal .env loader (notebook convenience)
# -----------------------------
def load_dotenv_minimal(path: str = ".env") -> None:
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):].strip()
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k and (k not in os.environ):
            os.environ[k] = v


load_dotenv_minimal(str(REPO_ROOT / ".env"))

if not os.getenv("OPENAI_API_KEY", "").strip():
    raise RuntimeError("Missing OPENAI_API_KEY. Add it to your environment or .env file.")

client = OpenAI()  # validates credentials on first real request

try:
    from IPython.display import display  # type: ignore
except Exception:  # pragma: no cover
    def display(x):  # type: ignore
        print(x)


# -----------------------------
# Output dirs (cache)
# -----------------------------
OUT_DIR = SOURCES_WORKSPACE_DIR / "final_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)
STAGEB_DIR = OUT_DIR / "stageB_blueprints"
STAGEB_DIR.mkdir(parents=True, exist_ok=True)

# Additional pipeline dirs
STAGEA_DIR = OUT_DIR / "stageA"
STAGEA_DIR.mkdir(parents=True, exist_ok=True)
STAGEC_DIR = OUT_DIR / "stageC"
STAGEC_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Run id (used for output file names; caches are stable across runs)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# -----------------------------
# Finalized decision from Stage B A/B testing
# -----------------------------
STAGEB_FINAL_VARIANT = "coverage_v1"  # ✅ finalized winner

# Model pricing (USD per 1M tokens) — keep updated if pricing changes.
MODEL_PRICES_USD_PER_1M = {
    "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
    "text-embedding-3-small": {"input": 0.02, "cached": 0.0, "output": 0.0},
}



# -----------------------------
# Notebook print helpers
# -----------------------------

def _fmt_int(x: object) -> str:
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)


def _fmt_usd(x: object) -> str:
    try:
        return f"${float(x):.4f}"
    except Exception:
        return str(x)


def _fmt_pct(x: object) -> str:
    try:
        return f"{100.0*float(x):.1f}%"
    except Exception:
        return str(x)


def print_section(title: str) -> None:
    t = str(title).strip()
    print("\n" + "=" * 72)
    print(t)
    print("=" * 72)


def print_kv(d: Dict[str, Any], *, sort: bool = False) -> None:
    items = list(d.items())
    if sort:
        items.sort(key=lambda kv: str(kv[0]))
    w = max((len(str(k)) for k, _ in items), default=0)
    for k, v in items:
        print(f"{str(k).ljust(w)} : {v}")


def print_table(rows: List[Dict[str, Any]], columns: List[str], *, max_rows: int = 50) -> None:
    rows = list(rows or [])
    if not rows:
        print("(no rows)")
        return
    cols = list(columns)
    rows2 = rows[: int(max_rows)]

    def cell(r, c):
        v = r.get(c, "")
        v = "" if v is None else str(v)
        v = v.replace("\n", " ").strip()
        return v

    widths = {c: max(len(c), max((len(cell(r, c)) for r in rows2), default=0)) for c in cols}
    widths = {c: min(w, 60) for c, w in widths.items()}

    def trunc(s, w):
        return s if len(s) <= w else (s[: max(0, w - 1)] + "…")

    header = " | ".join(trunc(c, widths[c]).ljust(widths[c]) for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in rows2:
        print(" | ".join(trunc(cell(r, c), widths[c]).ljust(widths[c]) for c in cols))
    if len(rows) > len(rows2):
        print(f"… {len(rows) - len(rows2)} more")


# -----------------------------
# Plot helpers
# -----------------------------
PLOTS_ENABLED = True
PLOT_MAX_CHAPTERS = 6
PLOT_MAX_POINTS = 2500


def short_label(s: object, max_len: int = 28) -> str:
    s2 = str(s or "")
    return s2 if len(s2) <= int(max_len) else (s2[: int(max_len) - 1] + "…")


def get_plt():
    try:
        import matplotlib.pyplot as plt  # type: ignore
        try:
            plt.style.use("seaborn-v0_8-whitegrid")
        except Exception:
            pass
        return plt
    except Exception as e:
        print(f"[plots] matplotlib not available: {e}")
        return None
# Stage B model (only a few calls per chapter, but we still cost-track)
BLUEPRINT_MODEL = "gpt-5-mini"

# Rebuild controls
FORCE_REBUILD_BLUEPRINTS = False

# -----------------------------
# Chapter specs (edit/replace these to use the pipeline on new chapters)
# -----------------------------


# -----------------------------
# Universal chapter inputs
# -----------------------------

def _slugify(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9]+", "_", str(s or "").strip().lower()).strip("_")
    s = re.sub(r"_+", "_", s)
    return s[:48] if len(s) > 48 else s


def normalize_chapters(chapters: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # Accepts minimal chapter specs and normalizes them.
    # Minimum required: title + original_text

    out: List[Dict[str, Any]] = []
    used_ids: set[str] = set()

    for i, ch in enumerate(chapters or []):
        if not isinstance(ch, dict):
            raise TypeError(f"CHAPTERS[{i}] must be a dict")

        title = str(ch.get("title", "") or "").strip() or f"Chapter {i+1}"

        original_text = ch.get("original_text", "")
        if isinstance(original_text, (list, tuple)):
            original_text = " ".join(str(x) for x in original_text)
        original_text = str(original_text or "").strip()

        lang = str(ch.get("original_text_language") or ch.get("language") or "en").strip().lower() or "en"

        chapter_id = str(ch.get("chapter_id") or "").strip()
        if not chapter_id:
            base = _slugify(title) or f"chapter_{i+1}"
            h = hashlib.sha1((title + "|" + original_text[:200]).encode("utf-8")).hexdigest()[:8]
            chapter_id = f"{base}_{h}"

        base_id = chapter_id
        j = 2
        while chapter_id in used_ids:
            chapter_id = f"{base_id}_{j}"
            j += 1
        used_ids.add(chapter_id)

        out.append({
            **ch,
            "chapter_id": chapter_id,
            "title": title,
            "original_text_language": lang,
            "original_text": original_text,
        })

    return out
CHAPTERS: List[Dict[str, Any]] = [
{
  "chapter_id": "fall_of_rome_economy",
  "title": "Analyse: Ökonomische Befunde im (west-)römischen Reich und ihre Beziehungen zu Politik, Gesellschaft und Militär",
  "original_text_language": "de",
  "original_text": (
        "Dieses Kapitel untersucht wirtschaftliche Faktoren im (west-)römischen Reich der Spätantike und stellt die Befunde so dar", 
        "dass sie für die Erklärung des Zerfalls- bzw. Transformationsprozesses nutzbar sind. Es arbeitet wirtschaftliche Mechanismen, "
        "Strukturveränderungen und Rahmenbedingungen heraus und ordnet sie als Befundbestand, wobei Befund, Deutung und Reichweite der "
        "Schlussfolgerungen getrennt ausgewiesen werden. Anschließend wird geprüft, in welcher Weise die ökonomischen Befunde mit politischen, "
        "sozialen und militärischen Entwicklungen im (west-)römischen Reich verknüpft werden können: welche Beziehungen das Material stützt, wo" 
        "Wechselwirkungen plausibel sind und wo Verbindungen unsicher bleiben. Das Kapitel endet mit einer zusammenfassenden Einordnung, auf welcher" 
        "Ebene wirtschaftliche Faktoren im Gesamtprozess erklärungsrelevant erscheinen."

  ),
}

]

CHAPTERS = normalize_chapters(CHAPTERS)

print_section("Config")
print_kv({
    "stageB_variant": STAGEB_FINAL_VARIANT,
    "run_id": RUN_ID,
    "chapters": len(CHAPTERS),
})

print("\nModel prices (USD per 1M tokens) — verify periodically:")
for m, p in MODEL_PRICES_USD_PER_1M.items():
    print(f"- {m}: in={p.get('input')} cached={p.get('cached')} out={p.get('output')}")

rows = []
for ch in CHAPTERS:
    t = ch.get('title','')
    txt = ch.get('original_text','')
    rows.append({
        'chapter_id': ch.get('chapter_id'),
        'lang': ch.get('original_text_language'),
        'text_chars': _fmt_int(len(str(txt or ''))),
        'title': (t[:80] + ('…' if len(t) > 80 else '')),
    })
print("\nChapters:")
print_table(rows, columns=['chapter_id','lang','text_chars','title'], max_rows=200)



Config
stageB_variant : coverage_v1
run_id         : 20260210_131329
chapters       : 1

Model prices (USD per 1M tokens) — verify periodically:
- gpt-5-nano: in=0.05 cached=0.005 out=0.4
- gpt-5-mini: in=0.25 cached=0.025 out=2.0
- text-embedding-3-small: in=0.02 cached=0.0 out=0.0

Chapters:
chapter_id           | lang | text_chars | title                                                       
---------------------+------+------------+-------------------------------------------------------------
fall_of_rome_economy | de   | 859        | Analyse: Ökonomische Befunde im (west-)römischen Reich und …


## Stage B (finalized): Chapter Blueprint generation (`coverage_v1`)

This is the only Stage B variant that goes into `source_final.ipynb`.


In [2]:
# -----------------------------
# Stage B: build blueprints (cached)
# Prompts: blueprint_v3, year_bounds_v2, query_plan_service_agents_v1 (OpenAlex + Semantic Scholar), gapfill_v2
# -----------------------------

from __future__ import annotations

import json
import re
import asyncio
from collections import Counter
from datetime import datetime, timezone
from typing import Any, Dict, List, Literal, Optional, Tuple

from pydantic import BaseModel, ConfigDict, Field
from agents import Agent, AgentOutputSchema, ModelSettings, Runner

CURRENT_YEAR = int(datetime.now(timezone.utc).year)

STAGEB_FINAL_VARIANT = "universal_prompts_v3"
STAGEB_PROMPT_VERSION = "universal_prompts_v3_2026-02-09_fix1"

STAGEB_FORCE_ALL = bool(globals().get("FORCE_REBUILD_BLUEPRINTS", False))
STAGEB_LOG_CACHE_HITS = True
STAGEB_LOG_CACHE_INVALID = True

STAGEB_QUERY_TARGET = 100
STAGEB_QUERY_PLAN_VARIANT = "service_agents_v1"  # per-API LLM agents generate queries (no deterministic query generation)
STAGEB_QUERY_PLAN_PROMPT_VERSION = "service_agents_v1_2026-02-10"

# Query-plan leniency + cost controls (avoid expensive retries)
STAGEB_QUERY_PLAN_MAX_ATTEMPTS = 2  # usually 1; retries only if output is unusable
STAGEB_QUERY_PLAN_TRUNCATE_TO_TARGET = True  # never exceed per-service targets
STAGEB_QUERY_PLAN_DEDUPE = True  # drop exact duplicate query strings per service
STAGEB_QUERY_PLAN_MIN_TOTAL_QUERIES = 10  # raise if we end up with fewer than this overall
STAGEB_DEFAULT_SOFT_MIN_YEAR = 1800
STAGEB_DEFAULT_NO_YEAR_SHARE = 0.20

# Hard timeout for LLM calls, so a stuck request cannot stall the notebook indefinitely.
STAGEB_AGENT_TIMEOUT_SEC = 600

STAGEB_BLUEPRINT_DIR = STAGEB_DIR / "blueprint_v3"
STAGEB_YEAR_BOUNDS_DIR = STAGEB_DIR / "year_bounds_v2"
STAGEB_QUERY_PLAN_DIR = STAGEB_DIR / "query_plan_service_agents_v1"
STAGEB_GAPFILL_DIR = STAGEB_DIR / "gapfill_v2"
for _d in [STAGEB_BLUEPRINT_DIR, STAGEB_YEAR_BOUNDS_DIR, STAGEB_QUERY_PLAN_DIR, STAGEB_GAPFILL_DIR]:
    _d.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Pydantic models (new prompt schemas)
# -----------------------------

class ScopeObj(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    in_: List[str] = Field(default_factory=list, alias="in")
    out: List[str] = Field(default_factory=list)


class AnchorsLang(BaseModel):
    en: List[str] = Field(default_factory=list)
    de: List[str] = Field(default_factory=list)


class BlueprintAnchors(BaseModel):
    disambiguators: AnchorsLang
    evidence_proxies: AnchorsLang
    mechanisms: AnchorsLang
    high_precision: AnchorsLang


class BlueprintFacet(BaseModel):
    facet_id: str
    label_en: str
    label_de: str
    terms_en: List[str] = Field(default_factory=list)
    terms_de: List[str] = Field(default_factory=list)
    evidence_proxies_en: List[str] = Field(default_factory=list)
    evidence_proxies_de: List[str] = Field(default_factory=list)
    safe_negatives: List[str] = Field(default_factory=list)
    risky_negatives: List[str] = Field(default_factory=list)


class NegativePool(BaseModel):
    en: List[str] = Field(default_factory=list)
    de: List[str] = Field(default_factory=list)


class BlueprintV3(BaseModel):
    chapter_id: str
    scope: ScopeObj
    must_cover: List[str] = Field(default_factory=list)
    should_cover: List[str] = Field(default_factory=list)
    must_avoid: List[str] = Field(default_factory=list)
    anchors: BlueprintAnchors
    facets: List[BlueprintFacet] = Field(default_factory=list)
    negative_pool: NegativePool


class YearBoundsV2(BaseModel):
    chapter_id: str
    soft_min_year: int
    soft_max_year: int
    no_year_share: float = Field(..., ge=0.0, le=0.40)
    disambiguator_strength: Literal["strong", "medium"]
    rationale_en: str = ""
    rationale_de: str = ""


class QueryFiltersV2(BaseModel):
    year_min: Optional[int] = None
    year_max: Optional[int] = None
    language: Optional[Literal["en", "de"]] = None
    concept_ids: List[str] = Field(default_factory=list)


class QueryObjV2(BaseModel):
    id: str
    service: Literal["openalex", "semanticscholar"]
    kind: Literal["anchor", "facet", "proxy", "authority"]
    language: Literal["en", "de"]
    cap: int
    use_no_year: bool = False
    facet_id: str = ""
    query: str
    filters: QueryFiltersV2 = Field(default_factory=QueryFiltersV2)
    notes: str = ""


class QueryPlanGuaranteesV2(BaseModel):
    context_gate_coverage_share: float = 0.0
    facet_coverage: Dict[str, int] = Field(default_factory=dict)
    dedupe_ok: bool = True


class QueryPlanV2(BaseModel):
    chapter_id: str
    queries: List[QueryObjV2] = Field(default_factory=list)
    guarantees: QueryPlanGuaranteesV2 = Field(default_factory=QueryPlanGuaranteesV2)


# Draft query objects for LLM-only query planning (service-specific agents)
class QueryObjDraftV1(BaseModel):
    # Intentionally lenient (we normalize + clamp after model parsing)
    kind: str = "facet"
    language: str = "en"
    cap: int = 0
    use_no_year: bool = False
    facet_id: str = ""
    query: str = ""
    filters: QueryFiltersV2 = Field(default_factory=QueryFiltersV2)
    notes: str = ""


class OpenAlexQueryPlanDraftV1(BaseModel):
    chapter_id: str
    service: Literal["openalex"] = "openalex"
    queries: List[QueryObjDraftV1] = Field(default_factory=list)


class SemanticScholarQueryPlanDraftV1(BaseModel):
    chapter_id: str
    service: Literal["semanticscholar"] = "semanticscholar"
    queries: List[QueryObjDraftV1] = Field(default_factory=list)


class GapFoundV2(BaseModel):
    gap: str
    evidence: str


class GapChangeV2(BaseModel):
    action: Literal["add", "replace"]
    replace_query_id: str = ""
    new_query: QueryObjV2
    targets: List[str] = Field(default_factory=list)
    reason: str = ""


class GapFillV2(BaseModel):
    chapter_id: str
    gaps_found: List[GapFoundV2] = Field(default_factory=list)
    changes: List[GapChangeV2] = Field(default_factory=list)


# -----------------------------
# Compatibility model (consumed by Stage C/C3)
# -----------------------------

class ChapterBlueprint(BaseModel):
    chapter_id: str
    language: str = Field("en")
    scope_statement: str
    must_cover: List[str]
    should_cover: List[str]
    must_avoid: List[str]
    main_query: str
    facet_queries: List[str]
    keywords: List[str]
    key_concepts: List[str]
    preferred_source_types: Optional[List[str]] = None
    negative_query_terms: Optional[List[str]] = None
    scoring_guidance: str
    notes: Optional[str] = None
    anchors_en: List[str] = Field(default_factory=list)
    anchors_de: List[str] = Field(default_factory=list)
    facet_blocks: List[Dict[str, Any]] = Field(default_factory=list)
    year_bounds: Dict[str, Any] = Field(default_factory=dict)
    query_plan_size: int = 0
    query_plan_path: Optional[str] = None


# -----------------------------
# Cost helpers (used later, keep stable)
# -----------------------------

def price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})


def cost_from_usage(usage, model: str) -> dict:
    prices = price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)
        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)
    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }


def _zero_usage() -> Dict[str, Any]:
    return {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}


def _acc_usage(total: Dict[str, Any], part: Dict[str, Any]) -> None:
    for k in ["requests", "input_tokens", "cached_input_tokens", "output_tokens", "cost_usd"]:
        total[k] = total.get(k, 0) + (part.get(k, 0) if isinstance(part, dict) else 0)


# -----------------------------
# Normalization helpers
# -----------------------------

_S2_FORBIDDEN_RE = re.compile(r"(?i)\b(AND|OR|NOT)\b")


def dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out: List[str] = []
    for x in items or []:
        s = str(x or "").strip()
        if not s:
            continue
        k = s.lower()
        if k in seen:
            continue
        seen.add(k)
        out.append(s)
    return out


def _clean_str(x: Any) -> str:
    s = str(x or "").strip()
    return re.sub(r"\s+", " ", s)


def _slugify(x: str, max_len: int = 40) -> str:
    s = _clean_str(x).lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    s = re.sub(r"_+", "_", s)
    return (s[: int(max_len)] if s else "item")


def _uniq(xs: List[Any], *, max_items: Optional[int] = None) -> List[str]:
    out, seen = [], set()
    for x in xs or []:
        s = _clean_str(x)
        if not s:
            continue
        k = s.lower()
        if k in seen:
            continue
        seen.add(k)
        out.append(s)
        if max_items is not None and len(out) >= int(max_items):
            break
    return out


def _ensure_len(xs: List[str], *, min_len: int, max_len: int, fallback: List[str]) -> List[str]:
    out = _uniq(xs, max_items=max_len)
    for f in _uniq(fallback):
        if len(out) >= int(min_len):
            break
        if f.lower() not in {x.lower() for x in out}:
            out.append(f)
    return out[: int(max_len)]


def _extract_terms(text: str, *, max_terms: int = 24) -> List[str]:
    s = _clean_str(text).lower()
    s = re.sub(r"[^\w\s]", " ", s)
    toks = [t for t in s.split() if len(t) >= 3]
    stop = {
        "the",
        "and",
        "for",
        "with",
        "from",
        "this",
        "that",
        "eine",
        "einer",
        "und",
        "der",
        "die",
        "das",
    }
    out = []
    for t in toks:
        if t in stop or t in out:
            continue
        out.append(t)
        if len(out) >= int(max_terms):
            break
    return out


def _q(x: str) -> str:
    s = _clean_str(x).replace('"', "").replace(",", " ").replace(":", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return f'"{s}"' if (s and " " in s) else s


def _norm_for_match(s: str) -> str:
    s = _clean_str(s).lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def _contains_term(text: str, term: str) -> bool:
    t = _norm_for_match(term)
    if not t:
        return False
    return t in _norm_for_match(text)


def _sanitize_semanticscholar_query(q: str) -> str:
    s = _clean_str(q)
    s = _S2_FORBIDDEN_RE.sub(" ", s)
    s = re.sub(r"[][(){}|+*^~]", " ", s)
    s = s.replace(":", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s[:240].strip()


def _sanitize_openalex_query(q: str) -> str:
    s = _clean_str(q)
    s = s.replace(",", " ").replace(":", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _cap(service: str, val: int) -> int:
    x = int(val or 0)
    if service == "openalex":
        return max(60, min(120, x if x else 100))
    return max(40, min(120, x if x else 80))


def _valid_concept_ids(ids: List[str]) -> List[str]:
    out = []
    for x in ids or []:
        s = str(x or "").strip()
        if re.fullmatch(r"C\d+", s):
            out.append(s)
    return dedupe_preserve_order(out)[:10]


# -----------------------------
# Deterministic fallbacks (only used if LLM fails)
# -----------------------------

def _det_blueprint_v3(ch: Dict[str, Any]) -> BlueprintV3:
    cid = str(ch.get("chapter_id") or "chapter")
    title = _clean_str(ch.get("title") or cid)
    text = _clean_str(ch.get("original_text") or "")
    terms = _extract_terms(f"{title} {text}", max_terms=60)

    in_scope = _ensure_len([title] + terms[:8], min_len=3, max_len=10, fallback=[title])
    out_scope = _ensure_len([], min_len=2, max_len=8, fallback=["unrelated domains", "popular media"])

    must_cover = _ensure_len(_extract_terms(text, max_terms=18), min_len=8, max_len=10, fallback=terms[:10] + [title])
    should_cover = _ensure_len(_extract_terms(text, max_terms=26)[8:], min_len=6, max_len=10, fallback=terms[8:18])
    must_avoid = _ensure_len([], min_len=6, max_len=10, fallback=["modern policy", "contemporary data", "unrelated technology"])

    # Best-effort anchors: treat early terms as disambiguators, later as mechanisms/proxies.
    dis_en = _ensure_len([title] + terms[:5], min_len=4, max_len=14, fallback=[title])
    dis_de = _ensure_len(terms[:6], min_len=4, max_len=14, fallback=dis_en[:6])
    ev_en = _ensure_len(terms[6:18], min_len=4, max_len=18, fallback=["evidence", "dataset", "archaeology"])
    ev_de = _ensure_len(terms[6:18], min_len=4, max_len=18, fallback=ev_en[:8])
    mech_en = _ensure_len(terms[18:32], min_len=4, max_len=18, fallback=["mechanism", "drivers", "change"])
    mech_de = _ensure_len(terms[18:32], min_len=4, max_len=18, fallback=mech_en[:8])
    hp_en = _ensure_len(terms[32:44], min_len=3, max_len=16, fallback=[title])
    hp_de = _ensure_len(terms[32:44], min_len=3, max_len=16, fallback=hp_en[:8])

    facets: List[BlueprintFacet] = []
    for i in range(8):
        seed = must_cover[i % len(must_cover)] if must_cover else title
        f_terms = _extract_terms(seed, max_terms=16)
        facets.append(
            BlueprintFacet(
                facet_id=f"facet_{i+1}_{_slugify(f_terms[0] if f_terms else f'part_{i+1}', max_len=16)}",
                label_en=" ".join(seed.split()[:6]) or f"Facet {i+1}",
                label_de=" ".join(seed.split()[:6]) or f"Facette {i+1}",
                terms_en=_ensure_len(f_terms[:10], min_len=6, max_len=10, fallback=mech_en),
                terms_de=_ensure_len(f_terms[:10], min_len=6, max_len=10, fallback=mech_de),
                evidence_proxies_en=_ensure_len(ev_en[:6], min_len=3, max_len=6, fallback=ev_en),
                evidence_proxies_de=_ensure_len(ev_de[:6], min_len=3, max_len=6, fallback=ev_de),
                safe_negatives=[],
                risky_negatives=[],
            )
        )

    return BlueprintV3(
        chapter_id=cid,
        scope=ScopeObj(in_=in_scope, out=out_scope),
        must_cover=must_cover[:8],
        should_cover=should_cover[:6],
        must_avoid=must_avoid[:6],
        anchors=BlueprintAnchors(
            disambiguators=AnchorsLang(en=dis_en, de=dis_de),
            evidence_proxies=AnchorsLang(en=ev_en, de=ev_de),
            mechanisms=AnchorsLang(en=mech_en, de=mech_de),
            high_precision=AnchorsLang(en=hp_en, de=hp_de),
        ),
        facets=facets,
        negative_pool=NegativePool(
            en=_ensure_len([], min_len=8, max_len=30, fallback=must_avoid + ["modern", "contemporary"]),
            de=_ensure_len([], min_len=8, max_len=30, fallback=["modern", "zeitgenössisch"] + must_avoid),
        ),
    )


def _det_year_bounds_v2(chapter_id: str) -> YearBoundsV2:
    return YearBoundsV2(
        chapter_id=chapter_id,
        soft_min_year=int(STAGEB_DEFAULT_SOFT_MIN_YEAR),
        soft_max_year=int(CURRENT_YEAR),
        no_year_share=float(STAGEB_DEFAULT_NO_YEAR_SHARE),
        disambiguator_strength="strong",
        rationale_en="Fallback: broad year window + strong disambiguators to control drift.",
        rationale_de="Fallback: breites Jahresfenster + starke Disambiguatoren gegen Topic-Drift.",
    )


def _oa_expr(dis: List[str], mech: List[str], ev: List[str], neg: List[str]) -> str:
    parts = []
    if dis:
        parts.append(_q(dis[0]))
    if mech:
        parts.append(_q(mech[0]))
    if ev:
        parts.append(_q(ev[0]))
    expr = " AND ".join([p for p in parts if p])
    if neg:
        n = " OR ".join([_q(x) for x in _uniq(neg, max_items=2) if _q(x)])
        if n:
            expr = f"{expr} AND NOT ({n})" if expr else f"NOT ({n})"
    return _clean_str(expr)


def _s2_expr(dis: List[str], mech: List[str], ev: List[str]) -> str:
    parts = []
    if dis:
        parts.append(_q(dis[0]))
    if mech:
        parts.append(_q(mech[0]))
    if ev:
        parts.append(_q(ev[0]))
    return _sanitize_semanticscholar_query(" ".join([p for p in parts if p]))


def _compute_guarantees(bp: BlueprintV3, y: YearBoundsV2, plan: QueryPlanV2) -> QueryPlanV2:
    facets = {f.facet_id for f in (bp.facets or [])}
    facet_counts: Dict[str, int] = {fid: 0 for fid in facets}

    dis_en = bp.anchors.disambiguators.en or []
    dis_de = bp.anchors.disambiguators.de or []

    covered = 0
    for q in plan.queries or []:
        if q.facet_id and q.facet_id in facet_counts:
            facet_counts[q.facet_id] += 1

        dis_terms = (dis_en if q.language == "en" else dis_de) or dis_en or dis_de
        if any(_contains_term(q.query, t) for t in (dis_terms or [])):
            covered += 1

    share = (covered / max(1, len(plan.queries))) if plan.queries else 0.0
    dedupe_ok = len({_norm_for_match(q.query) for q in plan.queries}) == len(plan.queries)
    plan.guarantees = QueryPlanGuaranteesV2(
        context_gate_coverage_share=float(share),
        facet_coverage={k: int(v) for k, v in sorted(facet_counts.items())},
        dedupe_ok=bool(dedupe_ok),
    )
    return plan


def _det_query_plan_v2(
    chapter_id: str,
    bp: BlueprintV3,
    y: YearBoundsV2,
    *,
    total: int = 100,
    services: Dict[str, int] | None = None,
    languages: Dict[str, int] | None = None,
    kinds: Dict[str, int] | None = None,
) -> QueryPlanV2:
    services = services or {"openalex": 50, "semanticscholar": 50}
    languages = languages or {"en": 50, "de": 50}
    kinds = kinds or {"anchor": 50, "facet": 30, "proxy": 15, "authority": 5}

    facets = list(bp.facets or [])
    facet_ids = [f.facet_id for f in facets] or ["facet_1"]

    svc_slots = (["openalex"] * int(services.get("openalex", 0))) + (["semanticscholar"] * int(services.get("semanticscholar", 0)))
    lang_slots = (["en"] * int(languages.get("en", 0))) + (["de"] * int(languages.get("de", 0)))
    kind_slots: List[str] = []
    for k, n in (kinds or {}).items():
        kind_slots.extend([k] * int(n))

    while len(svc_slots) < int(total):
        svc_slots.append("openalex" if len(svc_slots) % 2 == 0 else "semanticscholar")
    while len(lang_slots) < int(total):
        lang_slots.append("en" if len(lang_slots) % 2 == 0 else "de")
    while len(kind_slots) < int(total):
        kind_slots.append("facet")
    svc_slots = svc_slots[: int(total)]
    lang_slots = lang_slots[: int(total)]
    kind_slots = kind_slots[: int(total)]

    def _spread(xs: List[str]) -> List[str]:
        out: List[str] = []
        buckets = {k: [v for v in xs if v == k] for k in sorted(set(xs))}
        keys = sorted(buckets.keys())
        i = 0
        while len(out) < len(xs):
            k = keys[i % len(keys)]
            if buckets[k]:
                out.append(buckets[k].pop())
            i += 1
        return out

    svc_slots = _spread(svc_slots)
    lang_slots = _spread(lang_slots)
    kind_slots = _spread(kind_slots)

    want_no_year = int(round(float(y.no_year_share) * float(total)))
    no_year_idx = set(range(0, want_no_year))

    queries: List[QueryObjV2] = []
    triple_counts: Counter[Tuple[str, str, str]] = Counter()

    for i in range(int(total)):
        service = svc_slots[i]
        lang = lang_slots[i]
        kind = kind_slots[i]
        use_no_year = i in no_year_idx

        facet_id = "" if kind == "authority" else facet_ids[i % len(facet_ids)]
        facet = next((f for f in facets if f.facet_id == facet_id), None)

        dis = (bp.anchors.disambiguators.en if lang == "en" else bp.anchors.disambiguators.de) or bp.anchors.disambiguators.en or bp.anchors.disambiguators.de
        mech_pool = (bp.anchors.mechanisms.en if lang == "en" else bp.anchors.mechanisms.de) or bp.anchors.mechanisms.en or bp.anchors.mechanisms.de
        ev_pool = (bp.anchors.evidence_proxies.en if lang == "en" else bp.anchors.evidence_proxies.de) or bp.anchors.evidence_proxies.en or bp.anchors.evidence_proxies.de
        facet_terms = (facet.terms_en if (facet and lang == "en") else (facet.terms_de if facet else [])) if kind != "authority" else []
        facet_ev = (facet.evidence_proxies_en if (facet and lang == "en") else (facet.evidence_proxies_de if facet else [])) if kind != "authority" else []

        d = dis[i % len(dis)] if dis else ""
        m_pool = facet_terms or mech_pool
        e_pool = facet_ev or ev_pool
        m = (m_pool[(i * 3) % len(m_pool)] if m_pool else "")
        e = (e_pool[(i * 5) % len(e_pool)] if e_pool else "")

        triple = (_norm_for_match(d), _norm_for_match(e), _norm_for_match(m))
        if all(triple):
            while triple_counts[triple] >= 2 and m_pool and e_pool:
                m = m_pool[(i * 3 + triple_counts[triple] + 1) % len(m_pool)]
                e = e_pool[(i * 5 + triple_counts[triple] + 1) % len(e_pool)]
                triple = (_norm_for_match(d), _norm_for_match(e), _norm_for_match(m))
        triple_counts[triple] += 1

        qtxt = _oa_expr([d], [m], [e], []) if service == "openalex" else _s2_expr([d], [m], [e])

        cap = _cap(service, 100 if kind in ("anchor", "authority") else (80 if kind == "facet" else 60))
        q = QueryObjV2(
            id=f"Q{i+1:03d}",
            service=service,  # type: ignore[arg-type]
            kind=kind,  # type: ignore[arg-type]
            language=lang,  # type: ignore[arg-type]
            cap=cap,
            use_no_year=bool(use_no_year),
            facet_id=str(facet_id or ""),
            query=qtxt,
            filters=QueryFiltersV2(
                year_min=None if use_no_year else int(y.soft_min_year),
                year_max=None if use_no_year else int(y.soft_max_year),
                language=lang,  # type: ignore[arg-type]
                concept_ids=[],
            ),
            notes=f"deterministic {kind} query",
        )
        queries.append(q)

    plan = QueryPlanV2(chapter_id=chapter_id, queries=queries)
    return _compute_guarantees(bp, y, plan)


def _enforce_query_requirements(bp: BlueprintV3, y: YearBoundsV2, q: QueryObjV2, stats: Optional[Dict[str, int]] = None, *, enforce_disambiguator: bool) -> QueryObjV2:
    st: Optional[Dict[str, int]] = stats if isinstance(stats, dict) else None
    lang = q.language
    facet = next((f for f in (bp.facets or []) if f.facet_id == q.facet_id), None) if q.facet_id else None

    dis_list = (bp.anchors.disambiguators.en if lang == "en" else bp.anchors.disambiguators.de) or bp.anchors.disambiguators.en or bp.anchors.disambiguators.de
    mech_list = (bp.anchors.mechanisms.en if lang == "en" else bp.anchors.mechanisms.de) or bp.anchors.mechanisms.en or bp.anchors.mechanisms.de
    ev_list = (bp.anchors.evidence_proxies.en if lang == "en" else bp.anchors.evidence_proxies.de) or bp.anchors.evidence_proxies.en or bp.anchors.evidence_proxies.de

    facet_terms = []
    facet_ev = []
    safe_negs = []
    if facet is not None:
        facet_terms = facet.terms_en if lang == "en" else facet.terms_de
        facet_ev = facet.evidence_proxies_en if lang == "en" else facet.evidence_proxies_de
        safe_negs = facet.safe_negatives or []

    m = re.match(r"^Q(\d+)$", str(q.id or "").strip())
    idx = (int(m.group(1)) - 1) if m else 0

    dis_term = dis_list[idx % len(dis_list)] if dis_list else ""
    mech_pool = facet_terms or mech_list
    ev_pool = facet_ev or ev_list
    mech_term = (mech_pool[(idx * 3) % len(mech_pool)] if mech_pool else "")
    ev_term = (ev_pool[(idx * 5) % len(ev_pool)] if ev_pool else "")

    needs_dis = bool(enforce_disambiguator) and bool(dis_term) and (not _contains_term(q.query, dis_term))
    needs_mech = bool(mech_term) and (not any(_contains_term(q.query, t) for t in ([mech_term] + mech_list[:3] + facet_terms[:3])))
    needs_ev = bool(ev_term) and (not any(_contains_term(q.query, t) for t in ([ev_term] + ev_list[:3] + facet_ev[:3])))

    add_terms = []
    if needs_dis:
        add_terms.append(dis_term)
    if needs_mech:
        add_terms.append(mech_term)
    if needs_ev:
        add_terms.append(ev_term)

    if q.service == "semanticscholar":
        before = str(q.query or "")
        q.query = _sanitize_semanticscholar_query(q.query)
        if st is not None and q.query != before:
            st["sanitized_s2"] = int(st.get("sanitized_s2", 0)) + 1
    else:
        q.query = _sanitize_openalex_query(q.query)

    if add_terms:
        if st is not None:
            if needs_dis:
                st["injected_dis"] = int(st.get("injected_dis", 0)) + 1
            if needs_mech:
                st["injected_mech"] = int(st.get("injected_mech", 0)) + 1
            if needs_ev:
                st["injected_ev"] = int(st.get("injected_ev", 0)) + 1
        if q.service == "openalex":
            base = _clean_str(q.query)
            for t in add_terms:
                qt = _q(t)
                if qt:
                    base = f"{base} AND {qt}" if base else qt
            if safe_negs:
                n = " OR ".join([_q(x) for x in _uniq(safe_negs, max_items=2) if _q(x)])
                if n and "NOT" not in base.upper():
                    base = f"{base} AND NOT ({n})"
            q.query = _clean_str(base)
        else:
            base = _sanitize_semanticscholar_query(q.query)
            for t in add_terms:
                qt = _q(t)
                if qt:
                    base = f"{base} {qt}" if base else qt
            q.query = _sanitize_semanticscholar_query(base)

    q.cap = _cap(q.service, q.cap)
    q.filters.concept_ids = _valid_concept_ids(q.filters.concept_ids)
    q.filters.language = q.language
    if q.use_no_year:
        q.filters.year_min = None
        q.filters.year_max = None
    else:
        q.filters.year_min = int(q.filters.year_min or y.soft_min_year)
        q.filters.year_max = int(q.filters.year_max or y.soft_max_year)
    return q


def _enforce_plan_requirements(bp: BlueprintV3, y: YearBoundsV2, plan: QueryPlanV2) -> QueryPlanV2:
    qs = list(plan.queries or [])
    if not qs:
        return _det_query_plan_v2(plan.chapter_id, bp, y, total=int(STAGEB_QUERY_TARGET))

    if y.disambiguator_strength == "strong":
        enforce_idx = set(range(len(qs)))
    else:
        want = int((0.70 * len(qs)) + 0.999)
        prio = [i for i, q in enumerate(qs) if q.kind in ("anchor", "facet")] + [i for i, q in enumerate(qs) if q.kind == "proxy"] + [i for i, q in enumerate(qs) if q.kind == "authority"]
        enforce_idx = set(prio[:want])

    stats: Dict[str, int] = {"changed_queries": 0, "sanitized_s2": 0, "injected_dis": 0, "injected_mech": 0, "injected_ev": 0}
    fixed: List[QueryObjV2] = []
    for i, q in enumerate(qs):
        before = str(q.query or "")
        q2 = _enforce_query_requirements(bp, y, q, stats, enforce_disambiguator=(i in enforce_idx))
        if q2.query != before:
            stats["changed_queries"] = int(stats.get("changed_queries", 0)) + 1
        fixed.append(q2)

    for i, q in enumerate(fixed, start=1):
        q.id = f"Q{i:03d}"

    plan.queries = fixed
    if any(int(stats.get(k, 0)) > 0 for k in ["changed_queries", "sanitized_s2", "injected_dis", "injected_mech", "injected_ev"]):
        cid = str(plan.chapter_id or "chapter")
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>    return _compute_guarantees(bp, y, plan)


# -----------------------------
# Prompt templates (must match user-provided prompts)
# -----------------------------

<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
PROMPT_BLUEPRINT_V3_USER = """Create a retrieval blueprint for a chapter/topic. This blueprint will be used to generate 100 API queries later.

INPUT (JSON):
{input_json}

TASK:
1) Extract the *topic scope* (what is in-scope / out-of-scope) as short bullets.
2) Produce MUST-COVER / SHOULD-COVER / MUST-AVOID lists.
3) Produce retrieval anchors in FOUR TYPES, each in EN and DE:
   A) DISAMBIGUATORS (time/place/field markers that narrow the corpus)
   B) EVIDENCE_PROXIES (primary data types, measurement methods, corpora, instruments)
   C) MECHANISMS_VARIABLES (the explanatory variables / mechanisms)
   D) HIGH-PRECISION TERMS (named corpora, technical terms, key phrases; topic-specific)
4) Produce FACETS (8–12) that partition the topic. Each facet must include:
   - label_en, label_de
   - terms_en (6–10), terms_de (6–10)
   - evidence_proxies_en (3–6), evidence_proxies_de (3–6)
   - safe_negatives (1–4), risky_negatives (0–3)
5) Produce an initial NEGATIVE TERM POOL (EN/DE) for common confounders *relevant to this topic*.

OBJECT BINDING RULE:
If a term could apply to many domains (e.g., "archaeology", "trade", "pollen analysis", "inflation"),
it must be paired with an object/setting gate (e.g., "<object> trade", "<object> pollen record").

CONSTRAINTS:
- Anchors must be retrieval-effective, not just conceptual. Avoid single-word generic anchors unless paired with disambiguators.
- Disambiguators must include time/place/field markers when the topic implies them.
- Evidence proxies must be concrete (data types, methods, corpora, instruments).
- Keep each string short (1–5 words), except named corpora can be longer.
- Output must be VALID JSON only, no markdown, no comments.
- CRITICAL: Create "context-gate" phrases inside anchors.disambiguators and anchors.high_precision.
  A context-gate phrase is a 2–6 word phrase that uniquely identifies the chapter's object/setting
  (e.g., named entity, place, population, time-period + entity, dataset name, canon label).
  At least 6 disambiguators.en and 6 disambiguators.de MUST be context-gates (not generic fields).
- Do NOT output bare discipline labels as disambiguators (e.g., "archaeology", "economic history").
  If you use a discipline/field marker, bind it to the object (e.g., "<object> archaeology").
- Disambiguators must include at least:
  (i) one time/period marker AND (ii) one place/setting/population marker AND (iii) one object marker
  (object marker = the core entity/phenomenon described in title/original_text).
  Each should be expressed as short phrases usable in search.
- High-precision terms: include at least 3 "hard anchors":
  named corpora/datasets/canonical sources/standard handbooks OR widely used technical phrases that
  are strongly specific to the topic.
- ASCII ONLY in outputs intended for queries: use hyphen "-" not "–", avoid curly quotes.


OUTPUT JSON SCHEMA:
{
  "chapter_id": "...",
  "scope": {"in": [...], "out": [...]},
  "must_cover": [...],
  "should_cover": [...],
  "must_avoid": [...],
  "anchors": {
    "disambiguators": {"en": [...], "de": [...]},
    "evidence_proxies": {"en": [...], "de": [...]},
    "mechanisms": {"en": [...], "de": [...]},
    "high_precision": {"en": [...], "de": [...]}
  },
  "facets": [
    {
      "facet_id": "snake_case",
      "label_en": "...",
      "label_de": "...",
      "terms_en": [...],
      "terms_de": [...],
      "evidence_proxies_en": [...],
      "evidence_proxies_de": [...],
      "safe_negatives": [...],
      "risky_negatives": [...]
    }
  ],
  "negative_pool": {"en": [...], "de": [...]}
}

SELF-CHECK BEFORE YOU ANSWER:
- JSON parses
- facet_id unique; >= 8 facets
- EN and DE are populated
- At least 6 disambiguators per language are "context-gates" (2–6 words, object-bound; NOT generic)
- No disambiguator is a bare discipline label (must be object-bound)
- At least 3 high_precision terms per language are "hard anchors" (named corpus/dataset/canon/handbook)
Return ONLY the JSON.
"""


<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
PROMPT_YEAR_BOUNDS_V2_USER = """INPUT (JSON):
{input_json}

TASK:
1) Choose publication-year bounds:
   - soft_min_year (integer)
   - soft_max_year (integer)
2) Choose no_year_share (0.0–0.4) meaning % queries that should run without year filters.
3) Write a disambiguator policy for query text:
   - disambiguator_strength: "strong" | "medium"
   - strong means: every query must contain ≥1 disambiguator anchor.
   - medium means: at least 70% must contain ≥1 disambiguator anchor.
4) Output rationale in EN and DE, short.

CONSTRAINTS:
- Optimize completeness, but prevent obvious era/field drift by using disambiguator policy (not year bounds).
- Output VALID JSON only.

OUTPUT:
{
  "chapter_id": "...",
  "soft_min_year": 1800,
  "soft_max_year": 2026,
  "no_year_share": 0.20,
  "disambiguator_strength": "strong|medium",
  "rationale_en": "...",
  "rationale_de": "..."
}"""


<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
PROMPT_OPENALEX_QUERY_PLAN_V1_USER = """INPUT (JSON):
{input_json}

CONTEXT (how your queries are used):
- Each query string becomes the VALUE of an OpenAlex /works filter of the form:
    title_and_abstract.search:<YOUR_QUERY>
- Additional filters (years, language) are applied separately via the filters object.

OPENALEX QUERY DIALECT (CRITICAL):
- You MAY use boolean operators AND / OR / NOT and parentheses.
- Do NOT include commas in the query string (commas separate filter parts in OpenAlex).
- Avoid ':' in the query string.
- Avoid newlines and tabs.
- Keep each query reasonably short (aim <= 280 characters; prefer 2-4 clauses).

TASK:
Generate up to targets.queries_total query objects for service=openalex (do NOT exceed targets.queries_total).
Aim to match these targets, but quality > exact counts:
- targets.languages (en/de counts)
- targets.kinds (anchor/facet/proxy/authority counts)
- About round(year_policy.no_year_share * targets.queries_total) queries should have use_no_year=true
If you cannot meet a target without producing near-duplicates or dialect violations, return fewer queries.
Order queries by expected yield/precision (best first).

OUTPUT SCHEMA (STRICT):
{
  "chapter_id": "...",
  "service": "openalex",
  "queries": [
    {
      "kind": "anchor|facet|proxy|authority",
      "language": "en|de",
      "cap": 60,
      "use_no_year": false,
      "facet_id": "<facet_id> or empty string only if kind=authority",
      "query": "...",
      "filters": {
        "year_min": 1800,
        "year_max": 2026,
        "language": "en|de",
        "concept_ids": []
      },
      "notes": "short rationale / targets"
    }
  ]
}

FILTER RULES:
- If use_no_year=true then set filters.year_min and filters.year_max to null.
- filters.language MUST equal the query.language.
- filters.concept_ids MUST be [] (do NOT invent OpenAlex concept IDs; concept filtering is handled elsewhere).

CONTENT RULES (make queries "as perfect as possible"):
- Every NON-authority query must include:
  (1) >=1 disambiguator/context-gate term from blueprint.anchors.disambiguators (object-bound, not generic)
  (2) >=1 mechanism term from blueprint.anchors.mechanisms (or facet terms)
  (3) >=1 evidence/proxy term from blueprint.anchors.evidence_proxies (or facet evidence proxies)
- Anchor queries are highest precision and MUST include:
  - >=2 disambiguator/context-gate phrases
  - >=1 high_precision term when available
  - >=1 mechanism and >=1 evidence/proxy
- Facet queries must clearly focus on their facet_id using that facet's terms/evidence proxies.
- Proxy queries should vary evidence/method angles (datasets, measurements, proxies) to broaden recall without drifting out of scope.
- Authority queries (few) should target surveys/handbooks/reviews/state-of-the-field, still kept in-domain via disambiguators.

DIVERSITY + COVERAGE:
- Avoid near-duplicates (not just reordered terms).
- Ensure every facet_id from blueprint.facets is covered by multiple queries.
- Spread blueprint.must_cover items across anchor/facet queries (do not copy/paste one template).

NEGATIVES:
- Use NOT sparingly and only for high-risk confounders from blueprint.negative_pool or facet safe_negatives.
- Prefer recall over aggressive exclusion.

SELF-CHECK BEFORE YOU ANSWER:
- Valid JSON only (no markdown, no comments)
- Number of queries <= targets.queries_total
- facet_id is valid (or empty only for authority)
- No commas or ':' in query strings
Return ONLY the JSON."""


<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
PROMPT_SEMANTICSCHOLAR_QUERY_PLAN_V1_USER = """INPUT (JSON):
{input_json}

CONTEXT (how your queries are used):
- Queries are sent to Semantic Scholar's paper/search endpoint as the `query` parameter.
- The pipeline will sanitize queries by removing boolean words (AND/OR/NOT), parentheses/brackets, and ':'; then it truncates to 240 chars.
- Therefore: write queries that already survive this sanitization unchanged.

SEMANTIC SCHOLAR DIALECT (CRITICAL):
- Do NOT use AND / OR / NOT, wildcards, parentheses, brackets, braces, pipes, '+' or '*'.
- Do NOT include ':' characters.
- Keep queries short and phrase-heavy (aim <= 12-16 tokens; <= 240 characters).
- Prefer many diverse queries over one mega-query.

TASK:
Generate up to targets.queries_total query objects for service=semanticscholar (do NOT exceed targets.queries_total).
Aim to match these targets, but quality > exact counts:
- targets.languages (en/de counts)
- targets.kinds (anchor/facet/proxy/authority counts)
- About round(year_policy.no_year_share * targets.queries_total) queries should have use_no_year=true
If you cannot meet a target without producing near-duplicates or dialect violations, return fewer queries.
Order queries by expected yield/precision (best first).

OUTPUT SCHEMA (STRICT):
{
  "chapter_id": "...",
  "service": "semanticscholar",
  "queries": [
    {
      "kind": "anchor|facet|proxy|authority",
      "language": "en|de",
      "cap": 40,
      "use_no_year": false,
      "facet_id": "<facet_id> or empty string only if kind=authority",
      "query": "...",
      "filters": {
        "year_min": 1800,
        "year_max": 2026,
        "language": "en|de",
        "concept_ids": []
      },
      "notes": "short rationale / targets"
    }
  ]
}

FILTER RULES:
- If use_no_year=true then set filters.year_min and filters.year_max to null.
- filters.language MUST equal the query.language (metadata only; S2 does not enforce language).
- filters.concept_ids MUST be []

CONTENT RULES (make queries "as perfect as possible"):
- Every NON-authority query must include:
  (1) >=1 disambiguator/context-gate phrase (object-bound) from blueprint.anchors.disambiguators
  (2) >=1 mechanism term (or facet terms)
  (3) >=1 evidence/proxy term (or facet evidence proxies)
- Anchor queries MUST include >=2 disambiguator/context-gate phrases and >=1 high_precision term when available.
- Facet queries must clearly focus on their facet_id using that facet's terms/evidence proxies.
- Authority queries (few) should target reviews/surveys/handbooks/state-of-the-field, still kept in-domain.

DIVERSITY + COVERAGE:
- Avoid near-duplicates.
- Ensure every facet_id from blueprint.facets is covered by multiple queries.
- Spread blueprint.must_cover items across the plan.

SELF-CHECK BEFORE YOU ANSWER:
- Valid JSON only (no markdown, no comments)
- Number of queries <= targets.queries_total
- facet_id is valid (or empty only for authority)
- Query strings contain none of these: AND OR NOT [] () {} | + * ^ ~ :
- Query strings are <= 240 characters
Return ONLY the JSON."""


# Legacy single-agent query-plan prompt (kept for reference; NOT used in this notebook variant)
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
PROMPT_QUERY_PLAN_V2_USER = """INPUT (JSON):
{input_json}

SERVICE DIALECT RULES:
OpenAlex:
- Use boolean operators AND/OR/NOT in the query string where helpful. 
- Use filters separately for years/language/concepts (do NOT embed those as long text rules). :contentReference[oaicite:5]{index=5}

Semantic Scholar:
- Use plain keyword/phrase queries optimized for paper/search (avoid complex custom boolean syntax).
- Keep query strings short; rely on running many diverse queries instead of one mega-query. :contentReference[oaicite:6]{index=6}

TASK:
Generate 100 query objects. Each query object must include:
- id: "Q001"..."Q100"
- service: "openalex"|"semanticscholar"
- kind: "anchor"|"facet"|"proxy"|"authority"
- language: "en"|"de"
- cap: integer (suggested: OA 60–120; S2 40–120)
- use_no_year: boolean (exactly round(no_year_share*100) queries True)
- facet_id: one of blueprint facets or "" for authority
- query: the query string (service dialect rules apply)
- filters: service-specific filters object (years, language, concept_ids if relevant)

ANCHOR STRICTNESS (applies when kind="anchor"):
- Each anchor query MUST include:
  (1) >= 2 disambiguator context-gates (object+time and object+place/setting, if available)
  (2) >= 1 high_precision term (named corpus/dataset/canon/handbook/technical phrase), when available
  (3) >= 1 mechanism term
  (4) >= 1 evidence/proxy term
- Anchor queries should be SHORT and PHRASE-HEAVY:
  - OpenAlex: prefer quoted multiword phrases + AND; keep to <= 4 clauses.
  - Semantic Scholar: <= 12 tokens plus up to 2 quoted phrases.
- ASCII ONLY: use "-" not "–"; avoid fancy quotes.

OBJECT BINDING RULE:
If a term could apply to many domains (e.g., "archaeology", "trade", "pollen analysis", "inflation"),
it must be paired with an object/setting gate (e.g., "<object> trade", "<object> pollen record").


BUILDING RULES (CRITICAL):
1) Every query must include:
   - ≥1 mechanism term
   - ≥1 evidence/proxy term
   - ≥1 disambiguator term if disambiguator_strength="strong"
2) Diversity:
   - No more than 2 queries may share the same triple:
     (primary_disambiguator, primary_evidence_proxy, primary_mechanism)
   - Do not reuse the same anchor bundle across many queries.
   - For the triple constraint, "primary_disambiguator" MUST be a context-gate phrase
  (multiword, object-bound) rather than a generic field label.
3) Coverage:
   - Every MUST-COVER item must be explicitly targeted by ≥10 queries across the plan.
   - Every facet must receive ≥6 queries.
4) Negatives:
   - For OpenAlex, use NOT terms in the query string for the strongest confounders when needed.
   - For Semantic Scholar, keep negatives minimal (only if very high risk of drift).
5) Authority queries (5):
   - Focus on high-level surveys, handbooks, historiography, “state of the field”.
   - Still include disambiguators so they stay in-domain.
COVERAGE IMPLEMENTATION (do this explicitly):
- Allocate queries so that each MUST-COVER item is targeted by >= 10 queries.
- Implementation rule: in notes, annotate each query with:
  "targets: must_cover=<item>; facet=<facet_id>"
- Ensure each facet has >= 6 queries (already required).


CONCEPT FILTER STRATEGY:
- For OpenAlex: 70% of OA queries should include concept_ids (from blueprint high_precision or disambiguators if appropriate),
  30% should omit concept_ids to protect recall.

OUTPUT FORMAT (STRICT JSON):
{
  "chapter_id": "...",
  "queries": [
    {
      "id": "Q001",
      "service": "openalex",
      "kind": "anchor",
      "language": "en",
      "cap": 100,
      "use_no_year": true,
      "facet_id": "taxation_and_fiscal_systems",
      "query": "...",
      "filters": {
        "year_min": 1800,
        "year_max": 2026,
        "language": "en",
        "concept_ids": ["C...","C..."]
      }
    }
    ...
  ]
}

SELF-CHECK BEFORE YOU ANSWER:
- Exactly 100 queries
- IDs are complete Q001..Q100
- language counts match targets
- service counts match targets
- use_no_year count matches year_policy
- each facet has ≥6 queries
- no invalid facet_id
Return ONLY JSON."""


<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>
PROMPT_GAPFILL_V2_USER = """INPUT (JSON):
{input_json}

TASK:
1) Identify the 1–3 biggest gaps (must-cover items or facets underperforming in top-ranked results).
2) For each gap, propose:
   - either a new query (preferred) OR a replacement for a low-productivity query
3) Queries must follow the service dialect rules:
   - OpenAlex: boolean ok, filters separate
   - Semantic Scholar: short keyword/phrase query
4) Each proposed query must explicitly state:
   - which gap it targets
   - why it should reduce drift and improve precision

OUTPUT (VALID JSON ONLY):
{
  "chapter_id": "...",
  "gaps_found": [
    {"gap": "...", "evidence": "..."}
  ],
  "changes": [
    {
      "action": "add|replace",
      "replace_query_id": "Q044|",
      "new_query": {
        "service": "...",
        "kind": "...",
        "language": "...",
        "cap": 80,
        "use_no_year": false,
        "facet_id": "...",
        "query": "...",
        "filters": {...}
      },
      "targets": ["must_cover: ...", "facet: ..."],
      "reason": "..."
    }
  ]
}"""


def _prompt_with_input(template: str, input_obj: Dict[str, Any]) -> str:
    return template.replace("{input_json}", json.dumps(input_obj, ensure_ascii=False, indent=2))


# -----------------------------
# Agents + cache helpers
# -----------------------------

blueprint_v3_agent = Agent(
    name="StageB blueprint_v3",
    model=BLUEPRINT_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=PROMPT_BLUEPRINT_V3_SYSTEM,
    output_type=AgentOutputSchema(BlueprintV3, strict_json_schema=False),
)

year_bounds_v2_agent = Agent(
    name="StageB year_bounds_v2",
    model=BLUEPRINT_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=PROMPT_YEAR_BOUNDS_V2_SYSTEM,
    output_type=AgentOutputSchema(YearBoundsV2, strict_json_schema=False),
)

openalex_query_plan_v1_agent = Agent(
    name="StageB query_plan_service_agents_v1 (OpenAlex)",
    model=BLUEPRINT_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=PROMPT_OPENALEX_QUERY_PLAN_V1_SYSTEM,
    output_type=AgentOutputSchema(OpenAlexQueryPlanDraftV1, strict_json_schema=False),
)

semanticscholar_query_plan_v1_agent = Agent(
    name="StageB query_plan_service_agents_v1 (Semantic Scholar)",
    model=BLUEPRINT_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=PROMPT_SEMANTICSCHOLAR_QUERY_PLAN_V1_SYSTEM,
    output_type=AgentOutputSchema(SemanticScholarQueryPlanDraftV1, strict_json_schema=False),
)

gapfill_v2_agent = Agent(
    name="StageB gapfill_v2",
    model=BLUEPRINT_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=PROMPT_GAPFILL_V2_SYSTEM,
    output_type=AgentOutputSchema(GapFillV2, strict_json_schema=False),
)


async def _run_agent(agent: Agent, prompt: str) -> Tuple[Any, Dict[str, Any]]:
    try:
        res = await asyncio.wait_for(Runner.run(agent, prompt), timeout=float(STAGEB_AGENT_TIMEOUT_SEC))
    except asyncio.TimeoutError as e:
        name = str(getattr(agent, "name", "") or "agent")
        raise RuntimeError(f"{name} timed out after {STAGEB_AGENT_TIMEOUT_SEC}s") from e
    usage = getattr(getattr(res, "context_wrapper", None), "usage", None)
    return res.final_output, (cost_from_usage(usage, model=BLUEPRINT_MODEL) if usage is not None else _zero_usage())


def _normalize_blueprint_v3(ch: Dict[str, Any], raw: Any) -> BlueprintV3:
    cid = str(ch.get("chapter_id") or "chapter")
    try:
        bp = raw if isinstance(raw, BlueprintV3) else BlueprintV3.model_validate(raw)
    except Exception:
        bp = _det_blueprint_v3(ch)

    base = _det_blueprint_v3(ch)

    in_scope = _ensure_len(list(getattr(bp.scope, "in_", []) or []), min_len=2, max_len=12, fallback=base.scope.in_)
    out_scope = _ensure_len(list(getattr(bp.scope, "out", []) or []), min_len=2, max_len=12, fallback=base.scope.out)

    def _fill_lang(xs: List[str], fallback: List[str], *, min_len: int, max_len: int) -> List[str]:
        return _ensure_len(xs or [], min_len=min_len, max_len=max_len, fallback=fallback)

    a = bp.anchors if hasattr(bp, "anchors") else base.anchors
    dis_en = _fill_lang(a.disambiguators.en, base.anchors.disambiguators.en, min_len=4, max_len=14)
    dis_de = _fill_lang(a.disambiguators.de, base.anchors.disambiguators.de, min_len=4, max_len=14)
    ev_en = _fill_lang(a.evidence_proxies.en, base.anchors.evidence_proxies.en, min_len=4, max_len=18)
    ev_de = _fill_lang(a.evidence_proxies.de, base.anchors.evidence_proxies.de, min_len=4, max_len=18)
    mech_en = _fill_lang(a.mechanisms.en, base.anchors.mechanisms.en, min_len=4, max_len=18)
    mech_de = _fill_lang(a.mechanisms.de, base.anchors.mechanisms.de, min_len=4, max_len=18)
    hp_en = _fill_lang(a.high_precision.en, base.anchors.high_precision.en, min_len=3, max_len=16)
    hp_de = _fill_lang(a.high_precision.de, base.anchors.high_precision.de, min_len=3, max_len=16)

    must_cover = _ensure_len(list(bp.must_cover or []), min_len=8, max_len=12, fallback=base.must_cover)
    should_cover = _ensure_len(list(bp.should_cover or []), min_len=6, max_len=12, fallback=base.should_cover)
    must_avoid = _ensure_len(list(bp.must_avoid or []), min_len=6, max_len=12, fallback=base.must_avoid)

    facets_in = list(bp.facets or [])
    facets: List[BlueprintFacet] = []
    seen: set[str] = set()
    for f in facets_in:
        fid = _slugify(getattr(f, "facet_id", "") or "", max_len=48)
        if not fid:
            fid = f"facet_{len(facets)+1}"
        if fid in seen:
            fid = f"{fid}_{len(facets)+1}"
        seen.add(fid)
        facets.append(
            BlueprintFacet(
                facet_id=fid,
                label_en=_clean_str(getattr(f, "label_en", "")) or fid.replace("_", " "),
                label_de=_clean_str(getattr(f, "label_de", "")) or fid.replace("_", " "),
                terms_en=_ensure_len(list(getattr(f, "terms_en", []) or []), min_len=6, max_len=10, fallback=mech_en),
                terms_de=_ensure_len(list(getattr(f, "terms_de", []) or []), min_len=6, max_len=10, fallback=mech_de),
                evidence_proxies_en=_ensure_len(list(getattr(f, "evidence_proxies_en", []) or []), min_len=3, max_len=6, fallback=ev_en),
                evidence_proxies_de=_ensure_len(list(getattr(f, "evidence_proxies_de", []) or []), min_len=3, max_len=6, fallback=ev_de),
                safe_negatives=_ensure_len(list(getattr(f, "safe_negatives", []) or []), min_len=1, max_len=6, fallback=[]),
                risky_negatives=_ensure_len(list(getattr(f, "risky_negatives", []) or []), min_len=0, max_len=6, fallback=[]),
            )
        )
        if len(facets) >= 12:
            break

    if len(facets) < 8:
        for f in base.facets:
            if len(facets) >= 8:
                break
            if f.facet_id in seen:
                continue
            seen.add(f.facet_id)
            facets.append(f)

    negp = bp.negative_pool if hasattr(bp, "negative_pool") else base.negative_pool
    neg_en = _ensure_len(list(getattr(negp, "en", []) or []), min_len=8, max_len=30, fallback=base.negative_pool.en + must_avoid)
    neg_de = _ensure_len(list(getattr(negp, "de", []) or []), min_len=8, max_len=30, fallback=base.negative_pool.de + must_avoid)

    return BlueprintV3(
        chapter_id=cid,
        scope=ScopeObj(in_=in_scope, out=out_scope),
        must_cover=must_cover[:8],
        should_cover=should_cover[:6],
        must_avoid=must_avoid[:6],
        anchors=BlueprintAnchors(
            disambiguators=AnchorsLang(en=dis_en, de=dis_de),
            evidence_proxies=AnchorsLang(en=ev_en, de=ev_de),
            mechanisms=AnchorsLang(en=mech_en, de=mech_de),
            high_precision=AnchorsLang(en=hp_en, de=hp_de),
        ),
        facets=facets,
        negative_pool=NegativePool(en=neg_en, de=neg_de),
    )


def _normalize_year_bounds_v2(ch: Dict[str, Any], raw: Any) -> YearBoundsV2:
    cid = str(ch.get("chapter_id") or "chapter")
    try:
        y = raw if isinstance(raw, YearBoundsV2) else YearBoundsV2.model_validate(raw)
    except Exception:
        y = _det_year_bounds_v2(cid)
    soft_max = min(CURRENT_YEAR, max(1800, int(y.soft_max_year or CURRENT_YEAR)))
    soft_min = max(1800, int(y.soft_min_year or STAGEB_DEFAULT_SOFT_MIN_YEAR))
    if soft_min > soft_max:
        soft_min = max(1800, soft_max - 60)
    nys = max(0.0, min(0.40, float(y.no_year_share)))
    dis_strength = y.disambiguator_strength if y.disambiguator_strength in ("strong", "medium") else "strong"
    return YearBoundsV2(
        chapter_id=cid,
        soft_min_year=soft_min,
        soft_max_year=soft_max,
        no_year_share=nys,
        disambiguator_strength=dis_strength,  # type: ignore[arg-type]
        rationale_en=_clean_str(y.rationale_en),
        rationale_de=_clean_str(y.rationale_de),
    )


def _normalize_query_plan_v2(ch: Dict[str, Any], bp: BlueprintV3, y: YearBoundsV2, raw: Any) -> QueryPlanV2:
    cid = str(ch.get("chapter_id") or "chapter")
    if raw is None:
        plan = _det_query_plan_v2(cid, bp, y, total=int(STAGEB_QUERY_TARGET))
    else:
        try:
            plan = raw if isinstance(raw, QueryPlanV2) else QueryPlanV2.model_validate(raw)
        except Exception:
            plan = _det_query_plan_v2(cid, bp, y, total=int(STAGEB_QUERY_TARGET))

    plan.chapter_id = cid

    qs: List[QueryObjV2] = []
    valid_facets = {f.facet_id for f in (bp.facets or [])}
    for q in list(plan.queries or []):
        svc = str(getattr(q, "service", "") or "").strip().lower()
        if svc in ("semanticscholar_bulk", "s2"):
            svc = "semanticscholar"
        if svc not in ("openalex", "semanticscholar"):
            continue

        kind = str(getattr(q, "kind", "") or "").strip().lower()
        if kind not in ("anchor", "facet", "proxy", "authority"):
            kind = "facet"

        lang = str(getattr(q, "language", "") or "").strip().lower()
        if lang not in ("en", "de"):
            lang = "en"

        facet_id = str(getattr(q, "facet_id", "") or "")
        if kind == "authority":
            facet_id = ""
        elif facet_id not in valid_facets:
            facet_id = next(iter(valid_facets), "")

        qtxt = _clean_str(getattr(q, "query", "") or "")
        if not qtxt:
            dis = bp.anchors.disambiguators.en if lang == "en" else bp.anchors.disambiguators.de
            mech = bp.anchors.mechanisms.en if lang == "en" else bp.anchors.mechanisms.de
            ev = bp.anchors.evidence_proxies.en if lang == "en" else bp.anchors.evidence_proxies.de
            qtxt = _oa_expr(dis, mech, ev, []) if svc == "openalex" else _s2_expr(dis, mech, ev)

        cap = _cap(svc, int(getattr(q, "cap", 0) or 0))
        use_no_year = bool(getattr(q, "use_no_year", False))
        notes = _clean_str(getattr(q, "notes", "") or "")
        f = getattr(q, "filters", None) or QueryFiltersV2()
        filters = QueryFiltersV2(
            year_min=(None if use_no_year else int(getattr(f, "year_min", None) or y.soft_min_year)),
            year_max=(None if use_no_year else int(getattr(f, "year_max", None) or y.soft_max_year)),
            language=lang,  # type: ignore[arg-type]
            concept_ids=_valid_concept_ids(list(getattr(f, "concept_ids", []) or [])),
        )

        qo = QueryObjV2(
            id=str(getattr(q, "id", "") or ""),
            service=svc,  # type: ignore[arg-type]
            kind=kind,  # type: ignore[arg-type]
            language=lang,  # type: ignore[arg-type]
            cap=cap,
            use_no_year=use_no_year,
            facet_id=facet_id,
            query=qtxt,
            filters=filters,
            notes=notes,
        )
        qs.append(qo)

    plan.queries = qs
    plan = _enforce_plan_requirements(bp, y, plan)

    if len(plan.queries) < 20:
        plan = _det_query_plan_v2(cid, bp, y, total=int(STAGEB_QUERY_TARGET))

    return plan


async def get_or_create_blueprint_v3(ch: Dict[str, Any]) -> Tuple[BlueprintV3, Dict[str, Any]]:
    cid = str(ch.get("chapter_id"))
    p = STAGEB_BLUEPRINT_DIR / f"{cid}.json"
    if p.exists() and not STAGEB_FORCE_ALL:
        try:
            raw = json.loads(p.read_text(encoding="utf-8"))
        except Exception as e:
            raw = None
            if bool(STAGEB_LOG_CACHE_INVALID):
                print(f"[StageB:{cid}] blueprint_v3 cache invalid (read/parse error) -> rebuilding: {e}")
        meta = raw.get("_meta") if isinstance(raw, dict) else {}
        if str((meta or {}).get("prompt_version") or "") == STAGEB_PROMPT_VERSION:
            if bool(STAGEB_LOG_CACHE_HITS):
                print(f"[StageB:{cid}] blueprint_v3 cache hit")
            return _normalize_blueprint_v3(ch, raw), _zero_usage()
        if bool(STAGEB_LOG_CACHE_INVALID):
            print(f"[StageB:{cid}] blueprint_v3 cache invalid (prompt_version mismatch) -> rebuilding")

    inp = {
        "chapter_id": cid,
        "title": str(ch.get("title") or ""),
        "original_text_language": str(ch.get("original_text_language") or "en"),
        "original_text": str(ch.get("original_text") or ""),
        "target_languages": ["en", "de"],
        "services": ["openalex", "semanticscholar"],
        "requirements": {"must_cover_count": 8, "should_cover_count": 6, "must_avoid_count": 6},
    }
    prompt = _prompt_with_input(PROMPT_BLUEPRINT_V3_USER, inp)

    usage, raw_out = _zero_usage(), None
    try:
        raw_out, usage = await _run_agent(blueprint_v3_agent, prompt)
    except Exception as e:
        print(f"[StageB:{cid}] blueprint_v3 failed; deterministic fallback used: {e}")

    bp = _normalize_blueprint_v3(ch, raw_out)
    obj = bp.model_dump(by_alias=True)
    obj["_meta"] = {
        "stage": "blueprint_v3",
        "model": BLUEPRINT_MODEL,
        "variant": STAGEB_FINAL_VARIANT,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "prompt_version": STAGEB_PROMPT_VERSION,
    }
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return bp, usage


async def get_or_create_year_bounds_v2(ch: Dict[str, Any], bp: BlueprintV3) -> Tuple[YearBoundsV2, Dict[str, Any]]:
    cid = str(ch.get("chapter_id"))
    p = STAGEB_YEAR_BOUNDS_DIR / f"{cid}.json"
    if p.exists() and not STAGEB_FORCE_ALL:
        try:
            raw = json.loads(p.read_text(encoding="utf-8"))
        except Exception as e:
            raw = None
            if bool(STAGEB_LOG_CACHE_INVALID):
                print(f"[StageB:{cid}] year_bounds_v2 cache invalid (read/parse error) -> rebuilding: {e}")
        meta = raw.get("_meta") if isinstance(raw, dict) else {}
        if str((meta or {}).get("prompt_version") or "") == STAGEB_PROMPT_VERSION:
            if bool(STAGEB_LOG_CACHE_HITS):
                print(f"[StageB:{cid}] year_bounds_v2 cache hit")
            return _normalize_year_bounds_v2(ch, raw), _zero_usage()
        if bool(STAGEB_LOG_CACHE_INVALID):
            print(f"[StageB:{cid}] year_bounds_v2 cache invalid (prompt_version mismatch) -> rebuilding")

    inp = {
        "chapter_id": cid,
        "blueprint": bp.model_dump(by_alias=True),
        "target_languages": ["en", "de"],
        "default_soft_min_year": int(STAGEB_DEFAULT_SOFT_MIN_YEAR),
        "default_soft_max_year": int(CURRENT_YEAR),
        "no_year_share_target": float(STAGEB_DEFAULT_NO_YEAR_SHARE),
    }
    prompt = _prompt_with_input(PROMPT_YEAR_BOUNDS_V2_USER, inp)

    usage, raw_out = _zero_usage(), None
    try:
        raw_out, usage = await _run_agent(year_bounds_v2_agent, prompt)
    except Exception as e:
        print(f"[StageB:{cid}] year_bounds_v2 failed; deterministic fallback used: {e}")

    y = _normalize_year_bounds_v2(ch, raw_out)
    obj = y.model_dump()
    obj["_meta"] = {
        "stage": "year_bounds_v2",
        "model": BLUEPRINT_MODEL,
        "variant": STAGEB_FINAL_VARIANT,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "prompt_version": STAGEB_PROMPT_VERSION,
    }
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return y, usage


def _scale_counts(counts: Dict[str, int], total: int) -> Dict[str, int]:
    items = [(str(k), int(v)) for k, v in (counts or {}).items()]
    keys = [k for k, _ in items if k]
    if not keys:
        return {}
    total = int(total or 0)
    if total <= 0:
        return {k: 0 for k in keys}

    old_total = sum(max(0, v) for _, v in items)
    if old_total <= 0:
        base = total // len(keys)
        rem = total - (base * len(keys))
        out = {k: base for k in keys}
        for k in keys[:rem]:
            out[k] += 1
        return out

    raw = [(k, (max(0, v) * float(total) / float(old_total))) for k, v in items]
    out = {k: int(x) for k, x in raw}
    remainder = total - sum(out.values())
    fracs = sorted([(x - int(x), k) for k, x in raw], reverse=True)
    i = 0
    while remainder > 0 and fracs:
        out[fracs[i % len(fracs)][1]] += 1
        remainder -= 1
        i += 1
    return out


def _split_counts_two_services(
    total_counts: Dict[str, int],
    n_a: int,
    n_b: int,
) -> tuple[Dict[str, int], Dict[str, int]]:
    n_a = int(n_a or 0)
    n_b = int(n_b or 0)
    total = n_a + n_b
    items = [(str(k), int(v)) for k, v in (total_counts or {}).items()]
    keys = [k for k, _ in items if k]
    if not keys or total <= 0:
        return ({k: 0 for k in keys}, {k: 0 for k in keys})

    floors: Dict[str, int] = {}
    fracs: List[tuple[float, str]] = []
    for k, v in items:
        v = max(0, int(v))
        exact = (v * float(n_a)) / float(total)
        a = int(exact)
        floors[k] = a
        fracs.append((exact - a, k))

    rem = n_a - sum(floors.values())
    fracs.sort(key=lambda t: (t[0], t[1]), reverse=True)
    for i in range(max(0, rem)):
        floors[fracs[i % len(fracs)][1]] += 1

    a_counts = {k: int(floors.get(k, 0)) for k in keys}
    b_counts = {k: max(0, int(total_counts.get(k, 0)) - a_counts[k]) for k in keys}
    return a_counts, b_counts


def _validate_service_draft_v1(
    draft: Any,
    *,
    service: str,
    chapter_id: str,
    bp: BlueprintV3,
    y: YearBoundsV2,
    targets: Dict[str, Any],
) -> List[str]:
    errs: List[str] = []
    if draft is None:
        return ['no output']

    got_cid = getattr(draft, 'chapter_id', None)
    if str(got_cid or '') != str(chapter_id):
        errs.append(f'chapter_id mismatch (got {got_cid!r})')

    qs = list(getattr(draft, 'queries', []) or [])

    valid_facets = {f.facet_id for f in (bp.facets or [])}

    for i, q in enumerate(qs, start=1):
        kind = str(getattr(q, 'kind', '') or '')
        lang = str(getattr(q, 'language', '') or '')
        cap = int(getattr(q, 'cap', 0) or 0)
        facet_id = str(getattr(q, 'facet_id', '') or '')
        qtxt = str(getattr(q, 'query', '') or '')

        if not qtxt.strip():
            errs.append(f'Q{i}: empty query')
        if any(ch in qtxt for ch in ('\n', '\r', '\t')):
            errs.append(f'Q{i}: query contains newline/tab')

        if kind == 'authority':
            if facet_id:
                errs.append(f'Q{i}: authority must use empty facet_id')
        else:
            if facet_id not in valid_facets:
                errs.append(f'Q{i}: invalid facet_id={facet_id!r}')

        if service == 'openalex':
            if ',' in qtxt:
                errs.append(f'Q{i}: OpenAlex query must not contain commas')
            if ':' in qtxt:
                errs.append(f'Q{i}: OpenAlex query must not contain :')
            if cap < 60 or cap > 120:
                errs.append(f'Q{i}: OpenAlex cap out of range (60-120): {cap}')
        else:
            if len(qtxt) > 240:
                errs.append(f'Q{i}: S2 query too long ({len(qtxt)} chars)')
            if _S2_FORBIDDEN_RE.search(qtxt):
                errs.append(f'Q{i}: S2 query must not contain AND/OR/NOT')
            if re.search(r'[][(){}|+*^~:]', qtxt):
                errs.append(f'Q{i}: S2 query contains forbidden punctuation')
            if cap < 40 or cap > 120:
                errs.append(f'Q{i}: S2 cap out of range (40-120): {cap}')

        f = getattr(q, 'filters', None) or QueryFiltersV2()
        flang = str(getattr(f, 'language', '') or '')
        if flang != lang:
            errs.append(f'Q{i}: filters.language must equal query.language ({lang})')
        if list(getattr(f, 'concept_ids', []) or []):
            errs.append(f'Q{i}: filters.concept_ids must be []')

        use_no_year = bool(getattr(q, 'use_no_year', False))
        if use_no_year:
            if getattr(f, 'year_min', None) is not None or getattr(f, 'year_max', None) is not None:
                errs.append(f'Q{i}: use_no_year=true requires year_min/year_max=null')
        else:
            ymin = getattr(f, 'year_min', None)
            ymax = getattr(f, 'year_max', None)
            if ymin is None or ymax is None:
                errs.append(f'Q{i}: missing year_min/year_max (use_no_year=false)')
            else:
                try:
                    ymin_i = int(ymin)
                    ymax_i = int(ymax)
                except Exception:
                    errs.append(f'Q{i}: year_min/year_max must be integers')
                else:
                    if ymin_i > ymax_i:
                        errs.append(f'Q{i}: year_min > year_max ({ymin_i} > {ymax_i})')
                    if ymin_i < int(y.soft_min_year) or ymax_i > int(y.soft_max_year):
                        errs.append(
                            f'Q{i}: year range {ymin_i}-{ymax_i} outside soft bounds {y.soft_min_year}-{y.soft_max_year}'
                        )

    return errs


def _autofix_service_draft_v1(
    draft: Any,
    *,
    service: str,
    chapter_id: str,
    bp: BlueprintV3,
    y: YearBoundsV2,
    targets: Dict[str, Any],
) -> tuple[Any, Dict[str, int]]:
    stats: Dict[str, int] = {
        "dropped_unparseable": 0,
        "dropped_empty": 0,
        "dropped_dupe": 0,
        "sanitized_oa": 0,
        "sanitized_s2": 0,
        "fixed_kind": 0,
        "fixed_language": 0,
        "fixed_facet_id": 0,
        "fixed_years": 0,
        "fixed_caps": 0,
        "fixed_filters": 0,
    }
    if draft is None:
        return None, stats

    try:
        draft.chapter_id = str(chapter_id)
    except Exception:
        pass

    want_total = int(targets.get("queries_total") or 0)
    valid_facet_ids = [str(f.facet_id) for f in (bp.facets or []) if str(getattr(f, "facet_id", "") or "").strip()]
    valid_facet_set = set(valid_facet_ids)

    def _norm_kind(k: Any) -> str:
        s = str(k or "").strip().lower()
        if s in ("authority", "authorities", "review", "survey", "handbook", "overview", "state_of_the_field"):
            return "authority"
        if s in ("anchor", "anchors", "context_gate", "context-gate", "high_precision", "high-precision", "main"):
            return "anchor"
        if s in ("proxy", "proxies", "evidence", "method", "measurement"):
            return "proxy"
        if s in ("facet", "facets", "subtopic", "topic"):
            return "facet"
        return s if s in ("anchor", "facet", "proxy", "authority") else "facet"

    def _norm_lang(l: Any) -> str:
        s = str(l or "").strip().lower()
        if s in ("english", "eng", "en-us", "en-gb"):
            return "en"
        if s in ("german", "deu", "ger"):
            return "de"
        return s if s in ("en", "de") else "en"

    qs_in = list(getattr(draft, "queries", []) or [])
    out: List[QueryObjDraftV1] = []
    seen: set[str] = set()

    for q_in in qs_in:
        try:
            q = q_in if isinstance(q_in, QueryObjDraftV1) else QueryObjDraftV1.model_validate(q_in)
        except Exception:
            stats["dropped_unparseable"] += 1
            continue

        kind = _norm_kind(getattr(q, "kind", "facet"))
        if str(getattr(q, "kind", "")) != kind:
            stats["fixed_kind"] += 1

        lang = _norm_lang(getattr(q, "language", "en"))
        if str(getattr(q, "language", "")) != lang:
            stats["fixed_language"] += 1

        use_no_year = bool(getattr(q, "use_no_year", False))

        facet_id = str(getattr(q, "facet_id", "") or "").strip()
        if kind == "authority":
            if facet_id:
                stats["fixed_facet_id"] += 1
            facet_id = ""
        else:
            if valid_facet_set and facet_id not in valid_facet_set:
                stats["fixed_facet_id"] += 1
                facet_id = valid_facet_ids[len(out) % len(valid_facet_ids)]

        qtxt_raw = str(getattr(q, "query", "") or "")
        qtxt = _clean_str(qtxt_raw)
        if service == "openalex":
            qtxt2 = _sanitize_openalex_query(qtxt)
            if qtxt2 != qtxt:
                stats["sanitized_oa"] += 1
            qtxt = qtxt2
        else:
            qtxt2 = _sanitize_semanticscholar_query(qtxt)
            if qtxt2 != qtxt:
                stats["sanitized_s2"] += 1
            qtxt = qtxt2

        if not qtxt:
            stats["dropped_empty"] += 1
            continue

        if bool(STAGEB_QUERY_PLAN_DEDUPE):
            k = qtxt.lower()
            if k in seen:
                stats["dropped_dupe"] += 1
                continue
            seen.add(k)

        cap_in = int(getattr(q, "cap", 0) or 0)
        cap = _cap(service, cap_in)
        if cap != cap_in:
            stats["fixed_caps"] += 1

        f = getattr(q, "filters", None) or QueryFiltersV2()
        year_min = getattr(f, "year_min", None)
        year_max = getattr(f, "year_max", None)
        if use_no_year:
            if year_min is not None or year_max is not None:
                stats["fixed_years"] += 1
            year_min = None
            year_max = None
        else:
            try:
                ymin = int(year_min) if year_min is not None else int(y.soft_min_year)
                ymax = int(year_max) if year_max is not None else int(y.soft_max_year)
            except Exception:
                ymin = int(y.soft_min_year)
                ymax = int(y.soft_max_year)
                stats["fixed_years"] += 1
            if ymin > ymax:
                ymin, ymax = ymax, ymin
                stats["fixed_years"] += 1
            ymin2 = max(int(y.soft_min_year), int(ymin))
            ymax2 = min(int(y.soft_max_year), int(ymax))
            if ymin2 != ymin or ymax2 != ymax:
                stats["fixed_years"] += 1
            year_min, year_max = ymin2, ymax2

        filters = QueryFiltersV2(
            year_min=year_min,
            year_max=year_max,
            language=lang,  # type: ignore[arg-type]
            concept_ids=[],
        )
        if getattr(f, "language", None) != lang or list(getattr(f, "concept_ids", []) or []):
            stats["fixed_filters"] += 1

        out.append(
            QueryObjDraftV1(
                kind=kind,
                language=lang,
                cap=cap,
                use_no_year=use_no_year,
                facet_id=facet_id,
                query=qtxt,
                filters=filters,
                notes=_clean_str(getattr(q, "notes", "") or ""),
            )
        )

        if want_total and bool(STAGEB_QUERY_PLAN_TRUNCATE_TO_TARGET) and len(out) >= want_total:
            break

    try:
        draft.queries = out
    except Exception:
        if service == "openalex":
            draft = OpenAlexQueryPlanDraftV1(chapter_id=str(chapter_id), queries=out)
        else:
            draft = SemanticScholarQueryPlanDraftV1(chapter_id=str(chapter_id), queries=out)
    return draft, stats


def _drafts_to_query_plan_v2(
    chapter_id: str,
    bp: BlueprintV3,
    y: YearBoundsV2,
    oa: OpenAlexQueryPlanDraftV1,
    s2: SemanticScholarQueryPlanDraftV1,
) -> QueryPlanV2:
    out: List[QueryObjV2] = []

    def _to_q(service: str, dq: QueryObjDraftV1) -> QueryObjV2:
        f = dq.filters or QueryFiltersV2()
        filters = QueryFiltersV2(
            year_min=f.year_min,
            year_max=f.year_max,
            language=dq.language,  # type: ignore[arg-type]
            concept_ids=[],
        )
        facet_id = '' if dq.kind == 'authority' else str(dq.facet_id or '')
        return QueryObjV2(
            id='',
            service=service,  # type: ignore[arg-type]
            kind=dq.kind,  # type: ignore[arg-type]
            language=dq.language,  # type: ignore[arg-type]
            cap=_cap(service, int(dq.cap or 0)),
            use_no_year=bool(dq.use_no_year),
            facet_id=facet_id,
            query=_clean_str(dq.query),
            filters=filters,
            notes=_clean_str(getattr(dq, 'notes', '') or ''),
        )

    for dq in list(getattr(oa, 'queries', []) or []):
        out.append(_to_q('openalex', dq))
    for dq in list(getattr(s2, 'queries', []) or []):
        out.append(_to_q('semanticscholar', dq))

    for i, q in enumerate(out, start=1):
        q.id = f'Q{i:03d}'

    plan = QueryPlanV2(chapter_id=chapter_id, queries=out)
    return _compute_guarantees(bp, y, plan)


async def get_or_create_query_plan_service_agents_v1(
    ch: Dict[str, Any],
    bp: BlueprintV3,
    y: YearBoundsV2,
) -> Tuple[QueryPlanV2, Dict[str, Any]]:
    cid = str(ch.get('chapter_id'))
    p = STAGEB_QUERY_PLAN_DIR / f'{cid}.json'
    if p.exists() and not STAGEB_FORCE_ALL:
        try:
            raw = json.loads(p.read_text(encoding='utf-8'))
        except Exception as e:
            raw = None
            if bool(STAGEB_LOG_CACHE_INVALID):
                print(
                    f'[StageB:{cid}] query_plan_service_agents_v1 cache invalid (read/parse error) -> rebuilding: {e}'
                )
        meta = raw.get('_meta') if isinstance(raw, dict) else {}
        if str((meta or {}).get('prompt_version') or '') == STAGEB_QUERY_PLAN_PROMPT_VERSION:
            try:
                plan = QueryPlanV2.model_validate(raw)
            except Exception as e:
                if bool(STAGEB_LOG_CACHE_INVALID):
                    print(
                        f'[StageB:{cid}] query_plan_service_agents_v1 cache invalid (schema error) -> rebuilding: {e}'
                    )
            else:
                if bool(STAGEB_LOG_CACHE_HITS):
                    print(f'[StageB:{cid}] query_plan_service_agents_v1 cache hit')
                return plan, _zero_usage()
        elif bool(STAGEB_LOG_CACHE_INVALID):
            print(f'[StageB:{cid}] query_plan_service_agents_v1 cache invalid (prompt_version mismatch) -> rebuilding')

    base_services = {'openalex': 50, 'semanticscholar': 50}
    base_languages = {'en': 50, 'de': 50}
    base_kinds = {'anchor': 50, 'facet': 30, 'proxy': 15, 'authority': 5}

    total = int(STAGEB_QUERY_TARGET)
    services = _scale_counts(base_services, total)
    lang_total = _scale_counts(base_languages, total)
    kinds_total = _scale_counts(base_kinds, total)

    oa_n = int(services.get('openalex', 0) or 0)
    s2_n = int(services.get('semanticscholar', 0) or 0)
    if oa_n + s2_n != total:
        oa_n = max(0, total - s2_n)

    oa_lang, s2_lang = _split_counts_two_services(lang_total, oa_n, s2_n)
    oa_kinds, s2_kinds = _split_counts_two_services(kinds_total, oa_n, s2_n)

    targets_oa = {
        'queries_total': oa_n,
        'languages': oa_lang,
        'kinds': oa_kinds,
    }
    targets_s2 = {
        'queries_total': s2_n,
        'languages': s2_lang,
        'kinds': s2_kinds,
    }

    base_inp = {
        'chapter_id': cid,
        'blueprint': bp.model_dump(by_alias=True),
        'year_policy': {
            'soft_min_year': int(y.soft_min_year),
            'soft_max_year': int(y.soft_max_year),
            'no_year_share': float(y.no_year_share),
            'disambiguator_strength': str(y.disambiguator_strength),
        },
    }
    inp_oa = dict(base_inp)
    inp_oa['targets'] = targets_oa
    inp_s2 = dict(base_inp)
    inp_s2['targets'] = targets_s2

    usage_total = _zero_usage()

    def _base_prompts() -> tuple[str, str]:
        return (
            _prompt_with_input(PROMPT_OPENALEX_QUERY_PLAN_V1_USER, inp_oa),
            _prompt_with_input(PROMPT_SEMANTICSCHOLAR_QUERY_PLAN_V1_USER, inp_s2),
        )

    prompt_oa, prompt_s2 = _base_prompts()
    last_errs_oa: List[str] = []
    last_errs_s2: List[str] = []

    oa_draft: Optional[OpenAlexQueryPlanDraftV1] = None
    s2_draft: Optional[SemanticScholarQueryPlanDraftV1] = None
    oa_stats: Dict[str, int] = {}
    s2_stats: Dict[str, int] = {}

    for attempt in range(1, int(STAGEB_QUERY_PLAN_MAX_ATTEMPTS) + 1):
        need_oa = oa_draft is None
        need_s2 = s2_draft is None
        tasks: Dict[str, Any] = {}
        if need_oa:
            tasks["openalex"] = _run_agent(openalex_query_plan_v1_agent, prompt_oa)
        if need_s2:
            tasks["semanticscholar"] = _run_agent(semanticscholar_query_plan_v1_agent, prompt_s2)
        results: Dict[str, Any] = {}
        if tasks:
            done = await asyncio.gather(*tasks.values(), return_exceptions=True)
            for k, v in zip(list(tasks.keys()), done):
                results[k] = v

        if "openalex" in results:
            r_oa = results["openalex"]
            oa_out, u_oa = None, _zero_usage()
            if isinstance(r_oa, Exception):
                oa_draft = None
                last_errs_oa = [f"agent error: {r_oa}"]
            else:
                oa_out, u_oa = r_oa
                try:
                    oa_draft = (
                        oa_out
                        if isinstance(oa_out, OpenAlexQueryPlanDraftV1)
                        else OpenAlexQueryPlanDraftV1.model_validate(oa_out)
                    )
                except Exception as e:
                    oa_draft = None
                    last_errs_oa = [f"schema error: {e}"]
                else:
                    oa_draft, oa_stats = _autofix_service_draft_v1(
                        oa_draft, service="openalex", chapter_id=cid, bp=bp, y=y, targets=targets_oa
                    )
                _acc_usage(usage_total, u_oa)

        if "semanticscholar" in results:
            r_s2 = results["semanticscholar"]
            s2_out, u_s2 = None, _zero_usage()
            if isinstance(r_s2, Exception):
                s2_draft = None
                last_errs_s2 = [f"agent error: {r_s2}"]
            else:
                s2_out, u_s2 = r_s2
                try:
                    s2_draft = (
                        s2_out
                        if isinstance(s2_out, SemanticScholarQueryPlanDraftV1)
                        else SemanticScholarQueryPlanDraftV1.model_validate(s2_out)
                    )
                except Exception as e:
                    s2_draft = None
                    last_errs_s2 = [f"schema error: {e}"]
                else:
                    s2_draft, s2_stats = _autofix_service_draft_v1(
                        s2_draft, service="semanticscholar", chapter_id=cid, bp=bp, y=y, targets=targets_s2
                    )
                _acc_usage(usage_total, u_s2)

        errs_oa = (
            _validate_service_draft_v1(oa_draft, service="openalex", chapter_id=cid, bp=bp, y=y, targets=targets_oa)
            if oa_draft is not None
            else last_errs_oa
        )
        errs_s2 = (
            _validate_service_draft_v1(
                s2_draft, service="semanticscholar", chapter_id=cid, bp=bp, y=y, targets=targets_s2
            )
            if s2_draft is not None
            else last_errs_s2
        )
        last_errs_oa = list(errs_oa or [])
        last_errs_s2 = list(errs_s2 or [])

        oa_final = oa_draft if oa_draft is not None else OpenAlexQueryPlanDraftV1(chapter_id=cid, queries=[])
        s2_final = s2_draft if s2_draft is not None else SemanticScholarQueryPlanDraftV1(chapter_id=cid, queries=[])
        total_q = len(list(getattr(oa_final, "queries", []) or [])) + len(list(getattr(s2_final, "queries", []) or []))

        if total_q >= int(STAGEB_QUERY_PLAN_MIN_TOTAL_QUERIES):
            plan = _drafts_to_query_plan_v2(cid, bp, y, oa_final, s2_final)
            if len(plan.queries) != total:
                print(
                    f"[StageB:{cid}] query_plan_service_agents_v1: using {len(plan.queries)}/{total} queries (oa={len(oa_final.queries)} s2={len(s2_final.queries)})"
                )
            obj = plan.model_dump()
            obj['_meta'] = {
                'stage': 'query_plan_service_agents_v1',
                'model': BLUEPRINT_MODEL,
                'variant': STAGEB_FINAL_VARIANT,
                'created_at_utc': datetime.now(timezone.utc).isoformat(),
                'prompt_version': STAGEB_QUERY_PLAN_PROMPT_VERSION,
                'plan_variant': STAGEB_QUERY_PLAN_VARIANT,
                'target_total': int(total),
                'actual_total': int(len(plan.queries)),
                'actual_by_service': {
                    'openalex': int(len(oa_final.queries)),
                    'semanticscholar': int(len(s2_final.queries)),
                },
                'autofix': {'openalex': oa_stats, 'semanticscholar': s2_stats},
                'draft_errors': {'openalex': errs_oa[:20], 'semanticscholar': errs_s2[:20]},
            }
            p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
            return plan, usage_total

        if attempt >= int(STAGEB_QUERY_PLAN_MAX_ATTEMPTS):
            break

        # Retry only if output is unusable (too few queries). Prefer rerunning only the service(s) that produced no usable queries.
        rerun_oa = (oa_draft is None) or (len(list(getattr(oa_final, "queries", []) or [])) == 0)
        rerun_s2 = (s2_draft is None) or (len(list(getattr(s2_final, "queries", []) or [])) == 0)
        if not rerun_oa and not rerun_s2:
            rerun_oa = True
            rerun_s2 = True

        if rerun_oa:
            oa_draft = None
            prompt_oa, _ = _base_prompts()
            if errs_oa:
                prompt_oa = prompt_oa + f" FIX: {' | '.join([str(x) for x in errs_oa[:20]])}. Return corrected JSON only."
        if rerun_s2:
            s2_draft = None
            _, prompt_s2 = _base_prompts()
            if errs_s2:
                prompt_s2 = prompt_s2 + f" FIX: {' | '.join([str(x) for x in errs_s2[:20]])}. Return corrected JSON only."

        print(
            f"[StageB:{cid}] query_plan_service_agents_v1 attempt {attempt} produced only {total_q}/{total} usable queries; retrying (oa_errs={len(errs_oa)} s2_errs={len(errs_s2)})"
        )

    raise RuntimeError(
        f"[StageB:{cid}] query_plan_service_agents_v1 failed after {STAGEB_QUERY_PLAN_MAX_ATTEMPTS} attempts. OpenAlex errors: {last_errs_oa}. Semantic Scholar errors: {last_errs_s2}"
    )


async def get_or_create_query_plan_v2(ch: Dict[str, Any], bp: BlueprintV3, y: YearBoundsV2) -> Tuple[QueryPlanV2, Dict[str, Any]]:
    # Backward-compatible alias (old name) for notebooks/cells that still call get_or_create_query_plan_v2.
    return await get_or_create_query_plan_service_agents_v1(ch, bp, y)


def _compat_blueprint(bp: BlueprintV3, y: YearBoundsV2, plan: QueryPlanV2, plan_path: str) -> ChapterBlueprint:
    dis = _uniq(bp.anchors.disambiguators.en + bp.anchors.disambiguators.de, max_items=12)
    mech = _uniq(bp.anchors.mechanisms.en + bp.anchors.mechanisms.de, max_items=16)
    ev = _uniq(bp.anchors.evidence_proxies.en + bp.anchors.evidence_proxies.de, max_items=16)
    neg = _uniq(
        bp.negative_pool.en
        + bp.negative_pool.de
        + [n for f in bp.facets for n in (f.safe_negatives + f.risky_negatives)],
        max_items=20,
    )
    facet_terms = _uniq([t for f in bp.facets for t in (f.terms_en + f.terms_de)], max_items=40)

    # Build a compact "main query" for Stage C query-text scoring.
    main_q = ""
    for q in plan.queries:
        if q.service == "openalex" and q.kind == "anchor":
            main_q = q.query
            break
    if not main_q:
        main_q = _oa_expr(dis, mech, ev, neg[:2])

    facet_queries = []
    for f in bp.facets[:12]:
        qtxt = _oa_expr(
            dis,
            f.terms_en + f.terms_de,
            f.evidence_proxies_en + f.evidence_proxies_de,
            f.safe_negatives,
        )
        facet_queries.append(qtxt)
    facet_queries = _ensure_len(facet_queries, min_len=8, max_len=14, fallback=[main_q])

    keywords = _uniq(dis + mech + ev + facet_terms + bp.must_cover + bp.should_cover, max_items=60)
    key_concepts = _uniq(bp.anchors.high_precision.en + bp.anchors.high_precision.de + dis, max_items=24)

    scope_statement = "In-scope: " + "; ".join(_uniq(bp.scope.in_, max_items=8)) + " | Out-of-scope: " + "; ".join(_uniq(bp.scope.out, max_items=8))
    scoring_guidance = "Prefer works that match disambiguators + mechanisms + evidence proxies; penalize negatives and out-of-scope signals."

    return ChapterBlueprint(
        chapter_id=bp.chapter_id,
        language="en",
        scope_statement=scope_statement,
        must_cover=bp.must_cover,
        should_cover=bp.should_cover,
        must_avoid=bp.must_avoid,
        main_query=main_q,
        facet_queries=facet_queries,
        keywords=keywords,
        key_concepts=key_concepts,
        preferred_source_types=None,
        negative_query_terms=neg,
        scoring_guidance=scoring_guidance,
        notes="BlueprintV3 compatibility object for Stage C/C3.",
        anchors_en=_uniq(bp.anchors.disambiguators.en + bp.anchors.mechanisms.en + bp.anchors.evidence_proxies.en, max_items=20),
        anchors_de=_uniq(bp.anchors.disambiguators.de + bp.anchors.mechanisms.de + bp.anchors.evidence_proxies.de, max_items=20),
        facet_blocks=[f.model_dump() for f in bp.facets],
        year_bounds={
            "soft_min_year": y.soft_min_year,
            "soft_max_year": y.soft_max_year,
            "no_year_share": y.no_year_share,
            "disambiguator_strength": y.disambiguator_strength,
        },
        query_plan_size=len(plan.queries),
        query_plan_path=str(plan_path),
    )


# -----------------------------
# Run Stage B for all chapters
# -----------------------------

blueprint_v3_by_chapter: Dict[str, BlueprintV3] = {}
year_bounds_v2_by_chapter: Dict[str, YearBoundsV2] = {}
query_plans_by_chapter: Dict[str, QueryPlanV2] = {}

blueprints: Dict[str, ChapterBlueprint] = {}

bp_stage_costs = {
    "blueprint_v3": _zero_usage(),
    "year_bounds_v2": _zero_usage(),
    "query_plan_service_agents_v1": _zero_usage(),
}
bp_totals = _zero_usage()

for ch in CHAPTERS:
    cid = str(ch.get("chapter_id"))

    try:
        bp3, u1 = await get_or_create_blueprint_v3(ch)
        yb, u2 = await get_or_create_year_bounds_v2(ch, bp3)
        plan, u3 = await get_or_create_query_plan_v2(ch, bp3, yb)
    except Exception as e:
        # No deterministic fallback for query planning in this variant.
        raise RuntimeError(f"[StageB:{cid}] ERROR: Stage B failed (LLM-only query planning): {e}") from e

    blueprint_v3_by_chapter[cid] = bp3
    year_bounds_v2_by_chapter[cid] = yb
    query_plans_by_chapter[cid] = plan

    blueprints[cid] = _compat_blueprint(bp3, yb, plan, str(STAGEB_QUERY_PLAN_DIR / f"{cid}.json"))

    _acc_usage(bp_stage_costs["blueprint_v3"], u1)
    _acc_usage(bp_stage_costs["year_bounds_v2"], u2)
    _acc_usage(bp_stage_costs["query_plan_service_agents_v1"], u3)

for k in ["blueprint_v3", "year_bounds_v2", "query_plan_service_agents_v1"]:
    _acc_usage(bp_totals, bp_stage_costs[k])


# -----------------------------
# Summary output
# -----------------------------

rows_bp = []
for ch in CHAPTERS:
    cid = str(ch.get("chapter_id"))
    bp = blueprint_v3_by_chapter[cid]
    y = year_bounds_v2_by_chapter[cid]
    p = query_plans_by_chapter[cid]

    rows_bp.append(
        {
            "chapter_id": cid,
            "facets": len(bp.facets),
            "dis_en": len(bp.anchors.disambiguators.en),
            "dis_de": len(bp.anchors.disambiguators.de),
            "mech_en": len(bp.anchors.mechanisms.en),
            "ev_en": len(bp.anchors.evidence_proxies.en),
            "queries_total": len(p.queries),
            "queries_oa": sum(1 for q in p.queries if q.service == "openalex"),
            "queries_s2": sum(1 for q in p.queries if q.service == "semanticscholar"),
            "no_year": sum(1 for q in p.queries if q.use_no_year),
            "dis_strength": y.disambiguator_strength,
            "dis_share": round(float(p.guarantees.context_gate_coverage_share or 0.0), 3),
            "soft_years": f"{y.soft_min_year}-{y.soft_max_year}",
            "main_query": blueprints[cid].main_query[:120],
        }
    )

print_table(
    rows_bp,
    columns=[
        "chapter_id",
        "facets",
        "dis_en",
        "dis_de",
        "mech_en",
        "ev_en",
        "queries_total",
        "queries_oa",
        "queries_s2",
        "no_year",
        "dis_strength",
        "dis_share",
        "soft_years",
        "main_query",
    ],
    max_rows=200,
)

print("\\nSTAGE B COSTS (USD):")
print_kv(
    {
        "blueprint_v3": round(float(bp_stage_costs["blueprint_v3"]["cost_usd"]), 6),
        "year_bounds_v2": round(float(bp_stage_costs["year_bounds_v2"]["cost_usd"]), 6),
        "query_plan_service_agents_v1": round(float(bp_stage_costs["query_plan_service_agents_v1"]["cost_usd"]), 6),
        "total": round(float(bp_totals["cost_usd"]), 6),
    }
)


# -----------------------------
# Stage B details for debugging
# -----------------------------

PRINT_BLUEPRINT_DETAILS = True
PRINT_BLUEPRINT_SEARCH_JSON = False
BLUEPRINT_DETAILS_MAX_CHAPTERS = 20
PRINT_ALL_STAGEB_QUERIES = True
STAGEB_QUERY_PREVIEW_MAX = 200


def _print_numbered(title: str, items: List[str]) -> None:
    xs = _uniq(items)
    print(f"{title} ({len(xs)}):")
    if not xs:
        print("  (none)")
        return
    w = len(str(len(xs)))
    for i, x in enumerate(xs, start=1):
        print(f"  {str(i).rjust(w)}. {x}")


def _facet_rows(bp: BlueprintV3) -> List[Dict[str, Any]]:
    rows = []
    for f in bp.facets:
        rows.append(
            {
                "facet_id": f.facet_id,
                "label_en": f.label_en,
                "label_de": f.label_de,
                "terms_en": ", ".join(_uniq(f.terms_en, max_items=6)),
                "terms_de": ", ".join(_uniq(f.terms_de, max_items=6)),
                "evidence_en": ", ".join(_uniq(f.evidence_proxies_en, max_items=5)),
                "evidence_de": ", ".join(_uniq(f.evidence_proxies_de, max_items=5)),
                "safe_neg": ", ".join(_uniq(f.safe_negatives, max_items=4)),
                "risky_neg": ", ".join(_uniq(f.risky_negatives, max_items=4)),
            }
        )
    return rows


def _query_rows(plan: QueryPlanV2) -> List[Dict[str, Any]]:
    out = []
    for q in plan.queries:
        out.append(
            {
                "id": q.id,
                "service": q.service,
                "kind": q.kind,
                "lang": q.language,
                "cap": q.cap,
                "no_year": q.use_no_year,
                "facet_id": q.facet_id or "(authority)",
                "query": q.query[:160],
            }
        )
    return out


if bool(PRINT_BLUEPRINT_DETAILS):
    for ch in CHAPTERS[: int(BLUEPRINT_DETAILS_MAX_CHAPTERS)]:
        cid = str(ch.get("chapter_id"))
        bp = blueprint_v3_by_chapter[cid]
        y = year_bounds_v2_by_chapter[cid]
        plan = query_plans_by_chapter[cid]

        print("\\n" + "=" * 92)
        print(f"Stage B details: {cid}")
        print("=" * 92)

        print("\\nSCOPE (IN):")
        _print_numbered("in", bp.scope.in_)
        print("\\nSCOPE (OUT):")
        _print_numbered("out", bp.scope.out)

        print("\\nRUBRIC:")
        _print_numbered("must_cover", bp.must_cover)
        _print_numbered("should_cover", bp.should_cover)
        _print_numbered("must_avoid", bp.must_avoid)

        print("\\nANCHORS (disambiguators):")
        _print_numbered("disambiguators.en", bp.anchors.disambiguators.en)
        _print_numbered("disambiguators.de", bp.anchors.disambiguators.de)
        print("\\nANCHORS (mechanisms):")
        _print_numbered("mechanisms.en", bp.anchors.mechanisms.en)
        _print_numbered("mechanisms.de", bp.anchors.mechanisms.de)
        print("\\nANCHORS (evidence_proxies):")
        _print_numbered("evidence_proxies.en", bp.anchors.evidence_proxies.en)
        _print_numbered("evidence_proxies.de", bp.anchors.evidence_proxies.de)
        print("\\nANCHORS (high_precision):")
        _print_numbered("high_precision.en", bp.anchors.high_precision.en)
        _print_numbered("high_precision.de", bp.anchors.high_precision.de)

        print("\\nFACETS:")
        print_table(
            _facet_rows(bp),
            columns=["facet_id", "label_en", "label_de", "terms_en", "terms_de", "evidence_en", "evidence_de", "safe_neg", "risky_neg"],
            max_rows=200,
        )

        qr = _query_rows(plan)
        c_service = Counter([r["service"] for r in qr])
        c_kind = Counter([r["kind"] for r in qr])
        c_lang = Counter([r["lang"] for r in qr])
        print("\\nQUERY PLAN COUNTS:")
        print_kv(
            {
                "service_openalex": c_service.get("openalex", 0),
                "service_semanticscholar": c_service.get("semanticscholar", 0),
                "kind_anchor": c_kind.get("anchor", 0),
                "kind_facet": c_kind.get("facet", 0),
                "kind_proxy": c_kind.get("proxy", 0),
                "kind_authority": c_kind.get("authority", 0),
                "lang_en": c_lang.get("en", 0),
                "lang_de": c_lang.get("de", 0),
                "no_year": sum(1 for q in plan.queries if q.use_no_year),
                "dis_strength": y.disambiguator_strength,
                "dis_coverage_share": round(float(plan.guarantees.context_gate_coverage_share or 0.0), 3),
                "soft_years": f"{y.soft_min_year}-{y.soft_max_year}",
            }
        )

        print("\\nSEARCH ITEMS (all generated queries):")
        print_table(
            qr,
            columns=["id", "service", "kind", "lang", "cap", "no_year", "facet_id", "query"],
            max_rows=(9999 if bool(PRINT_ALL_STAGEB_QUERIES) else int(STAGEB_QUERY_PREVIEW_MAX)),
        )

        if bool(PRINT_BLUEPRINT_SEARCH_JSON):
            print("\\nQUERY_PLAN_JSON:")
            print(json.dumps(plan.model_dump(), ensure_ascii=False, indent=2))



[StageB:fall_of_rome_economy] query_plan_service_agents_v1: using 97/100 queries (oa=49 s2=48)
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen>\nSTAGE B COSTS (USD):
blueprint_v3                 : 0.011551
year_bounds_v2               : 0.002558
query_plan_service_agents_v1 : 0.035305
total                        : 0.049414
\n============================================================================================
Stage B details: fall_of_rome_economy
\nSCOPE (IN):
in (8):
  1. Economic factors in the Western Roman Empire and late antiquity
  2. Fiscal institutions, taxation, and state revenue systems
  3. Monetary evidence: coinage, hoards, debasement, and circulation
  4. Trade networks, amphorae, and transport costs within western provinces
  5. Agricultural production, land tenure, ruralization and settlement change
  6. Connections between economic change a

## Stage A (production): API retrieval (OpenAlex + Semantic Scholar)

Build the Stage A corpus per chapter (cached to disk).


In [3]:
# -----------------------------
# Stage A: API retrieval (OpenAlex + Semantic Scholar)
# -----------------------------

# This cell writes per-chapter StageA corpora to:
#   final_pipeline/stageA/<chapter_id>/<sig>/stageA_combined.csv

from __future__ import annotations

import os
import re
import json
import time
import random
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import List, Optional, Dict, Any, Iterable

import requests
import numpy as np
import pandas as pd

OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY", "").strip()
if not OPENALEX_API_KEY:
    print("Warning: OPENALEX_API_KEY not set; continuing without it (may be slower / more rate limits).")

S2_API_KEY = os.getenv("SEMANTICSCHOLAR_API_KEY", "").strip()
S2_MIN_INTERVAL = 1.0  # seconds (rate limit: 1 rps)
_S2_LAST_TS = 0.0
if S2_API_KEY:
    print("Semantic Scholar API key detected (SEMANTICSCHOLAR_API_KEY). Using authenticated requests (1 req/sec).")
else:
    print("Note: SEMANTICSCHOLAR_API_KEY not set; Semantic Scholar will use long retry/backoff without a key.")

# -----------------------------
# Stage A knobs
# -----------------------------
FETCH_FORCE = False
RUN_OPENALEX = True
RUN_SEMANTIC_SCHOLAR = True

STAGEA_PRINT_QUERY_PLAN = True
STAGEA_PRINT_QUERY_TABLE_MAX = 260

# OpenAlex concept/topic filtering (used for extra queries + authority pull)
OA_CONCEPT_FILTER_ENABLED = True
OA_CONCEPTS_URL = "https://api.openalex.org/concepts"
OA_CONCEPT_TERMS_MAX = 20
OA_CONCEPTS_PER_TERM = 5
OA_CONCEPT_MAX_IDS = 6
OA_CONCEPT_LEVEL_MAX = 2
OA_CONCEPT_MIN_WORKS = 5000
OA_CONCEPT_FILTER_FOR_OPENALEX = True
OA_CONCEPT_FILTER_SHARE = 0.30  # apply concept filter only to a subset of OpenAlex queries
OA_CONCEPT_FILTER_ALWAYS_KINDS = {"authority"}  # always concept-filter broad pulls

# OpenAlex
OA_BASE_URL = "https://api.openalex.org/works"
OA_PER_PAGE = 100
OA_MAX_WORKS_PER_QUERY = 180
OA_TIMEOUT_SEC = 30
OA_REQUIRE_ABSTRACT = False  # keep recall; penalize missing abstracts later
OA_FILTER_LANGUAGES = True
OA_FILTER_DATES = True
OA_TITLE_ABSTRACT_FILTER_KEY = "title_and_abstract.search"
OA_TITLE_ABSTRACT_FILTER_KEY_NO_STEM = "title_and_abstract.search.no_stem"
OA_USE_NO_STEM = False
OA_AUTHORITY_SORT = "cited_by_count:desc"
OA_QUERY_MODE = "search"  # "search" (recommended) | "title_and_abstract_filter"

# Semantic Scholar
S2_BASE = "https://api.semanticscholar.org/graph/v1"
S2_SEARCH_URL = f"{S2_BASE}/paper/search"
S2_BULK_SEARCH_URL = f"{S2_BASE}/paper/search/bulk"  # unused (kept for backward compatibility)
S2_BATCH_URL  = f"{S2_BASE}/paper/batch"
S2_USE_BULK_SEARCH = False  # always use /paper/search (no custom boolean dialect)
S2_REQUIRE_ABSTRACT_AFTER_BATCH = False  # keep recall; penalize missing abstracts later
S2_ABSTRACT_MIN_CHARS = 50

S2_LIMIT = 100
S2_MAX_PAGES_PER_QUERY = 20
S2_FETCH_ABSTRACTS_VIA_BATCH = True
S2_BATCH_SIZE = 200
S2_CACHE_ENABLED = True
S2_CACHE_TTL_DAYS = 30
S2_VERBOSE = (not bool(S2_API_KEY))

# Robust retries/backoff (important without S2_API_KEY)
S2_TIMEOUT_SEC = 30
S2_REQUEST_MAX_RETRIES = 60
S2_REQUEST_MAX_SECONDS = 900
S2_BACKOFF_INITIAL_SEC = 2.0
S2_BACKOFF_MAX_SEC = 300.0
S2_BACKOFF_JITTER = 0.25
S2_SUCCESS_SLEEP_SEC = 0.2

# Hard time budget per chapter to prevent multi-hour notebook runs.
S2_CHAPTER_MAX_SECONDS = 1200  # 20 minutes (set None to disable)

# Shared Semantic Scholar cache (re-used across notebooks)
S2_CACHE_DIR = SOURCES_WORKSPACE_DIR / "eval_dataset" / "fetch" / "s2_cache"
S2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Stage A: fetch OpenAlex + Semantic Scholar per chapter
# and build per-chapter StageA CSVs
# -----------------------------

# Output files:
# - eval_dataset/datasets/<dataset_tag>/fetch/openalex_<chapter_id>.csv
# - eval_dataset/datasets/<dataset_tag>/fetch/semantic_scholar_<chapter_id>.csv
# - eval_dataset/datasets/<dataset_tag>/stageA/stageA_combined_oa_s2_<chapter_id>.csv
# - eval_dataset/datasets/<dataset_tag>/stageA/stageA_all_chapters.csv

def dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        x = str(x).strip()
        if not x:
            continue
        k = x.lower()
        if k in seen:
            continue
        seen.add(k)
        out.append(x)
    return out

def query_list_for_chapter(bp) -> List[str]:
    if hasattr(bp, "main_query"):
        return dedupe_preserve_order([str(bp.main_query or "")] + list(bp.facet_queries or []))
    if isinstance(bp, dict):
        return dedupe_preserve_order([str(bp.get("main_query") or "")] + list(bp.get("facet_queries") or []))
    return []


def expanded_query_list_for_chapter(_bp) -> List[str]:
    # Kept for backward compatibility with later cells.
    return []


def query_plan_for_chapter(chapter_id: str) -> QueryPlanV2:
    if "query_plans_by_chapter" not in globals() or not isinstance(query_plans_by_chapter, dict):
        raise RuntimeError("query_plans_by_chapter not found. Run Stage B cell first.")
    plan = query_plans_by_chapter.get(chapter_id)
    if plan is None:
        raise KeyError(f"No query plan found for chapter_id={chapter_id}")
    if hasattr(plan, "queries"):
        return plan
    return QueryPlanV2.model_validate(plan)


def openalex_concept_terms_for_chapter(bp) -> List[str]:
    if hasattr(bp, "anchors_en"):
        terms = list(getattr(bp, "anchors_en", []) or []) + list(getattr(bp, "anchors_de", []) or []) + list(getattr(bp, "key_concepts", []) or [])
        return dedupe_preserve_order(terms)[: int(OA_CONCEPT_TERMS_MAX)]
    if isinstance(bp, dict):
        terms = list(bp.get("anchors_en") or []) + list(bp.get("anchors_de") or []) + list(bp.get("key_concepts") or [])
        return dedupe_preserve_order(terms)[: int(OA_CONCEPT_TERMS_MAX)]
    return []


# ----------------------------
# OpenAlex
# ----------------------------

def abstract_from_inverted_index(inv: Optional[Dict[str, List[int]]]) -> Optional[str]:
    if not inv:
        return None

    pairs: List[tuple[int, str]] = []
    for word, positions in inv.items():
        if not positions:
            continue
        for p in positions:
            if isinstance(p, int):
                pairs.append((p, word))

    if not pairs:
        return None

    pairs.sort(key=lambda x: x[0])
    max_pos = pairs[-1][0]
    words = [""] * (max_pos + 1)
    for pos, w in pairs:
        if 0 <= pos <= max_pos:
            words[pos] = w

    text = " ".join(w for w in words if w).strip()
    return text or None

def venue_from_primary_location(work: Dict[str, Any]) -> Optional[str]:
    pl = work.get("primary_location") or {}
    src = pl.get("source") or {}
    return src.get("display_name")

def first_n_authors(work: Dict[str, Any], n: int = 6) -> str:
    authors = []
    for a in (work.get("authorships") or [])[:n]:
        name = ((a.get("author") or {}).get("display_name"))
        if name:
            authors.append(name)
    return "; ".join(authors)

def oa_get(params: Dict[str, Any], max_retries: int = 6) -> dict:
    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        r = requests.get(OA_BASE_URL, params=params, timeout=OA_TIMEOUT_SEC)
        if r.status_code in (429, 500, 502, 503, 504):
            if attempt == max_retries:
                raise RuntimeError(f"OpenAlex error {r.status_code} | URL: {r.url} | Body: {r.text[:400]}")
            time.sleep(backoff)
            backoff *= 2
            continue
        if r.status_code >= 400:
            raise RuntimeError(f"OpenAlex error {r.status_code} | URL: {r.url} | Body: {r.text[:400]}")
        return r.json()
    raise RuntimeError("OpenAlex retry loop exhausted")

def oa_get_url(url: str, params: Dict[str, Any], max_retries: int = 6) -> dict:
    """OpenAlex request helper for endpoints other than /works."""

    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        r = requests.get(url, params=params, timeout=OA_TIMEOUT_SEC)
        if r.status_code in (429, 500, 502, 503, 504):
            if attempt == max_retries:
                raise RuntimeError(f"OpenAlex error {r.status_code} | URL: {r.url} | Body: {r.text[:400]}")
            time.sleep(backoff)
            backoff *= 2
            continue
        if r.status_code >= 400:
            raise RuntimeError(f"OpenAlex error {r.status_code} | URL: {r.url} | Body: {r.text[:400]}")
        return r.json()
    raise RuntimeError("OpenAlex retry loop exhausted")


def _oa_id_short(x: Any) -> Optional[str]:
    s = str(x or "").strip()
    if not s:
        return None
    return s.rsplit("/", 1)[-1]


def openalex_concept_ids_from_terms(terms: List[str]) -> List[str]:
    """Map blueprint terms -> a small set of OpenAlex concept IDs.

    We prefer broader concepts (low level) with enough works to avoid over-filtering.
    """

    if not bool(OA_CONCEPT_FILTER_ENABLED):
        return []

    out: List[str] = []
    for term in (terms or []):
        t = str(term or "").strip()
        if not t:
            continue

        params: Dict[str, Any] = {
            "search": t,
            "per-page": int(OA_CONCEPTS_PER_TERM),
            "select": "id,display_name,level,works_count",
        }
        if OPENALEX_API_KEY:
            params["api_key"] = OPENALEX_API_KEY

        try:
            data = oa_get_url(OA_CONCEPTS_URL, params=params)
        except Exception as e:
            print(f"[OpenAlex:concepts] warning: concept lookup failed for term={t!r}: {e}")
            continue

        for c in (data.get("results") or []):
            lvl = int(c.get("level") or 99)
            wc = int(c.get("works_count") or 0)
            if lvl > int(OA_CONCEPT_LEVEL_MAX):
                continue
            if wc < int(OA_CONCEPT_MIN_WORKS):
                continue
            cid = _oa_id_short(c.get("id"))
            if cid:
                out.append(cid)
                break

        if len(out) >= int(OA_CONCEPT_MAX_IDS):
            break

    out = dedupe_preserve_order(out)
    return out[: int(OA_CONCEPT_MAX_IDS)]


def openalex_concept_filter(concept_ids: List[str]) -> Optional[str]:
    ids = [(_oa_id_short(x) or "") for x in (concept_ids or [])]
    ids = [x for x in ids if x]
    if not ids:
        return None
    # OpenAlex supports OR within the same filter key via `|`.
    return "concept.id:" + "|".join(ids)


def fetch_openalex_query(
    q: Optional[str],
    max_works: Optional[int],
    *,
    filter_: Optional[str] = None,
    sort: Optional[str] = None,
    query_meta: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    cursor = "*"
    select = (
        "id,display_name,publication_year,type,doi,cited_by_count,"
        "authorships,primary_location,abstract_inverted_index"
    )

    q_str = str(q or "").strip()
    meta = query_meta or {}
    query_display = str(meta.get("query") or q_str or "__query__")

    while cursor:
        params: Dict[str, Any] = {
            "per-page": OA_PER_PAGE,
            "cursor": cursor,
            "select": select,
        }
        if q_str:
            params["search"] = q_str
        if filter_:
            params["filter"] = str(filter_)
        if sort:
            params["sort"] = str(sort)
        if OPENALEX_API_KEY:
            params["api_key"] = OPENALEX_API_KEY

        data = oa_get(params)

        for w in data.get("results", []) or []:
            rows.append({
                "query": query_display,
                "query_id": str(meta.get("query_id") or ""),
                "query_kind": str(meta.get("query_kind") or ""),
                "query_service": "openalex",
                "query_language": str(meta.get("query_language") or ""),
                "query_cap": int(meta.get("query_cap", max_works or OA_MAX_WORKS_PER_QUERY) or OA_MAX_WORKS_PER_QUERY),
                "query_use_no_year": bool(meta.get("query_use_no_year", False)),
                "title": w.get("display_name"),
                "year": w.get("publication_year"),
                "type": w.get("type"),
                "venue": venue_from_primary_location(w),
                "cited_by": w.get("cited_by_count"),
                "authors(first6)": first_n_authors(w, n=6),
                "doi": w.get("doi"),
                "openalex_id": w.get("id"),
                "abstract": abstract_from_inverted_index(w.get("abstract_inverted_index")),
            })

            if max_works is not None and len(rows) >= int(max_works):
                return rows

        cursor = (data.get("meta") or {}).get("next_cursor")
        if not cursor:
            break

    return rows

def _openalex_search_filter_key() -> str:
    return OA_TITLE_ABSTRACT_FILTER_KEY_NO_STEM if bool(OA_USE_NO_STEM) else OA_TITLE_ABSTRACT_FILTER_KEY


def _clean_openalex_query(q: str) -> str:
    s = str(q or "")
    s = s.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    # Avoid breaking OpenAlex filter syntax and keep queries robust.
    s = s.replace(",", " ").replace(":", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _should_apply_concept_filter(spec: Dict[str, Any]) -> bool:
    if not bool(OA_CONCEPT_FILTER_ENABLED) or not bool(OA_CONCEPT_FILTER_FOR_OPENALEX):
        return False
    kind = str(spec.get("kind") or "").strip().lower()
    if kind in set([str(x).strip().lower() for x in (OA_CONCEPT_FILTER_ALWAYS_KINDS or set())]):
        return True
    share = float(OA_CONCEPT_FILTER_SHARE) if OA_CONCEPT_FILTER_SHARE is not None else 0.0
    share = max(0.0, min(1.0, share))
    if share <= 0.0:
        return False
    qid = str(spec.get("id") or spec.get("query_id") or "")
    if not qid:
        return False
    h = hashlib.sha1(qid.encode("utf-8")).hexdigest()
    bucket = int(h[:2], 16) / 255.0
    return bucket < share


def _openalex_filters_from_spec(
    spec: Dict[str, Any],
    *,
    base_concept_filter: Optional[str] = None,
) -> tuple[str, bool]:
    parts: List[str] = []

    if bool(OA_REQUIRE_ABSTRACT):
        parts.append("has_abstract:true")

    filters_obj = spec.get("filters") if isinstance(spec.get("filters"), dict) else {}

    if bool(OA_FILTER_LANGUAGES):
        lang = str(filters_obj.get("language") or spec.get("language") or "").strip()
        if lang:
            parts.append("language:" + lang)

    if bool(OA_FILTER_DATES) and (not bool(spec.get("use_no_year", False))):
        ymin = int(filters_obj.get("year_min") or 0)
        ymax = int(filters_obj.get("year_max") or 0)
        if ymin > 0 and ymax > 0 and ymin <= ymax:
            parts.append(f"from_publication_date:{ymin}-01-01")
            parts.append(f"to_publication_date:{ymax}-12-31")

    # Concept filters: prefer per-query concept_ids; otherwise apply the shared concept filter to a subset.
    concept_applied = False
    concept_part: Optional[str] = None
    if isinstance(filters_obj.get("concept_ids"), list) and filters_obj.get("concept_ids"):
        concept_part = openalex_concept_filter(list(filters_obj.get("concept_ids") or []))
        concept_applied = bool(concept_part)
    elif base_concept_filter and _should_apply_concept_filter(spec):
        concept_part = str(base_concept_filter)
        concept_applied = True
    if concept_part:
        parts.append(concept_part)

    return ",".join([p for p in parts if p]), bool(concept_applied)


def fetch_openalex_for_chapter(chapter_id: str, plan: QueryPlanV2, *, concept_filter: Optional[str] = None) -> pd.DataFrame:
    """Fetch OpenAlex using structured query specs from QueryPlanV2."""

    all_rows: List[Dict[str, Any]] = []
    specs = [
        q.model_dump() if hasattr(q, "model_dump") else dict(q)
        for q in (plan.queries or [])
        if str(getattr(q, "service", "") if hasattr(q, "service") else q.get("service", "")).strip() == "openalex"
    ]
    total = int(len(specs))
    for qi, spec in enumerate(specs, start=1):
        qid = str(spec.get("id") or spec.get("query_id") or f"Q{qi:03d}")
        qtxt = _clean_openalex_query(str(spec.get("query") or spec.get("query_string") or "").strip())
        qkind = str(spec.get("kind") or "facet")
        qlang = str(spec.get("language") or "en")
        cap = int(spec.get("cap", OA_MAX_WORKS_PER_QUERY) or OA_MAX_WORKS_PER_QUERY)
        sort = OA_AUTHORITY_SORT if qkind == "authority" else None
        filt, concept_applied = _openalex_filters_from_spec(spec, base_concept_filter=concept_filter)
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>        q_param: Optional[str] = None
        filter_param: Optional[str] = filt
        if str(OA_QUERY_MODE).strip().lower() in ("search", "search_param", "search-parameter"):
            q_param = qtxt
        else:
            # Legacy mode: encode search into filter (very restrictive; typically low recall).
            q_param = None
            if qtxt:
                filter_param = f"{_openalex_search_filter_key()}:{qtxt}" + (("," + filt) if filt else "")
        all_rows.extend(
            fetch_openalex_query(
                q_param,
                max_works=cap,
                filter_=filter_param,
                sort=sort,
                query_meta={
                    "query_id": qid,
                    "query": qtxt,
                    "query_kind": qkind,
                    "query_language": qlang,
                    "query_cap": cap,
                    "query_use_no_year": bool(spec.get("use_no_year", False)),
                    "oa_concept_filter_applied": bool(concept_applied),
                },
            )
        )

    df_oa = pd.DataFrame(all_rows)
    if df_oa.empty:
        return df_oa

    df_oa.insert(0, "chapter_id", chapter_id)
    df_oa = df_oa.drop_duplicates(subset=["chapter_id", "query_id", "openalex_id"], keep="first").reset_index(drop=True)
    return df_oa

# ----------------------------
# Semantic Scholar
# ----------------------------

# Shared Semantic Scholar cache across dataset versions (avoids repeated API calls)
S2_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _now_utc() -> datetime:
    return datetime.now(timezone.utc)

def _cache_key(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]]) -> str:
    blob = {"m": method.upper(), "u": url, "p": params or {}, "b": body or {}}
    s = json.dumps(blob, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha1(s).hexdigest()

def s2_cache_get(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]]) -> Optional[Any]:
    if not S2_CACHE_ENABLED:
        return None
    key = _cache_key(method, url, params, body)
    path = S2_CACHE_DIR / f"{key}.json"
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        created = datetime.fromisoformat(payload["created"])
        if _now_utc() - created > timedelta(days=S2_CACHE_TTL_DAYS):
            return None
        return payload["data"]
    except Exception:
        return None

def s2_cache_set(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]], data: Any) -> None:
    if not S2_CACHE_ENABLED:
        return
    key = _cache_key(method, url, params, body)
    path = S2_CACHE_DIR / f"{key}.json"
    payload = {"created": _now_utc().isoformat(), "data": data}
    path.write_text(json.dumps(payload), encoding="utf-8")

def _parse_retry_after(resp: requests.Response) -> Optional[float]:
    ra = resp.headers.get("Retry-After")
    if not ra:
        return None
    try:
        return float(ra)
    except ValueError:
        return None

def s2_request(
    session: requests.Session,
    method: str,
    url: str,
    params: Optional[Dict[str, Any]] = None,
    body: Optional[Dict[str, Any]] = None,
    max_retries: Optional[int] = None,
    max_elapsed_seconds: Optional[float] = None,
) -> Any:
    """Semantic Scholar request with caching + long retry/backoff.

    Without an API key, S2 can rate-limit aggressively (429). We therefore:
    - retry for a long time (default up to ~1 hour per request)
    - respect Retry-After when present
    - use exponential backoff + jitter
    - treat network exceptions as retryable
    """
    global _S2_LAST_TS

    cached = s2_cache_get(method, url, params, body)
    if cached is not None:
        return cached

    max_retries = int(max_retries if max_retries is not None else S2_REQUEST_MAX_RETRIES)
    max_elapsed_seconds = float(max_elapsed_seconds if max_elapsed_seconds is not None else S2_REQUEST_MAX_SECONDS)

    start = time.monotonic()
    backoff = float(S2_BACKOFF_INITIAL_SEC)
    last_status: Any = None
    last_err: Optional[str] = None

    attempt = 0
    while True:
        attempt += 1
        # rate limit: 1 request/second
        now = time.time()
        wait = S2_MIN_INTERVAL - (now - _S2_LAST_TS)
        if wait > 0:
            time.sleep(wait)
        _S2_LAST_TS = time.time()

        resp: Optional[requests.Response]
        try:
            resp = session.request(method, url, params=params, json=body, timeout=S2_TIMEOUT_SEC)
            last_status = resp.status_code
            last_err = None
        except Exception as e:
            resp = None
            last_status = "exception"
            last_err = repr(e)

        if resp is not None and resp.status_code == 200:
            try:
                data = resp.json()
            except Exception as e:
                last_status = "json_error"
                last_err = repr(e)
            else:
                s2_cache_set(method, url, params, body, data)
                return data

        ra = None
        if resp is None:
            retryable = True
        elif resp.status_code in (429, 500, 502, 503, 504):
            retryable = True
            ra = _parse_retry_after(resp)
        elif resp.status_code in (408,):
            retryable = True
        else:
            raise RuntimeError(f"S2 error {resp.status_code} | body: {resp.text[:500]}")

        elapsed = time.monotonic() - start
        if attempt >= max_retries or elapsed >= max_elapsed_seconds:
            detail = f"last_status={last_status}"
            if last_err:
                detail += f" last_err={last_err}"
            raise RuntimeError(
                f"S2 retry budget exhausted after {elapsed:.0f}s and {attempt} attempts: {method} {url} ({detail})"
            )

        wait = float(backoff)
        if ra is not None:
            wait = max(wait, float(ra))
        wait = min(float(S2_BACKOFF_MAX_SEC), wait)
        if S2_BACKOFF_JITTER:
            jitter = 1.0 + random.uniform(-float(S2_BACKOFF_JITTER), float(S2_BACKOFF_JITTER))
            wait = max(0.0, wait * jitter)

        # Minimum sleep to avoid hammering the API
        wait = max(1.0, wait)

        remaining = max_elapsed_seconds - elapsed
        wait = min(wait, max(0.0, remaining))

        if S2_VERBOSE:
            print(
                f"[S2] status={last_status} retry in {wait:.1f}s | attempt {attempt}/{max_retries} | elapsed {elapsed:.0f}s"
            )
        time.sleep(wait)
        backoff = min(float(S2_BACKOFF_MAX_SEC), backoff * 2)

def clean_query(q: str) -> str:
    # Semantic Scholar paper search: keep queries short and avoid boolean operators/parentheses/wildcards.
    q = str(q or "")
    q = q.replace("\r", " ").replace("\n", " ")
    q = re.sub(r"(?i)\b(AND|OR|NOT)\b", " ", q)
    q = re.sub(r"[][(){}|+*^~]", " ", q)
    q = q.replace(":", " ")
    q = re.sub(r"\s+", " ", q).strip()
    return q[:240].strip()

def chunks(xs: List[str], n: int) -> Iterable[List[str]]:
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def fetch_s2_for_chapter(chapter_id: str, plan: QueryPlanV2, out_csv: Optional[Path] = None) -> pd.DataFrame:
    """Fetch Semantic Scholar using structured query specs (paper search endpoint)."""
    SEARCH_FIELDS = "paperId,title,year,authors,venue,citationCount,externalIds,url"
    DETAIL_FIELDS = "paperId,abstract"
    base_cols = [
        "chapter_id", "query", "query_id", "query_kind", "query_service", "query_language", "query_cap", "query_use_no_year",
        "paperId", "title", "year", "venue", "citationCount", "authors(first6)", "doi", "s2_url",
    ]
    csv_cols = base_cols + ["abstract"]

    session = requests.Session()
    session.headers.update({"User-Agent": "instantpaper-eval/1.0"})
    if S2_API_KEY:
        session.headers.update({"x-api-key": S2_API_KEY})

    specs = [
        q.model_dump() if hasattr(q, "model_dump") else dict(q)
        for q in (plan.queries or [])
        if str(getattr(q, "service", "") if hasattr(q, "service") else q.get("service", "")).strip() == "semanticscholar"
    ]
    progress_path: Optional[Path] = None
    abstracts_cache_path: Optional[Path] = None
    completed_queries: set[str] = set()
    t0 = time.monotonic()
    errors = 0
    seen: set[tuple[str, str]] = set()
    rows: List[Dict[str, Any]] = []

    qsig = hashlib.sha1(json.dumps(specs, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:12]
    if out_csv is not None:
        out_csv = Path(out_csv)
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        progress_path = out_csv.with_suffix(".progress.json")
        abstracts_cache_path = out_csv.with_suffix(".abstracts.json")
        if out_csv.exists():
            try:
                existing = pd.read_csv(out_csv)
                if "abstract" not in existing.columns:
                    existing["abstract"] = np.nan
                    tmp_csv = out_csv.with_suffix(".tmp")
                    existing.to_csv(tmp_csv, index=False, encoding="utf-8")
                    tmp_csv.replace(out_csv)
                for q0, pid0 in zip(existing.get("query_id", []), existing.get("paperId", [])):
                    if pd.isna(q0) or pd.isna(pid0):
                        continue
                    seen.add((str(q0), str(pid0)))
            except Exception:
                pass
        if progress_path.exists():
            try:
                payload = json.loads(progress_path.read_text(encoding="utf-8"))
                if payload.get("query_specs_sha1_12") == qsig:
                    completed_queries = set(payload.get("completed_query_ids", []) or [])
            except Exception:
                pass

    def save_progress() -> None:
        if progress_path is None:
            return
        payload = {
            "chapter_id": chapter_id,
            "query_specs_sha1_12": qsig,
            "completed_query_ids": sorted(completed_queries),
            "n_completed": int(len(completed_queries)),
            "n_total": int(len(specs)),
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        progress_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    def append_rows(rows_new: List[Dict[str, Any]]) -> None:
        if not rows_new:
            return
        if out_csv is None:
            rows.extend(rows_new)
            return
        df_new = pd.DataFrame(rows_new)
        for c in csv_cols:
            if c not in df_new.columns:
                df_new[c] = np.nan
        df_new = df_new[csv_cols]
        header = (not out_csv.exists()) or (out_csv.stat().st_size == 0)
        df_new.to_csv(out_csv, mode="a", index=False, header=header, encoding="utf-8")

    def _search_one(spec: Dict[str, Any]) -> List[Dict[str, Any]]:
        q_rows: List[Dict[str, Any]] = []
        query_text = clean_query(str(spec.get("query") or spec.get("query_string") or ""))
        qid = str(spec.get("id") or spec.get("query_id") or "")
        qkind = str(spec.get("kind") or "facet")
        qlang = str(spec.get("language") or "en")
        qcap = int(spec.get("cap", S2_LIMIT) or S2_LIMIT)
        use_no_year = bool(spec.get("use_no_year", False))
        year_filter = None
        if not use_no_year:
            fobj = spec.get("filters") if isinstance(spec.get("filters"), dict) else {}
            ymin = int(fobj.get("year_min") or 0)
            ymax = int(fobj.get("year_max") or 0)
            if ymin > 0 and ymax > 0 and ymin <= ymax:
                year_filter = f"{ymin}-{ymax}"

        token = None
        offset = 0
        use_bulk = False
        pages = 0
        while pages < int(S2_MAX_PAGES_PER_QUERY) and len(q_rows) < qcap:
            pages += 1
            params = {"query": query_text, "fields": SEARCH_FIELDS, "limit": min(int(S2_LIMIT), max(1, qcap - len(q_rows)))}
            if qkind == "authority":
                params["sort"] = "citationCount:desc"
            if year_filter:
                params["year"] = year_filter
            if use_bulk:
                if token:
                    params["token"] = token
                url = S2_BULK_SEARCH_URL
            else:
                params["offset"] = offset
                url = S2_SEARCH_URL

            try:
                data = s2_request(session, "GET", url, params=params, body=None)
            except Exception as e:
                # Fallback once to non-bulk if bulk endpoint is unavailable.
                if use_bulk:
                    use_bulk = False
                    token = None
                    offset = 0
                    pages = 0
                    continue
                raise e

            time.sleep(S2_SUCCESS_SLEEP_SEC)
            papers = data.get("data", []) or []
            for p in papers:
                pid = p.get("paperId")
                if not pid:
                    continue
                key = (qid, str(pid))
                if key in seen:
                    continue
                seen.add(key)
                authors = [a.get("name") for a in (p.get("authors") or [])[:6] if a.get("name")]
                ext = p.get("externalIds") or {}
                q_rows.append({
                    "chapter_id": chapter_id,
                    "query": query_text,
                    "query_id": qid,
                    "query_kind": qkind,
                    "query_service": "semanticscholar",
                    "query_language": qlang,
                    "query_cap": qcap,
                    "query_use_no_year": use_no_year,
                    "paperId": pid,
                    "title": p.get("title"),
                    "year": p.get("year"),
                    "venue": p.get("venue"),
                    "citationCount": p.get("citationCount"),
                    "authors(first6)": "; ".join(authors),
                    "doi": ext.get("DOI"),
                    "s2_url": p.get("url"),
                })
                if len(q_rows) >= qcap:
                    break

            if use_bulk:
                token = data.get("token")
                if not token or len(papers) == 0:
                    break
            else:
                if len(papers) < int(params["limit"]):
                    break
                offset += int(params["limit"])

        return q_rows

    for i, spec in enumerate(specs, start=1):
        if (S2_CHAPTER_MAX_SECONDS is not None) and (time.monotonic() - t0 > float(S2_CHAPTER_MAX_SECONDS)):
            print(f"[S2:{chapter_id}] time budget reached ({S2_CHAPTER_MAX_SECONDS}s) -> stopping early")
            break
        qid = str(spec.get("id") or spec.get("query_id") or f"Q{i:03d}")
        if qid in completed_queries:
            continue
        qkind = str(spec.get("kind") or "facet")
        qlang = str(spec.get("language") or "en")
        print(f"[S2:{chapter_id}] ({i}/{len(specs)}) {qid} | {qkind}/{qlang} | cap={int(spec.get('cap', S2_LIMIT) or S2_LIMIT)} | no_year={bool(spec.get('use_no_year', False))}")
        try:
            append_rows(_search_one(spec))
        except Exception as e:
            errors += 1
            print(f"[S2:{chapter_id}] ERROR: query {qid} failed (skipping): {e}")
        completed_queries.add(qid)
        save_progress()

    df_s2 = pd.read_csv(out_csv) if (out_csv is not None and out_csv.exists()) else pd.DataFrame(rows)
    if df_s2.empty:
        return df_s2

    df_s2 = df_s2.drop_duplicates(subset=["chapter_id", "query_id", "paperId"]).reset_index(drop=True)

    if S2_FETCH_ABSTRACTS_VIA_BATCH:
        abstracts: Dict[str, Optional[str]] = {}
        if abstracts_cache_path is not None and abstracts_cache_path.exists():
            try:
                abstracts.update(json.loads(abstracts_cache_path.read_text(encoding="utf-8")))
            except Exception:
                pass
        if "abstract" in df_s2.columns:
            for pid, abs_ in zip(df_s2.get("paperId", []), df_s2.get("abstract", [])):
                if pd.isna(pid) or pd.isna(abs_):
                    continue
                abstracts[str(pid)] = str(abs_)
        unique_ids = [str(pid) for pid in df_s2["paperId"].dropna().unique().tolist() if pid]
        missing_ids = [pid for pid in unique_ids if pid not in abstracts]
        if missing_ids:
            print(f"[S2:{chapter_id}] fetching abstracts via batch | missing={len(missing_ids)}/{len(unique_ids)}")
        for ids in chunks(missing_ids, S2_BATCH_SIZE):
            if (S2_CHAPTER_MAX_SECONDS is not None) and (time.monotonic() - t0 > float(S2_CHAPTER_MAX_SECONDS)):
                print(f"[S2:{chapter_id}] time budget reached during abstract fetch ({S2_CHAPTER_MAX_SECONDS}s) -> stopping early")
                break
            params = {"fields": DETAIL_FIELDS}
            body = {"ids": ids}
            try:
                batch = s2_request(session, "POST", S2_BATCH_URL, params=params, body=body)
            except Exception as e:
                errors += 1
                print(f"[S2:{chapter_id}] ERROR: abstract batch failed (skipping chunk): {e}")
                continue
            time.sleep(S2_SUCCESS_SLEEP_SEC)
            it = batch if isinstance(batch, list) else (batch.get("data", []) or [])
            for p in it:
                pid = p.get("paperId")
                if pid:
                    abstracts[str(pid)] = p.get("abstract")
            if abstracts_cache_path is not None:
                tmp = abstracts_cache_path.with_suffix(".tmp")
                tmp.write_text(json.dumps(abstracts, ensure_ascii=False), encoding="utf-8")
                tmp.replace(abstracts_cache_path)
        df_s2["abstract"] = df_s2["paperId"].astype(str).map(abstracts)

        if bool(S2_REQUIRE_ABSTRACT_AFTER_BATCH):
            before = int(len(df_s2))
            abs_len = df_s2["abstract"].fillna("").astype(str).str.len()
            df_s2 = df_s2[abs_len > int(S2_ABSTRACT_MIN_CHARS)].copy()
            print(f"[S2:{chapter_id}] require_abstract: kept {len(df_s2)}/{before} rows (min_chars={S2_ABSTRACT_MIN_CHARS})")

    if out_csv is not None:
        tmp_csv = out_csv.with_suffix(".tmp")
        df_s2.to_csv(tmp_csv, index=False, encoding="utf-8")
        tmp_csv.replace(out_csv)
    if errors:
        print(f"[S2:{chapter_id}] done with {errors} errors (see logs above)")
    return df_s2

# ----------------------------
# Stage A merge (OpenAlex + S2) per chapter
# ----------------------------

def standardize_openalex(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d["source"] = "openalex"
    d["source_id"] = d.get("openalex_id")
    d["citation_count"] = d.get("cited_by")
    for col in [
        "abstract", "doi", "title", "year", "venue", "type", "authors(first6)", "query",
        "query_id", "query_kind", "query_service", "query_language", "query_cap", "query_use_no_year",
        "chapter_id", "openalex_id",
    ]:
        if col not in d.columns:
            d[col] = np.nan
    return d[[
        "chapter_id",
        "source", "source_id", "query", "query_id", "query_kind", "query_service", "query_language", "query_cap", "query_use_no_year",
        "title", "year", "venue", "type", "authors(first6)", "doi", "citation_count", "abstract", "openalex_id"
    ]]

def standardize_s2(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d["source"] = "semantic_scholar"
    d["source_id"] = d.get("paperId")
    d["citation_count"] = d.get("citationCount")
    for col in [
        "abstract", "doi", "title", "year", "venue", "authors(first6)", "query",
        "query_id", "query_kind", "query_service", "query_language", "query_cap", "query_use_no_year",
        "paperId", "s2_url", "chapter_id",
    ]:
        if col not in d.columns:
            d[col] = np.nan
    d["type"] = np.nan
    d["openalex_id"] = np.nan
    return d[[
        "chapter_id",
        "source", "source_id", "query", "query_id", "query_kind", "query_service", "query_language", "query_cap", "query_use_no_year",
        "title", "year", "venue", "type", "authors(first6)", "doi", "citation_count", "abstract", "paperId", "s2_url"
    ]]

def normalize_doi(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    x = re.sub(r"^https?://(dx\.)?doi\.org/", "", x)
    x = re.sub(r"^doi:\s*", "", x)
    x = x.strip()
    return x or None

def normalize_title(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    x = re.sub(r"[\u2010-\u2015]", "-", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x or None

def longest_text(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    return max(vals, key=len)

def most_common_or_longest(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    vc = pd.Series(vals).value_counts()
    if len(vc) and vc.iloc[0] >= 2:
        return vc.index[0]
    return max(vals, key=len)

def first_nonnull(series):
    for v in series:
        if pd.notna(v) and v not in ("", None):
            return v
    return None

def within_source_key(df_in: pd.DataFrame) -> pd.Series:
    k = []
    for _, r in df_in.iterrows():
        if r.get("doi_norm"):
            k.append(f"doi:{r['doi_norm']}")
        elif pd.notna(r.get("source_id")):
            k.append(f"id:{r['source']}:{r['source_id']}")
        else:
            y = int(r["year"]) if pd.notna(r.get("year")) else ""
            k.append(f"ty:{r['title_norm']}|{y}")
    return pd.Series(k, index=df_in.index)

def cross_source_merge_key(r: pd.Series) -> str:
    if isinstance(r.get("doi_norm"), str) and r.get("doi_norm"):
        return f"doi:{r['doi_norm']}"
    if pd.notna(r.get("year")):
        return f"ty:{r['title_norm']}|{int(round(float(r['year'])))}"
    return f"t:{r['title_norm']}"

def split_semi_unique(values: Iterable[object]) -> List[str]:
    out: List[str] = []
    seen: set[str] = set()
    for v in values or []:
        if pd.isna(v):
            continue
        s = str(v)
        for part in s.split(";"):
            p = str(part).strip()
            if not p:
                continue
            k = p.lower()
            if k in seen:
                continue
            seen.add(k)
            out.append(p)
    return out


def build_stagea(df_oa_raw: pd.DataFrame, df_s2_raw: pd.DataFrame, chapter_id: str) -> pd.DataFrame:
    oa_std = standardize_openalex(df_oa_raw)
    s2_std = standardize_s2(df_s2_raw)

    combined_raw = pd.concat([oa_std, s2_std], ignore_index=True)
    if combined_raw.empty:
        return pd.DataFrame(columns=[
            "chapter_id", "merge_key", "sources", "source_count", "source_ids",
            "title", "year", "venue", "type", "authors(first6)", "doi", "doi_norm",
            "citation_count_max", "abstract", "queries", "query_ids", "query_kinds",
            "query_services", "query_languages", "openalex_id", "paperId", "s2_url",
            "merge_kind", "has_abstract",
        ])
    stageA = combined_raw.copy()

    stageA["doi_norm"] = stageA["doi"].map(normalize_doi)
    stageA["title_norm"] = stageA["title"].map(normalize_title)
    stageA["citation_count"] = pd.to_numeric(stageA["citation_count"], errors="coerce")

    stageA = stageA[stageA["title_norm"].notna()].reset_index(drop=True)
    stageA["within_key"] = within_source_key(stageA)

    agg_map = {
        "chapter_id": lambda s: s.iloc[0],
        "source": lambda s: s.iloc[0],
        "source_id": first_nonnull,
        "title": most_common_or_longest,
        "title_norm": lambda s: s.iloc[0],
        "year": lambda s: pd.to_numeric(s, errors="coerce").dropna().median() if s.notna().any() else np.nan,
        "venue": most_common_or_longest,
        "type": most_common_or_longest,
        "authors(first6)": most_common_or_longest,
        "doi": first_nonnull,
        "doi_norm": first_nonnull,
        "citation_count": lambda s: pd.to_numeric(s, errors="coerce").max(),
        "abstract": longest_text,
        "query": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()]))),
        "query_id": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()]))),
        "query_kind": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()]))),
        "query_service": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()]))),
        "query_language": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()]))),
        "query_cap": lambda s: "; ".join(sorted(set([str(int(x)) for x in pd.to_numeric(s, errors="coerce").dropna().tolist()]))),
        "query_use_no_year": lambda s: "; ".join(sorted(set([("true" if str(x).strip().lower() in ("true","1","yes") else "false") for x in s if pd.notna(x)]))),
        "openalex_id": first_nonnull,
        "paperId": first_nonnull,
        "s2_url": first_nonnull,
    }

    stageA_dedup = (
        stageA
        .groupby(["source", "within_key"], as_index=False)
        .agg(agg_map)
        .drop(columns=["within_key"])
    )

    stageA_dedup["merge_key"] = stageA_dedup.apply(cross_source_merge_key, axis=1)

    def merge_sources(group: pd.DataFrame) -> dict:
        sources = sorted(set(group["source"].dropna().tolist()))
        source_ids = {src: group.loc[group["source"] == src, "source_id"].dropna().astype(str).unique().tolist() for src in sources}

        return {
            "chapter_id": chapter_id,
            "merge_key": group["merge_key"].iloc[0],
            "sources": "; ".join(sources),
            "source_count": len(sources),
            "source_ids": str(source_ids),
            "title": most_common_or_longest(group["title"]),
            "year": pd.to_numeric(group["year"], errors="coerce").dropna().median() if group["year"].notna().any() else np.nan,
            "venue": most_common_or_longest(group["venue"]),
            "type": most_common_or_longest(group["type"]),
            "authors(first6)": most_common_or_longest(group["authors(first6)"]),
            "doi": first_nonnull(group["doi"]),
            "doi_norm": first_nonnull(group["doi_norm"]),
            "citation_count_max": pd.to_numeric(group["citation_count"], errors="coerce").max(),
            "abstract": longest_text(group["abstract"]),
            "queries": "; ".join(split_semi_unique(group.get("query", pd.Series([], dtype=object)).dropna().tolist())),
            "query_ids": "; ".join(split_semi_unique(group.get("query_id", pd.Series([], dtype=object)).dropna().tolist())),
            "query_kinds": "; ".join(split_semi_unique(group.get("query_kind", pd.Series([], dtype=object)).dropna().tolist())),
            "query_services": "; ".join(split_semi_unique(group.get("query_service", pd.Series([], dtype=object)).dropna().tolist())),
            "query_languages": "; ".join(split_semi_unique(group.get("query_language", pd.Series([], dtype=object)).dropna().tolist())),
            "openalex_id": first_nonnull(group.get("openalex_id", pd.Series([], dtype=object))),
            "paperId": first_nonnull(group.get("paperId", pd.Series([], dtype=object))),
            "s2_url": first_nonnull(group.get("s2_url", pd.Series([], dtype=object))),
        }

    merged_records = [merge_sources(g) for _, g in stageA_dedup.groupby("merge_key")]
    df_stageA = pd.DataFrame(merged_records)

    df_stageA["merge_kind"] = df_stageA["merge_key"].str.split(":", n=1).str[0]
    df_stageA["has_abstract"] = df_stageA["abstract"].notna() & (df_stageA["abstract"].astype(str).str.len() > 50)
    df_stageA = df_stageA.sort_values(by="citation_count_max", ascending=False, na_position="last").reset_index(drop=True)

    return df_stageA

# ----------------------------

# -----------------------------


def build_stagea_query_diagnostics(plan: QueryPlanV2, df_oa: pd.DataFrame, df_s2: pd.DataFrame, df_stageA: pd.DataFrame) -> pd.DataFrame:
    specs = [q.model_dump() if hasattr(q, "model_dump") else dict(q) for q in (plan.queries or [])]
    rows = []
    for q in specs:
        rows.append({
            "query_id": str(q.get("id") or q.get("query_id") or ""),
            "service": str(q.get("service") or ""),
            "kind": str(q.get("kind") or ""),
            "language": str(q.get("language") or ""),
            "cap": int(q.get("cap", 0) or 0),
            "use_no_year": bool(q.get("use_no_year", False)),
            "query": str(q.get("query") or q.get("query_string") or ""),
        })
    d = pd.DataFrame(rows)
    if d.empty:
        return d

    # Raw fetch counts per query_id
    if isinstance(df_oa, pd.DataFrame) and (not df_oa.empty) and ("query_id" in df_oa.columns):
        oa_counts = df_oa["query_id"].fillna("").astype(str).value_counts().to_dict()
    else:
        oa_counts = {}

    if isinstance(df_s2, pd.DataFrame) and (not df_s2.empty) and ("query_id" in df_s2.columns):
        s2_counts = df_s2["query_id"].fillna("").astype(str).value_counts().to_dict()
    else:
        s2_counts = {}

    d["raw_hits_openalex"] = d["query_id"].map(lambda qid: int(oa_counts.get(qid, 0)))
    d["raw_hits_s2"] = d["query_id"].map(lambda qid: int(s2_counts.get(qid, 0)))
    d["raw_hits_total"] = d["raw_hits_openalex"] + d["raw_hits_s2"]

    # Whether a given OpenAlex query had the concept filter applied (debugging recall).
    if isinstance(df_oa, pd.DataFrame) and (not df_oa.empty) and ("query_id" in df_oa.columns) and ("oa_concept_filter_applied" in df_oa.columns):
        try:
            cf_map = df_oa.groupby("query_id")["oa_concept_filter_applied"].max().to_dict()
        except Exception:
            cf_map = {}
    else:
        cf_map = {}
    d["oa_concept_filtered"] = d["query_id"].map(lambda qid: bool(cf_map.get(str(qid), False)))

    # Post-merge contribution (StageA deduped)
    hits = {qid: 0 for qid in d["query_id"].tolist()}
    uniq = {qid: 0 for qid in d["query_id"].tolist()}
    if isinstance(df_stageA, pd.DataFrame) and (not df_stageA.empty) and ("query_ids" in df_stageA.columns):
        qlists = df_stageA["query_ids"].fillna("").astype(str).map(lambda s: split_semi_unique([s]))
        for xs in qlists.tolist():
            for qid in xs:
                if qid in hits:
                    hits[qid] += 1
        for xs in qlists.tolist():
            if len(xs) == 1 and xs[0] in uniq:
                uniq[xs[0]] += 1

    d["stageA_hits"] = d["query_id"].map(lambda qid: int(hits.get(qid, 0)))
    d["stageA_unique_hits"] = d["query_id"].map(lambda qid: int(uniq.get(qid, 0)))
    total_docs = int(len(df_stageA)) if isinstance(df_stageA, pd.DataFrame) else 0
    d["stageA_share"] = d["stageA_hits"].map(lambda x: (float(x) / float(total_docs)) if total_docs else 0.0)

    d = d.sort_values(["stageA_hits", "stageA_unique_hits", "raw_hits_total", "query_id"], ascending=[False, False, False, True], kind="mergesort")
    return d

# Run Stage A for all chapters
# -----------------------------

stageA_by_chapter: Dict[str, pd.DataFrame] = {}
stageA_stats: Dict[str, Dict[str, Any]] = {}

for ch in CHAPTERS:
    cid = ch["chapter_id"]
    bp = blueprints[cid]
    plan = query_plan_for_chapter(cid)
    plan_specs = [q.model_dump() if hasattr(q, "model_dump") else dict(q) for q in (plan.queries or [])]
    plan_q_count = len(plan_specs)
    plan_oa_count = sum(1 for q in plan_specs if str(q.get("service", "")).strip() == "openalex")
    plan_s2_count = sum(1 for q in plan_specs if str(q.get("service", "")).strip() == "semanticscholar")
    plan_gap_count = 0
    plan_no_year = sum(1 for q in plan_specs if bool(q.get("use_no_year", False)))

    concept_terms = openalex_concept_terms_for_chapter(bp)

    sig_obj = {
        "query_plan": plan.model_dump() if hasattr(plan, "model_dump") else plan_specs,
        "query_plan_counts": {"total": plan_q_count, "openalex": plan_oa_count, "s2": plan_s2_count, "no_year": plan_no_year},
        "OA_MAX_WORKS_PER_QUERY": OA_MAX_WORKS_PER_QUERY,
        "OA_REQUIRE_ABSTRACT": OA_REQUIRE_ABSTRACT,
        "OA_FILTER_LANGUAGES": OA_FILTER_LANGUAGES,
        "OA_FILTER_DATES": OA_FILTER_DATES,
        "OA_QUERY_MODE": str(OA_QUERY_MODE),
        "OA_SEARCH_KEY": _openalex_search_filter_key(),
        "OA_AUTHORITY_SORT": OA_AUTHORITY_SORT,
        "OA_CONCEPT_FILTER_ENABLED": OA_CONCEPT_FILTER_ENABLED,
        "OA_CONCEPT_FILTER_FOR_OPENALEX": OA_CONCEPT_FILTER_FOR_OPENALEX,
        "OA_CONCEPT_TERMS": concept_terms,
        "OA_CONCEPT_MAX_IDS": OA_CONCEPT_MAX_IDS,
        "OA_CONCEPT_TERMS_MAX": OA_CONCEPT_TERMS_MAX,
        "OA_CONCEPTS_PER_TERM": OA_CONCEPTS_PER_TERM,
        "OA_CONCEPT_LEVEL_MAX": OA_CONCEPT_LEVEL_MAX,
        "OA_CONCEPT_MIN_WORKS": OA_CONCEPT_MIN_WORKS,
        "OA_CONCEPT_FILTER_SHARE": OA_CONCEPT_FILTER_SHARE,
        "OA_CONCEPT_FILTER_ALWAYS_KINDS": sorted(list(OA_CONCEPT_FILTER_ALWAYS_KINDS)),
        "S2_REQUIRE_ABSTRACT_AFTER_BATCH": S2_REQUIRE_ABSTRACT_AFTER_BATCH,
        "S2_ABSTRACT_MIN_CHARS": S2_ABSTRACT_MIN_CHARS,
        "S2_FETCH_ABSTRACTS_VIA_BATCH": S2_FETCH_ABSTRACTS_VIA_BATCH,
        "S2_BATCH_SIZE": S2_BATCH_SIZE,
        "S2_LIMIT": S2_LIMIT,
        "S2_MAX_PAGES_PER_QUERY": S2_MAX_PAGES_PER_QUERY,
        "S2_USE_BULK_SEARCH": False,
    }
    sig = hashlib.sha1(json.dumps(sig_obj, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:12]

    ch_dir = STAGEA_DIR / cid / sig
    ch_dir.mkdir(parents=True, exist_ok=True)
    oa_csv = ch_dir / "openalex.csv"
    s2_csv = ch_dir / "semantic_scholar.csv"
    stagea_csv = ch_dir / "stageA_combined.csv"
    concepts_json = ch_dir / "openalex_concepts.json"

    print(f"\n=== Stage A for chapter: {cid} | plan_q={plan_q_count} oa={plan_oa_count} s2={plan_s2_count} no_year={plan_no_year} | sig={sig} ===")


    if bool(globals().get('STAGEA_PRINT_QUERY_PLAN', True)):
        print_section(f"Stage A query plan [{cid}]")
        print_kv({
            'queries_total': plan_q_count,
            'queries_openalex': plan_oa_count,
            'queries_semanticscholar': plan_s2_count,
            'queries_no_year': plan_no_year,
            'oa_require_abstract': bool(OA_REQUIRE_ABSTRACT),
            'oa_filter_languages': bool(OA_FILTER_LANGUAGES),
            'oa_filter_dates': bool(OA_FILTER_DATES),
            'concept_filter': bool(OA_CONCEPT_FILTER_ENABLED),
            'concept_filter_share': OA_CONCEPT_FILTER_SHARE,
            'concept_filter_always_kinds': '; '.join(sorted(list(OA_CONCEPT_FILTER_ALWAYS_KINDS))),
            's2_bulk_search': False,
        })

        qrows = []
        for q in plan_specs:
            qrows.append({
                "id": str(q.get("id") or q.get("query_id") or ""),
                "svc": str(q.get("service", "")),
                "kind": str(q.get("kind", "")),
                "lang": str(q.get("language", "")),
                "cap": int(q.get("cap", 0) or 0),
                "no_year": bool(q.get("use_no_year", False)),
                "facet_id": str(q.get("facet_id") or ""),
                "query": str(q.get("query") or q.get("query_string") or ""),
            })
        print("\nQuery specs:")
        print_table(qrows, columns=["id", "svc", "kind", "lang", "cap", "no_year", "facet_id", "query"], max_rows=int(STAGEA_PRINT_QUERY_TABLE_MAX))

        if concept_terms:
            print("\nConcept lookup terms (OpenAlex concepts endpoint):")
            for i, t in enumerate(concept_terms, start=1):
                print(f"  {i:02d}. {t}")

    concept_ids: List[str] = []
    concept_filter: Optional[str] = None

    if bool(OA_CONCEPT_FILTER_ENABLED) and bool(OA_CONCEPT_FILTER_FOR_OPENALEX):
        # Cache concept IDs per chapter signature
        if concepts_json.exists() and (not FETCH_FORCE):
            try:
                concept_ids = json.loads(concepts_json.read_text(encoding="utf-8"))
            except Exception:
                concept_ids = []
        if not concept_ids:
            concept_ids = openalex_concept_ids_from_terms(concept_terms)
            try:
                concepts_json.write_text(json.dumps(concept_ids, ensure_ascii=False, indent=2), encoding="utf-8")
            except Exception:
                pass
        concept_filter = openalex_concept_filter(concept_ids)
        if concept_ids:
            print(f"[OpenAlex:{cid}] concept filter ids ({len(concept_ids)}): {concept_ids}")

        if concept_filter:
            oa_specs = [s for s in plan_specs if str(s.get('service','')).strip() == 'openalex']
            applied = sum(1 for s in oa_specs if _should_apply_concept_filter(s))
            print(f"[OpenAlex:{cid}] concept filter applied to {applied}/{len(oa_specs)} OpenAlex queries (share={OA_CONCEPT_FILTER_SHARE})")

    # OpenAlex
    if RUN_OPENALEX:
        if oa_csv.exists() and not FETCH_FORCE:
            df_oa = pd.read_csv(oa_csv)
            print(f"[OpenAlex:{cid}] cached rows: {len(df_oa)}")
        else:
            df_oa = fetch_openalex_for_chapter(cid, plan, concept_filter=concept_filter)
            df_oa.to_csv(oa_csv, index=False, encoding="utf-8")
            print(f"[OpenAlex:{cid}] saved: {oa_csv} | rows={len(df_oa)}")
    else:
        df_oa = pd.DataFrame()

    # Semantic Scholar
    if RUN_SEMANTIC_SCHOLAR:
        df_s2 = fetch_s2_for_chapter(cid, plan, out_csv=s2_csv)
        print(f"[S2:{cid}] ready: {s2_csv} | rows={len(df_s2)}")
    else:
        df_s2 = pd.DataFrame()

    if stagea_csv.exists() and not FETCH_FORCE:
        df_stageA = pd.read_csv(stagea_csv)
        print(f"[StageA:{cid}] cached rows: {len(df_stageA)}")
    else:
        df_stageA = build_stagea(df_oa, df_s2, chapter_id=cid)
        df_stageA.to_csv(stagea_csv, index=False, encoding="utf-8")
        print(f"[StageA:{cid}] saved: {stagea_csv} | rows={len(df_stageA)}")

    stageA_stats[cid] = {
        'chapter_id': cid,
        'plan_queries': int(plan_q_count),
        'plan_oa_queries': int(plan_oa_count),
        'plan_s2_queries': int(plan_s2_count),
        'plan_gapfill_queries': int(plan_gap_count),
        'plan_no_year_queries': int(plan_no_year),
        'openalex_rows': int(len(df_oa)) if isinstance(df_oa, pd.DataFrame) else 0,
        's2_rows': int(len(df_s2)) if isinstance(df_s2, pd.DataFrame) else 0,
        'stageA_rows': int(len(df_stageA)) if isinstance(df_stageA, pd.DataFrame) else 0,
    }

    # Query diagnostics (which queries actually contribute after dedupe)
    try:
        diag = build_stagea_query_diagnostics(plan, df_oa, df_s2, df_stageA)
        diag_csv = ch_dir / 'query_diagnostics.csv'
        if not diag.empty:
            diag.to_csv(diag_csv, index=False, encoding='utf-8')
            print(f'[StageA:{cid}] saved query diagnostics: {diag_csv} | rows={len(diag)}')
            # quick preview
            display(diag.head(15))
    except Exception as e:
        print(f'[StageA:{cid}] query diagnostics skipped: {e}')

    stageA_by_chapter[cid] = df_stageA

print_section("Stage A summary")
rows = []
for cid, df0 in stageA_by_chapter.items():
    stats = stageA_stats.get(cid) or {}
    rows.append({
        "chapter_id": cid,
        "plan_q": str(stats.get("plan_queries","")),
        "oa_q": str(stats.get("plan_oa_queries","")),
        "s2_q": str(stats.get("plan_s2_queries","")),
        "gap_q": str(stats.get("plan_gapfill_queries","")),
        "no_year_q": str(stats.get("plan_no_year_queries","")),
        "oa_rows": _fmt_int(stats.get("openalex_rows", "")),
        "s2_rows": _fmt_int(stats.get("s2_rows", "")),
        "stageA_rows": _fmt_int(len(df0)),
        "has_abs": _fmt_pct(float(df0.get("has_abstract", pd.Series([False]*len(df0))).mean()) if len(df0) else 0.0),
    })
print_table(rows, columns=["chapter_id","plan_q","oa_q","s2_q","gap_q","no_year_q","oa_rows","s2_rows","stageA_rows","has_abs"], max_rows=200)

# Stage A plots (quick debugging)
plt = get_plt()
if plt and bool(PLOTS_ENABLED) and isinstance(stageA_by_chapter, dict) and stageA_by_chapter:
    print_section("Stage A plots")
    try:
        plot_rows = []
        for cid, df0 in stageA_by_chapter.items():
            if df0 is None:
                continue
            df0 = df0.copy()
            has_abs = float(df0.get("has_abstract", pd.Series([False] * len(df0))).mean()) if len(df0) else 0.0
            plot_rows.append({
                "chapter_id": str(cid),
                "stageA_rows": int(len(df0)),
                "has_abs": has_abs,
            })

        pdf = pd.DataFrame(plot_rows)
        if not pdf.empty:
            pdf = pdf.sort_values("stageA_rows", ascending=True)
            if len(pdf) > int(PLOT_MAX_CHAPTERS):
                pdf = pdf.tail(int(PLOT_MAX_CHAPTERS)).copy()

            labels = [short_label(x, 28) for x in pdf["chapter_id"].tolist()]
            h = max(3.2, 0.55 * len(pdf) + 1.6)
            fig, axes = plt.subplots(1, 2, figsize=(12, h))

            axes[0].barh(labels, pdf["stageA_rows"].tolist(), color="#4C78A8")
            axes[0].set_title("Stage A candidates per chapter")
            axes[0].set_xlabel("rows")

            axes[1].barh(labels, [100.0 * x for x in pdf["has_abs"].tolist()], color="#72B7B2")
            axes[1].set_title("Abstract coverage (Stage A)")
            axes[1].set_xlabel("% with abstract")
            axes[1].set_xlim(0, 100)

            plt.tight_layout()
            plt.show()
    except Exception as e:
        print("[plots] Stage A plots skipped:", e)



Semantic Scholar API key detected (SEMANTICSCHOLAR_API_KEY). Using authenticated requests (1 req/sec).

=== Stage A for chapter: fall_of_rome_economy | plan_q=97 oa=49 s2=48 no_year=21 | sig=3b056e3a8fc4 ===

Stage A query plan [fall_of_rome_economy]
queries_total               : 97
queries_openalex            : 49
queries_semanticscholar     : 48
queries_no_year             : 21
oa_require_abstract         : False
oa_filter_languages         : True
oa_filter_dates             : True
concept_filter              : True
concept_filter_share        : 0.3
concept_filter_always_kinds : authority
s2_bulk_search              : False

Query specs:
id   | svc             | kind      | lang | cap | no_year | facet_id                       | query                                                       
-----+-----------------+-----------+------+-----+---------+--------------------------------+-------------------------------------------------------------
Q001 | openalex        | anchor    | en   | 

,query_id,service,kind,language,cap,use_no_year,query,raw_hits_openalex,raw_hits_s2,raw_hits_total,oa_concept_filtered,stageA_hits,stageA_unique_hits,stageA_share
8,Q009,openalex,anchor,en,60,True,Italy Late Antiquity AND Western Roman Empire ...,60,0,60,False,60,36,0.212014
87,Q088,semanticscholar,proxy,en,40,False,Western Roman coin hoards coin hoard chronolog...,0,40,40,False,39,35,0.137809
30,Q031,openalex,facet,en,60,False,Gaul rural economy AND provincial inscriptions...,37,0,37,False,37,14,0.130742
41,Q042,openalex,proxy,en,60,True,archaeological settlement survey AND rural set...,36,0,36,False,36,34,0.127208
39,Q040,openalex,proxy,en,60,False,coin hoard chronology AND silver content AND m...,24,0,24,False,24,18,0.084806
10,Q011,openalex,anchor,en,60,False,Late Antique North Africa AND Western Roman Em...,22,0,22,False,22,13,0.077739
24,Q025,openalex,facet,en,60,False,amphora stamp typology AND maritime trade rout...,21,0,21,False,21,14,0.074205
4,Q005,openalex,anchor,en,60,True,Late Roman West AND Western Roman Empire AND N...,19,0,19,False,19,15,0.067138
40,Q041,openalex,proxy,en,60,False,amphora stamp assemblages AND port activity AN...,17,0,17,False,17,10,0.060071
31,Q032,openalex,facet,en,60,False,port activity AND overland transport costs AND...,13,0,13,False,13,5,0.045936



Stage A summary
chapter_id           | plan_q | oa_q | s2_q | gap_q | no_year_q | oa_rows | s2_rows | stageA_rows | has_abs
---------------------+--------+------+------+-------+-----------+---------+---------+-------------+--------
fall_of_rome_economy | 97     | 49   | 48   | 0     | 21        | 284     | 70      | 283         | 66.4%  

Stage A plots


## Stage C (finalized): scoring weights

Chosen via `sources_test.ipynb` Stage C grid search (`stageC_grid_v1_best`).

**Final weights**
- `w_embed_max = 0.0` (use `score_embed_mean_top3` only)
- `w_embed = 0.7`
- `cite_weight = 0.08`


In [4]:
import numpy as np
import pandas as pd

# Finalized Stage C hyperparams (grid-search winner)
STAGEC_FINAL_RUN = "20260130_184909_7370e7a6e85f"  # reference from eval_dataset/experiments
STAGEC_W_EMBED_MAX = 0.0
STAGEC_W_EMBED = 0.7
STAGEC_CITE_WEIGHT = 0.08


def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out


def add_stagec_final_scores(df_in: pd.DataFrame) -> pd.DataFrame:
    """Adds `score_stageC_final` based on finalized Stage C weights.

    Required columns:
    - chapter_id
    - score_embed_max
    - score_embed_mean_top3
    - score_tfidf
    - score_cite_norm
    """
    required = [
        "chapter_id",
        "score_embed_max",
        "score_embed_mean_top3",
        "score_tfidf",
        "score_cite_norm",
    ]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing required columns for Stage C scoring: {missing}")

    df = df_in.copy()

    emb_max = pd.to_numeric(df["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(df["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    df["_emb_raw"] = float(STAGEC_W_EMBED_MAX) * emb_max + (1.0 - float(STAGEC_W_EMBED_MAX)) * emb_b

    df["_emb_n"] = minmax_by_group(df, "_emb_raw")
    df["_tf_n"] = minmax_by_group(df, "score_tfidf")
    df["_cite_n"] = minmax_by_group(df, "score_cite_norm")

    base = float(STAGEC_W_EMBED) * df["_emb_n"] + (1.0 - float(STAGEC_W_EMBED)) * df["_tf_n"]
    df["score_stageC_final"] = (1.0 - float(STAGEC_CITE_WEIGHT)) * base + float(STAGEC_CITE_WEIGHT) * df["_cite_n"]
    return df


print("Stage C finalized weights:")
print("- STAGEC_W_EMBED_MAX:", STAGEC_W_EMBED_MAX)
print("- STAGEC_W_EMBED:", STAGEC_W_EMBED)
print("- STAGEC_CITE_WEIGHT:", STAGEC_CITE_WEIGHT)


Stage C finalized weights:
- STAGEC_W_EMBED_MAX: 0.0
- STAGEC_W_EMBED: 0.7
- STAGEC_CITE_WEIGHT: 0.08


## Stage C (production): facet-union pool + embeddings + finalized scoring

Build a per-chapter candidate pool, score it, and write it to `final_pipeline/stageC/`.


In [5]:
# -----------------------------
# Stage C: pool + scoring (TF-IDF + embeddings) (cached)
# -----------------------------

import json
import hashlib
import re
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Retrieval/scoring hyperparams (same defaults used in eval_dataset_builder.ipynb)
TFIDF_MAX_FEATURES = 200_000
TFIDF_MIN_DF = 2
TFIDF_NGRAM_RANGE = (1, 2)
TOP_PER_QUERY = 400

EMBED_MODEL = "text-embedding-3-small"
MAX_CHARS_PER_EMBED = 3500
EMBED_BATCH_SIZE = 64

# Only used for score_embed_combo (not required for the finalized Stage C score)
W_EMBED_MAX = 0.70
W_EMBED_BREADTH = 0.30

EMBED_CACHE_DIR = SOURCES_WORKSPACE_DIR / ".embed_cache"
EMBED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

STAGEC_FORCE = False

# -----------------------------
# Authority + quality heuristics (Stage C)
# -----------------------------
# Implements (from RESOURCE_RANKING_IMPROVEMENTS.md):
# - A) robust citation normalization
# - B) citation velocity (citations/age)
# - C) multi-source reliability boost
# - D) missing-abstract penalty
# - E) (part) propagate authority later into Stage D relevance (implemented in Stage D cell)
# - F) survey/review/standard/framework-style boosts
# - Additional: blueprint negative_query_terms soft downrank
# - Additional: peer-reviewed type soft preference
# - Additional: suspicious venue downrank via configurable blocklist

CURRENT_YEAR = int(datetime.utcnow().year)

# A/B: citations robust + velocity
CITE_CLIP_Q = 0.95
CITE_VELOCITY_WEIGHT = 0.35
MISSING_YEAR_AGE_DEFAULT = 10  # used when year is missing

# C: multi-source reliability
MULTI_SOURCE_BOOST = 0.02

# D: missing abstract penalty (applied to `score_stageC_final`)
NO_ABSTRACT_PENALTY = 0.10

# Additional: blueprint negatives soft downrank
NEG_TERM_PENALTY_PER_MATCH = 0.06
NEG_TERM_PENALTY_CAP = 0.18

# F: survey/review/standard/framework-style boosts
SURVEY_LIKE_BOOST = 0.02
SURVEY_LIKE_REGEX = r"\b(survey|systematic review|review|taxonomy|reference architecture|standard|framework|state of the art)\b"

# Additional: prefer peer-reviewed types (soft)
PEER_REVIEWED_BOOST = 0.015
PREPRINT_PENALTY = 0.01
PEER_REVIEWED_TYPES = {
    "article",
    "review",
    "book",
    "book-chapter",
}
PREPRINT_TYPES = {
    "preprint",
}

# Additional: suspicious venue downrank via configurable regex blocklist
VENUE_BLOCKLIST_PATH = REPO_ROOT / "config" / "venue_blocklist_regex.txt"
VENUE_BLOCKLIST_PENALTY = 0.15


def robust_minmax(x: pd.Series, *, clip_q: float = 0.95) -> pd.Series:
    """Robust 0..1 scaling by clipping the upper tail.

    - Keeps zeros at 0.
    - Prevents single mega-outliers from compressing everything else.
    """

    v = pd.to_numeric(x, errors="coerce").fillna(0.0).astype(float)
    v = v.clip(lower=0.0)
    if len(v) == 0:
        return v

    if float(v.max()) <= 0.0:
        return v * 0.0

    cap = float(v.quantile(float(clip_q))) if 0.0 < float(clip_q) < 1.0 else float(v.max())
    cap = max(cap, 0.0)
    if cap <= 0.0:
        cap = float(v.max())

    v2 = v.clip(upper=cap)
    mx = float(v2.max())
    return (v2 / mx) if mx > 0.0 else (v2 * 0.0)


def load_regex_blocklist(path: Path) -> list[str]:
    if not path.exists():
        return []
    pats: list[str] = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = str(raw).strip()
        if not line or line.startswith("#"):
            continue
        # Validate regex early so we don't explode later.
        try:
            re.compile(line)
        except re.error as e:
            print(f"[StageC] WARNING: invalid venue regex ignored: {line!r} ({e})")
            continue
        pats.append(line)
    return pats


def venue_blocklist_mask(venues: pd.Series, patterns: list[str]) -> pd.Series:
    if not patterns:
        return pd.Series(False, index=venues.index)
    v = venues.fillna("").astype(str)
    m = pd.Series(False, index=venues.index)
    for pat in patterns:
        m = m | v.str.contains(pat, case=False, regex=True, na=False)
    return m


VENUE_BLOCKLIST_PATTERNS = load_regex_blocklist(VENUE_BLOCKLIST_PATH)


def apply_stagec_quality_adjustments(df_in: pd.DataFrame, *, blueprint: dict) -> pd.DataFrame:
    """Post-process Stage C score with low-cost quality/authority heuristics."""

    df = df_in.copy()
    if "score_stageC_final" not in df.columns or len(df) == 0:
        return df

    score = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0).astype(float)

    # C) multi-source reliability boost
    sc = pd.to_numeric(df.get("source_count", 1), errors="coerce").fillna(1).astype(int)
    score = score + float(MULTI_SOURCE_BOOST) * (sc >= 2).astype(float)

    # D) missing abstract penalty
    if "has_abstract" in df.columns:
        has_abs = df["has_abstract"].fillna(False).astype(bool)
    else:
        has_abs = df.get("abstract", "").fillna("").astype(str).str.len() > 50
        df["has_abstract"] = has_abs
    score = score * np.where(has_abs.to_numpy(dtype=bool), 1.0, 1.0 - float(NO_ABSTRACT_PENALTY))

    # F) survey/review/standard/framework-style boost
    title_l = df.get("title", "").fillna("").astype(str).str.lower()
    is_survey_like = title_l.str.contains(SURVEY_LIKE_REGEX, regex=True, na=False)
    df["is_survey_like"] = is_survey_like
    score = score + float(SURVEY_LIKE_BOOST) * is_survey_like.astype(float)

    # Additional: peer-reviewed type soft preference
    type_l = df.get("type", "").fillna("").astype(str).str.lower()
    is_peer = type_l.isin(PEER_REVIEWED_TYPES)
    is_preprint = type_l.isin(PREPRINT_TYPES)
    df["is_peer_reviewed_type"] = is_peer
    df["is_preprint_type"] = is_preprint
    score = score + float(PEER_REVIEWED_BOOST) * is_peer.astype(float) - float(PREPRINT_PENALTY) * is_preprint.astype(float)

    # Additional: suspicious venue downrank via blocklist
    venue = df.get("venue", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
    venue_bad = venue_blocklist_mask(venue, VENUE_BLOCKLIST_PATTERNS)
    df["venue_blocklisted"] = venue_bad
    score = score * np.where(venue_bad.to_numpy(dtype=bool), 1.0 - float(VENUE_BLOCKLIST_PENALTY), 1.0)

    # Additional: blueprint negatives soft downrank
    neg_terms = [str(x).strip().lower() for x in (blueprint.get("negative_query_terms") or []) if str(x).strip()]
    if neg_terms:
        text_l = df.get("doc_text", "").fillna("").astype(str).str.lower()
        hits = np.zeros(len(df), dtype=int)
        for term in neg_terms:
            hits += text_l.str.contains(term, regex=False, na=False).astype(int).to_numpy(dtype=int)
        df["neg_term_hits"] = hits
        penalty = np.minimum(float(NEG_TERM_PENALTY_CAP), float(NEG_TERM_PENALTY_PER_MATCH) * hits.astype(float))
        score = score * (1.0 - penalty)

    df["score_stageC_final"] = score
    return df


# -----------------------------
# Candidate pool utilities
# -----------------------------

def minmax(x):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    return (x - mn) / (mx - mn + 1e-12)

def truncate_text(s: str, max_chars: int) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\n", " ").strip()
    return s[:max_chars]

def build_chapter_query_text(bp: dict) -> str:
    parts = []
    main = bp["main_query"]
    parts.extend([main, main])
    parts.extend(bp.get("facet_queries", []))
    parts.extend(bp.get("keywords", []))
    parts.extend(bp.get("key_concepts", []))
    return " ".join(parts)

def tfidf_scores(query_text: str, vectorizer: TfidfVectorizer, X) -> np.ndarray:
    qv = vectorizer.transform([query_text])
    return cosine_similarity(qv, X).ravel()

def facet_union_pool(bp: dict, df_in: pd.DataFrame, vectorizer: TfidfVectorizer, X, top_per_query: int) -> pd.DataFrame:
    query_texts = [bp["main_query"], bp["main_query"]] + list(bp.get("facet_queries", []))
    pool_keys = set()
    for qt in query_texts:
        qv = vectorizer.transform([qt])
        sims = cosine_similarity(qv, X).ravel()
        top_idx = np.argsort(-sims)[:top_per_query]
        pool_keys.update(df_in.iloc[top_idx]["merge_key"].astype(str).tolist())
    out = df_in[df_in["merge_key"].astype(str).isin(pool_keys)].copy()
    out = out.drop_duplicates(subset=["merge_key"]).reset_index(drop=True)
    return out

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

def hash_obj(obj: Any) -> str:
    blob = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    return hashlib.sha1(blob.encode("utf-8")).hexdigest()

def save_npz(path: Path, keys: List[str], mat: np.ndarray):
    np.savez_compressed(path, keys=np.array(keys, dtype=object), embeds=mat.astype(np.float32))

def load_npz(path: Path):
    z = np.load(path, allow_pickle=True)
    return list(z["keys"]), z["embeds"].astype(np.float32)

embed_totals = {"requests": 0, "input_tokens": 0, "cost_usd": 0.0}

def embed_texts(texts: List[str], model: str, batch_size: int, max_retries: int = 6) -> np.ndarray:
    """Embeddings with retries + token/cost tracking."""
    price = MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0}).get("input", 0.0)
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        backoff = 1.0
        for attempt in range(1, max_retries + 1):
            try:
                resp = client.embeddings.create(model=model, input=batch)
                vecs = [np.array(item.embedding, dtype=np.float32) for item in resp.data]
                all_vecs.extend(vecs)

                usage = getattr(resp, "usage", None)
                tok = int(getattr(usage, "total_tokens", 0) or getattr(usage, "prompt_tokens", 0) or 0)
                embed_totals["requests"] += 1
                embed_totals["input_tokens"] += tok
                embed_totals["cost_usd"] += (tok / 1_000_000) * price
                break
            except Exception:
                if attempt == max_retries:
                    raise
                time.sleep(backoff)
                backoff *= 2
    return np.vstack(all_vecs)

def score_pool_with_embeddings(bp: dict, pool: pd.DataFrame) -> pd.DataFrame:
    pool = pool.copy()
    pool["doc_text_trunc"] = (
        pool["title"].fillna("") + "\n\n" + pool["abstract"].fillna("")
    ).apply(lambda s: truncate_text(s, MAX_CHARS_PER_EMBED))

    keys = pool["merge_key"].astype(str).tolist()

    # Doc embeddings cache
    doc_hash = hash_obj({"model": EMBED_MODEL, "max_chars": MAX_CHARS_PER_EMBED, "keys": keys})[:16]
    doc_npz = EMBED_CACHE_DIR / f"eval_doc_embeds_{EMBED_MODEL}_{doc_hash}.npz"

    # Query embeddings cache
    query_texts = [bp["main_query"], bp["main_query"]] + list(bp.get("facet_queries", []))
    q_hash = hash_obj({"model": EMBED_MODEL, "queries": query_texts})[:16]
    q_json = EMBED_CACHE_DIR / f"eval_query_embeds_{EMBED_MODEL}_{q_hash}.json"

    # Docs
    if doc_npz.exists():
        cached_keys, doc_embeds = load_npz(doc_npz)
        if cached_keys != keys:
            doc_embeds = embed_texts(pool["doc_text_trunc"].tolist(), model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
            save_npz(doc_npz, keys, doc_embeds)
    else:
        doc_embeds = embed_texts(pool["doc_text_trunc"].tolist(), model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
        save_npz(doc_npz, keys, doc_embeds)

    # Queries
    if q_json.exists():
        q_cached = json.loads(q_json.read_text(encoding="utf-8"))
        query_embeds = np.array(q_cached["embeddings"], dtype=np.float32)
    else:
        query_embeds = embed_texts(query_texts, model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
        q_json.write_text(
            json.dumps({"model": EMBED_MODEL, "query_texts": query_texts, "embeddings": query_embeds.tolist()}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    docN = l2_normalize(doc_embeds)
    qN = l2_normalize(query_embeds)
    S = docN @ qN.T

    pool["score_embed_max"] = S.max(axis=1)
    pool["score_embed_mean_top3"] = np.sort(S, axis=1)[:, -3:].mean(axis=1)
    pool["score_embed_norm"] = minmax(pool["score_embed_max"].values)
    pool["score_embed_mean_top3_norm"] = minmax(pool["score_embed_mean_top3"].values)
    pool["score_embed_combo"] = W_EMBED_MAX * pool["score_embed_norm"] + W_EMBED_BREADTH * pool["score_embed_mean_top3_norm"]

    # Facet assignment (exclude the two main-query columns)
    facets = list(bp.get("facet_queries", []))
    if facets:
        facet_S = S[:, 2:2+len(facets)]
        facet_best_i = facet_S.argmax(axis=1).astype(int)
        pool["facet_best_i"] = facet_best_i
        pool["facet_best_query"] = [facets[i] for i in facet_best_i]
    else:
        pool["facet_best_i"] = -1
        pool["facet_best_query"] = ""

    return pool

# -----------------------------
# Build Stage C pool per chapter
# -----------------------------

stageC_by_chapter: Dict[str, pd.DataFrame] = {}
stageC_frames = []

def _load_latest_stagea_csv(chapter_id: str) -> pd.DataFrame:
    ch_dir = STAGEA_DIR / chapter_id
    if not ch_dir.exists():
        raise FileNotFoundError(f"No StageA dir for {chapter_id}: {ch_dir}")
    cands = sorted(ch_dir.glob('*/stageA_combined.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not cands:
        raise FileNotFoundError(f"No stageA_combined.csv found under {ch_dir}")
    return pd.read_csv(cands[0])

for ch in CHAPTERS:
    cid = ch['chapter_id']
    bp_obj = blueprints[cid]
    bp = bp_obj.model_dump()

    # Load StageA corpus
    if 'stageA_by_chapter' in globals() and isinstance(stageA_by_chapter, dict) and cid in stageA_by_chapter:
        df = stageA_by_chapter[cid].copy()
    else:
        df = _load_latest_stagea_csv(cid)

    if df.empty:
        print(f"[{cid}] StageA empty; skipping")
        continue

    # Ensure required columns
    for c_req in ['merge_key', 'title', 'abstract']:
        if c_req not in df.columns:
            raise RuntimeError(f"[{cid}] StageA missing required column: {c_req}")

    df = df.copy()
    df['title'] = df['title'].fillna('')
    df['abstract'] = df['abstract'].fillna('')
    df['doc_text'] = (df['title'].astype(str) + '\n\n' + df['abstract'].astype(str)).str.strip()

    # Citation normalization (robust + velocity) (A/B)
    cites = pd.to_numeric(df.get('citation_count_max', 0), errors='coerce').fillna(0).clip(lower=0)
    df['score_cite_log'] = np.log1p(cites)

    df['score_cite_robust'] = robust_minmax(df['score_cite_log'], clip_q=CITE_CLIP_Q)

    years = pd.to_numeric(df.get('year', np.nan), errors='coerce')
    age = (float(CURRENT_YEAR) - years + 1.0).clip(lower=1.0)
    age = age.fillna(float(MISSING_YEAR_AGE_DEFAULT))
    cites_per_year = cites / age
    df['score_cite_velocity'] = robust_minmax(np.log1p(cites_per_year), clip_q=CITE_CLIP_Q)

    df['score_cite_norm'] = (1.0 - float(CITE_VELOCITY_WEIGHT)) * df['score_cite_robust'] + float(CITE_VELOCITY_WEIGHT) * df['score_cite_velocity']

    # Fit TF-IDF per chapter
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
    )
    X = vectorizer.fit_transform(df['doc_text'])

    chapter_query_text = build_chapter_query_text(bp)
    df['score_tfidf'] = tfidf_scores(chapter_query_text, vectorizer=vectorizer, X=X)

    pool = facet_union_pool(bp, df_in=df, vectorizer=vectorizer, X=X, top_per_query=TOP_PER_QUERY)
    print(f"[{cid}] pool size: {len(pool)} (from StageA {len(df)})")

    pool_scored = score_pool_with_embeddings(bp, pool)
    pool_scored = add_stagec_final_scores(pool_scored)
    pool_scored = apply_stagec_quality_adjustments(pool_scored, blueprint=bp)

    # Pretty preview for debugging
    try:
        top_prev = pool_scored.sort_values('score_stageC_final', ascending=False, kind='mergesort').head(5).copy()
        cols_prev = [
            'score_stageC_final','score_embed_max','score_tfidf','score_cite_norm',
            'title','year','type','facet_best_query','venue','sources','source_count',
            'has_abstract','venue_blocklisted','neg_term_hits','is_survey_like'
        ]
        cols_prev = [c for c in cols_prev if c in top_prev.columns]
        print(f'\n[{cid}] Top candidates (Stage C):')
        display(top_prev[cols_prev])
    except Exception:
        pass

    # Persist
    sig_obj = {
        'chapter_id': cid,
        'main_query': bp.get('main_query',''),
        'facet_queries': bp.get('facet_queries', []),
        'TOP_PER_QUERY': TOP_PER_QUERY,
        'EMBED_MODEL': EMBED_MODEL,
        'MAX_CHARS_PER_EMBED': MAX_CHARS_PER_EMBED,
        'TFIDF_MAX_FEATURES': TFIDF_MAX_FEATURES,
        'TFIDF_MIN_DF': TFIDF_MIN_DF,
        'TFIDF_NGRAM_RANGE': TFIDF_NGRAM_RANGE,
    }
    sig = hashlib.sha1(json.dumps(sig_obj, sort_keys=True).encode('utf-8')).hexdigest()[:12]
    out_dir = STAGEC_DIR / cid / sig
    out_dir.mkdir(parents=True, exist_ok=True)
    out_csv = out_dir / 'stageC_pool_scored.csv'
    if (not out_csv.exists()) or STAGEC_FORCE:
        pool_scored.to_csv(out_csv, index=False, encoding='utf-8')
    stageC_by_chapter[cid] = pool_scored
    stageC_frames.append(pool_scored)

stageC_all = pd.concat(stageC_frames, axis=0).reset_index(drop=True) if stageC_frames else pd.DataFrame()
stageC_all_path = STAGEC_DIR / f"{RUN_ID}_stageC_all_pool_scored.csv"
if not stageC_all.empty:
    stageC_all.to_csv(stageC_all_path, index=False, encoding='utf-8')
    print(f"Saved Stage C combined: {stageC_all_path} | rows={len(stageC_all)}")


print_section("Stage C summary")
print("Embedding usage:")
print_kv({
    "embed_model": EMBED_MODEL,
    "requests": _fmt_int(embed_totals.get("requests")),
    "input_tokens": _fmt_int(embed_totals.get("input_tokens")),
    "cost_usd": _fmt_usd(embed_totals.get("cost_usd")),
})

rows = []
for cid, pool in stageC_by_chapter.items():
    if pool is None or len(pool) == 0:
        continue
    rows.append({
        "chapter_id": cid,
        "pool": _fmt_int(len(pool)),
        "blocklisted": _fmt_int(int(pool.get("venue_blocklisted", pd.Series([False]*len(pool))).sum())),
        "no_abs": _fmt_int(int((~pool.get("has_abstract", pd.Series([True]*len(pool)))).sum())),
        "survey_like": _fmt_int(int(pool.get("is_survey_like", pd.Series([False]*len(pool))).sum())),
        "top_score": f"{float(pd.to_numeric(pool['score_stageC_final'], errors='coerce').fillna(0).max()):.4f}",
    })
print("\nPer-chapter diagnostics:")
print_table(rows, columns=["chapter_id","pool","blocklisted","no_abs","survey_like","top_score"], max_rows=200)

# Stage C plots (quick debugging)
plt = get_plt()
if plt and bool(PLOTS_ENABLED) and isinstance(stageC_by_chapter, dict) and stageC_by_chapter:
    print_section("Stage C plots")
    try:
        cids = list(stageC_by_chapter.keys())
        if len(cids) > int(PLOT_MAX_CHAPTERS):
            cids = cids[: int(PLOT_MAX_CHAPTERS)]

        n = len(cids)
        fig, axes = plt.subplots(n, 2, figsize=(12, max(3.2, 3.2 * n)), squeeze=False)

        for r, cid in enumerate(cids):
            dfp = stageC_by_chapter[cid]
            if dfp is None or len(dfp) == 0:
                continue

            s = pd.to_numeric(dfp.get("score_stageC_final", 0.0), errors="coerce").fillna(0.0)
            axes[r, 0].hist(s, bins=40, color="#4C78A8", alpha=0.85)
            axes[r, 0].set_title(f"{short_label(cid, 34)} — score_stageC_final")
            axes[r, 0].set_xlabel("score")
            axes[r, 0].set_ylabel("count")

            # Authority vs Stage C score scatter (sampled)
            if len(dfp) > int(PLOT_MAX_POINTS):
                samp = dfp.sample(n=int(PLOT_MAX_POINTS), random_state=42)
            else:
                samp = dfp

            x = pd.to_numeric(samp.get("score_cite_norm", 0.0), errors="coerce").fillna(0.0)
            y = pd.to_numeric(samp.get("score_stageC_final", 0.0), errors="coerce").fillna(0.0)
            bad = samp.get("venue_blocklisted", False)
            bad = bad.fillna(False).astype(bool) if hasattr(bad, "fillna") else pd.Series([False] * len(samp))

            axes[r, 1].scatter(x[~bad], y[~bad], s=8, alpha=0.35, label="ok", color="#4C78A8")
            if bool(bad.any()):
                axes[r, 1].scatter(x[bad], y[bad], s=10, alpha=0.6, label="blocklisted", color="#E45756")

            axes[r, 1].set_title(f"{short_label(cid, 34)} — cite vs Stage C")
            axes[r, 1].set_xlabel("score_cite_norm")
            axes[r, 1].set_ylabel("score_stageC_final")
            axes[r, 1].legend(loc="best")

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print("[plots] Stage C plots skipped:", e)



[fall_of_rome_economy] pool size: 283 (from StageA 283)

[fall_of_rome_economy] Top candidates (Stage C):


<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>  is_survey_like = title_l.str.contains(SURVEY_LIKE_REGEX, regex=True, na=False)


,score_stageC_final,score_embed_max,score_tfidf,score_cite_norm,title,year,type,facet_best_query,venue,sources,source_count,has_abstract,venue_blocklisted,neg_term_hits,is_survey_like
30,0.714777,0.593811,0.243008,0.552565,Coin Circulation and Mint Activity in the Late...,1978.0,None,"""Western Roman Empire"" AND ""coin debasement"" A...",None,semantic_scholar,1,False,False,0,False
118,0.706957,0.632992,0.126989,0.256885,Imperial Monetary Policy and Social Reaction i...,2018.0,None,"""Western Roman Empire"" AND ""coin debasement"" A...",Journal des Économistes et des Études Humaines,semantic_scholar,1,True,False,1,False
79,0.696898,0.617376,0.084850,0.305614,At the Edge of Empire: Iron Age and Early Roma...,2012.0,dissertation,"""Western Roman Empire"" AND ""Italy provincial s...",Leicester Research Archive (University of Leic...,openalex,1,True,False,0,False
81,0.693689,0.589706,0.095085,0.404273,Mints not Mines: a macroscale investigation of...,2023.0,None,"""Western Roman Empire"" AND ""coin debasement"" A...",Internet Archaeology,semantic_scholar,1,True,False,0,False
48,0.691621,0.579954,0.076745,0.651432,Archaeometric Characterisation and Assessment ...,2023.0,None,"""Western Roman Empire"" AND ""Italy provincial s...",The Heritage,semantic_scholar,1,True,False,0,False


Saved Stage C combined: <projektverzeichnis>\sources_workspace\final_pipeline\stageC\20260210_131329_stageC_all_pool_scored.csv | rows=283

Stage C summary
Embedding usage:
embed_model  : text-embedding-3-small
requests     : 6
input_tokens : 58,900
cost_usd     : $0.0012

Per-chapter diagnostics:
chapter_id           | pool | blocklisted | no_abs | survey_like | top_score
---------------------+------+-------------+--------+-------------+----------
fall_of_rome_economy | 283  | 0           | 95     | 8           | 0.7148   

Stage C plots


In [6]:
# -----------------------------
# Stage C.3 (finalized): LLM rerank within Stage C top-N
# -----------------------------
#
# Final decision from LOCO testing (2026-01-31):
# - Use the LLM only as a shortlist reranker to avoid full-rewrite errors.
# - Select top-N by `score_stageC_final`, then order that shortlist by LLM score.

import asyncio
import time
import random
from typing import Dict, Tuple, Literal

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm  # type: ignore
except Exception:  # pragma: no cover
    tqdm = None

STAGEC3_TOPN = 50
STAGEC3_TOPN_MAX = 250  # adaptive expansion ceiling (per chapter)
STAGEC3_TOPN_STEP = 25   # expand in chunks until enough non-excludes
STAGEC3_MIN_NON_EXCLUDE = 20  # ensure Stage D can select 20 without tail fallback

STAGEC3_MODEL = "gpt-5-nano"
STAGEC3_PROMPT_VERSION = "v2"

STAGEC3_ALPHA_LLM = 0.2  # LLM weight in Stage C.3 ordering/signal (alpha=0.2 was best in offline sweeps)

STAGEC3_CONFIDENCE_MIN = 50  # ignore low-confidence LLM judgments

STAGEC3_CONCURRENCY = 8
STAGEC3_MAX_RETRIES = 8
STAGEC3_BACKOFF_INITIAL = 1.0
STAGEC3_BACKOFF_MAX = 30.0
STAGEC3_ABSTRACT_MAX_CHARS = 2000

STAGEC3_FORCE_RERANK = False

STAGEC3_DIR = OUT_DIR / "stageC3_rerank_cache_v1"
STAGEC3_DIR.mkdir(parents=True, exist_ok=True)


class StageC3RerankOut(BaseModel):
    label: Literal["include", "maybe", "exclude"] = Field(...)
    confidence: int = Field(..., ge=0, le=100)
    score: int = Field(..., ge=0, le=100)
    notes: str = Field("", max_length=140)

stagec3_agent = Agent(
    name=f"StageC3 Shortlist Rerank ({STAGEC3_PROMPT_VERSION})",
    model=STAGEC3_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=(
        "You validate whether a paper fits the chapter rubric.\n"
        "Return ONLY the structured output.\n\n"
        "Scoring (score is 0-100, not 0-3):\n"
        "- 90-100: excellent fit (covers multiple MUST_COVER, avoids MUST_AVOID)\n"
        "- 70-89: strong fit\n"
        "- 40-69: partial/tangential fit\n"
        "- 0-39: out of scope / wrong domain / violates MUST_AVOID\n\n"
        "label must match score: include>=70, maybe 40-69, exclude<40.\n"
        "If keywords are used in a different domain/context than the chapter, label exclude and score<=10.\n"
        "If ABSTRACT is missing, be conservative: lower confidence and avoid high scores."
    ),
    output_type=StageC3RerankOut,
)


def _stagec3_rubric_signature(bp: ChapterBlueprint) -> str:
    payload = {
        "scope_statement": bp.scope_statement,
        "must_cover": bp.must_cover,
        "must_avoid": bp.must_avoid,
        "scoring_guidance": bp.scoring_guidance,
        "prompt_version": STAGEC3_PROMPT_VERSION,
        "model": STAGEC3_MODEL,
        "abstract_max_chars": STAGEC3_ABSTRACT_MAX_CHARS,
    }
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:8]


def _stagec3_prompt(bp: ChapterBlueprint, title: str, abstract: str) -> str:
    scope = bp.scope_statement
    must_cover = bp.must_cover or []
    must_avoid = bp.must_avoid or []
    guidance = bp.scoring_guidance or ""

    abs_txt = (abstract or "").strip()
    if STAGEC3_ABSTRACT_MAX_CHARS and len(abs_txt) > int(STAGEC3_ABSTRACT_MAX_CHARS):
        abs_txt = abs_txt[: int(STAGEC3_ABSTRACT_MAX_CHARS)]

    lines = []
    lines.append("Score this paper for inclusion in the chapter.")
    lines.append("Return only the schema fields.")
    lines.append("")
    lines.append("SCORING")
    lines.append("- score: integer 0-100 (not 0-3); use the full range")
    lines.append("- label must match score: include>=70, maybe 40-69, exclude<40")
    lines.append("- confidence: 0-100 certainty; use low confidence if ABSTRACT is missing")
    lines.append("- if domain/context mismatches the chapter, label exclude and score<=10")
    lines.append("")
    lines.append("RUBRIC")
    lines.append(f"SCOPE: {scope}")
    if guidance:
        lines.append(f"GUIDANCE: {guidance}")
    if must_cover:
        lines.append("MUST_COVER:")
        for b in must_cover:
            lines.append(f"- {b}")
    if must_avoid:
        lines.append("MUST_AVOID:")
        for b in must_avoid:
            lines.append(f"- {b}")
    lines.append("")
    lines.append("PAPER")
    lines.append(f"TITLE: {(title or '').strip()}")
    if abs_txt:
        lines.append(f"ABSTRACT: {abs_txt}")
    else:
        lines.append("ABSTRACT: (missing)")
    lines.append("")
    return "\n".join(lines)


def _minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)



async def stagec3_rerank_topn(
    df_in: pd.DataFrame,
    blueprints_by_chapter_id: Dict[str, ChapterBlueprint],
    *,
    topn: int = STAGEC3_TOPN,
    topn_max: int = STAGEC3_TOPN_MAX,
    topn_step: int = STAGEC3_TOPN_STEP,
    min_non_exclude: int = STAGEC3_MIN_NON_EXCLUDE,
    id_col: str = "merge_key",
) -> Tuple[pd.DataFrame, dict]:
    """Adds Stage C.3 shortlist rerank scores.

    Scientific intent:
    - We only use the LLM inside a shortlist (top-N by Stage C score).
    - In production, some chapters can have too many `exclude` labels inside top-N.
      If we then run Stage D (diversity selection), it may be forced to pick from the
      unscored tail and output irrelevant, unlabeled results.

    Therefore this function *adaptively expands* the per-chapter shortlist until we have
    at least `min_non_exclude` items labeled {include, maybe} (ignoring excludes), up to
    `topn_max`.

    Required columns in df_in:
    - chapter_id
    - title
    - abstract
    - score_stageC_final

    Returns (df_out, totals).
    """
    required = ["chapter_id", "title", "abstract", "score_stageC_final"]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing required columns for Stage C.3: {missing}")

    df = df_in.copy()

    print_section("Stage C.3 rerank")
    print_kv({
        "model": STAGEC3_MODEL,
        "topn": int(topn),
        "topn_max": int(topn_max),
        "min_non_exclude": int(min_non_exclude),
        "concurrency": int(STAGEC3_CONCURRENCY),
        "force_rerank": bool(STAGEC3_FORCE_RERANK),
    })

    # Ensure stable id
    if id_col not in df.columns:
        years = df["year"] if "year" in df.columns else pd.Series([""] * len(df), index=df.index)
        df[id_col] = [
            hashlib.sha1(f"{t}|{y}".encode("utf-8")).hexdigest()[:16]
            for t, y in zip(df["title"].fillna("").astype(str), years.fillna("").astype(str))
        ]

    # Output columns
    df["score_llm_rerank_v1"] = np.nan
    df["llm_notes"] = ""
    df["llm_label"] = ""
    df["llm_confidence"] = np.nan

    sem = asyncio.Semaphore(int(STAGEC3_CONCURRENCY))
    t0 = time.time()

    totals = {
        "seconds": 0.0,
        "requests": 0,
        "input_tokens": 0,
        "cached_input_tokens": 0,
        "output_tokens": 0,
        "cost_usd": 0.0,
        "cached_files": 0,
        "topn_used_by_chapter": {},
    }

    async def _run_one(chapter_id: str, doc_id: str, title: str, abstract: str) -> dict:
        bp = blueprints_by_chapter_id[chapter_id]
        sig = _stagec3_rubric_signature(bp)
        cache_dir = STAGEC3_DIR / sig / chapter_id
        cache_dir.mkdir(parents=True, exist_ok=True)
        doc_sig = hashlib.sha1(str(doc_id).encode("utf-8")).hexdigest()[:16]
        cache_path = cache_dir / f"{doc_sig}.json"

        if cache_path.exists() and not STAGEC3_FORCE_RERANK:
            payload = json.loads(cache_path.read_text(encoding="utf-8"))
            payload["_meta"] = payload.get("_meta") or {}
            payload["_meta"]["llm_cached"] = True
            payload["_meta"]["requests"] = 0
            payload["_meta"]["input_tokens"] = 0
            payload["_meta"]["cached_input_tokens"] = 0
            payload["_meta"]["output_tokens"] = 0
            payload["_meta"]["cost_usd"] = 0.0
            return payload

        prompt = _stagec3_prompt(bp, title=title, abstract=abstract)
        last_err = None
        for attempt in range(int(STAGEC3_MAX_RETRIES)):
            try:
                async with sem:
                    res = await Runner.run(stagec3_agent, prompt)
                usage = getattr(getattr(res, "context_wrapper", None), "usage", None)
                meta = cost_from_usage(usage, model=STAGEC3_MODEL) if usage is not None else {
                    "requests": 1,
                    "input_tokens": 0,
                    "cached_input_tokens": 0,
                    "output_tokens": 0,
                    "cost_usd": 0.0,
                }
                out = {
                    "label": str(res.final_output.label),
                    "confidence": int(res.final_output.confidence),
                    "score": int(res.final_output.score),
                    "notes": str(res.final_output.notes or ""),
                    "_meta": {**meta, "llm_cached": False, "rubric_sig": sig, "model": STAGEC3_MODEL},
                }
                cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
                return out
            except Exception as e:
                last_err = e
                backoff = min(float(STAGEC3_BACKOFF_MAX), float(STAGEC3_BACKOFF_INITIAL) * (2 ** attempt))
                backoff = backoff * (0.75 + 0.5 * random.random())
                await asyncio.sleep(backoff)

        raise RuntimeError(
            f"Stage C.3 rerank failed after retries for chapter_id={chapter_id} doc_id={doc_id}: {last_err}"
        )

    async def _wrapped(ix: int, chapter_id: str, doc_id: str, title: str, abstract: str) -> Tuple[int, dict]:
        payload = await _run_one(chapter_id, doc_id, title, abstract)
        return ix, payload

    # Ranked candidate indices per chapter (by Stage C score)
    ranked: Dict[str, list[int]] = {}
    desired: Dict[str, int] = {}

    for chapter_id, g in df.groupby("chapter_id"):
        cid = str(chapter_id)
        if cid not in blueprints_by_chapter_id:
            raise KeyError(f"Missing blueprint for chapter_id '{cid}'")

        sort_cols = ["score_stageC_final"]
        asc = [False]
        if id_col in g.columns:
            sort_cols.append(id_col)
            asc.append(True)

        ranked[cid] = g.sort_values(sort_cols, ascending=asc, kind="mergesort").index.to_list()
        desired[cid] = int(min(int(topn), len(ranked[cid])))

    # Iteratively score + expand
    enqueued: set[int] = set()

    def _enqueue_upto(cid: str, n: int) -> list[asyncio.Task]:
        tasks_local: list[asyncio.Task] = []
        for ix in ranked[cid][: int(n)]:
            if ix in enqueued:
                continue
            row = df.loc[ix]
            doc_id = str(row[id_col])
            title0 = str(row.get("title") or "")
            abstract0 = str(row.get("abstract") or "")
            tasks_local.append(asyncio.create_task(_wrapped(ix, cid, doc_id, title0, abstract0)))
            enqueued.add(ix)
        return tasks_local

    # Initial enqueue
    tasks: list[asyncio.Task] = []
    for cid, n in desired.items():
        tasks.extend(_enqueue_upto(cid, n))

    round_i = 0
    while tasks:
        round_i += 1
        print(
            f"[StageC3] Round {round_i}: scoring {len(tasks)} docs | model={STAGEC3_MODEL} | concurrency={STAGEC3_CONCURRENCY}"
        )

        iterator = asyncio.as_completed(tasks)
        pbar = None
        if tqdm is not None:
            pbar = tqdm(iterator, total=len(tasks), desc=f"Stage C.3 rerank ({STAGEC3_MODEL})")
            iterator = pbar

        done = 0
        for fut in iterator:
            ix, payload = await fut
            done += 1

            df.loc[ix, "score_llm_rerank_v1"] = payload.get("score")
            df.loc[ix, "llm_notes"] = payload.get("notes", "")
            df.loc[ix, "llm_label"] = payload.get("label", "")
            df.loc[ix, "llm_confidence"] = payload.get("confidence")

            m = payload.get("_meta") or {}
            totals["requests"] += int(m.get("requests", 0) or 0)
            totals["input_tokens"] += int(m.get("input_tokens", 0) or 0)
            totals["cached_input_tokens"] += int(m.get("cached_input_tokens", 0) or 0)
            totals["output_tokens"] += int(m.get("output_tokens", 0) or 0)
            totals["cost_usd"] += float(m.get("cost_usd", 0.0) or 0.0)
            totals["cached_files"] += int(bool(m.get("llm_cached", False)))

            if pbar is None and (done % 25 == 0 or done == len(tasks)):
                elapsed = float(time.time() - t0)
                print(
                    f"[StageC3] {done}/{len(tasks)} | req={totals['requests']} | cached={totals['cached_files']} | cost=${totals['cost_usd']:.4f} | sec={elapsed:.0f}"
                )

        # Decide whether to expand any chapter
        expanded_any = False
        for cid, n in list(desired.items()):
            idx = ranked[cid][: int(n)]
            labels = df.loc[idx, "llm_label"].fillna("").astype(str)
            non_ex = int((labels.ne("") & ~labels.eq("exclude")).sum())

            # store latest desired
            totals["topn_used_by_chapter"][cid] = int(n)

            if non_ex >= int(min_non_exclude):
                continue

            cap = int(min(int(topn_max), len(ranked[cid])))
            if int(n) >= cap:
                continue

            new_n = int(min(cap, int(n) + int(topn_step)))
            desired[cid] = new_n
            expanded_any = True
            print(
                f"[StageC3] Expanding chapter '{cid}': non_exclude={non_ex} < {min_non_exclude} → topn {n}→{new_n}"
            )

        if not expanded_any:
            break

        # Enqueue only newly added indices
        tasks = []
        for cid, n in desired.items():
            tasks.extend(_enqueue_upto(cid, n))

    totals["seconds"] = float(time.time() - t0)

    # Mark which docs were included in the scored shortlist (per chapter)
    df["_in_topn"] = False
    for cid, n in desired.items():
        df.loc[ranked[cid][: int(n)], "_in_topn"] = True

    # Final Stage C.3 score: only promote items that are non-exclude and pass confidence gating
    df["score_stageC3_topn_final"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    cite_ok = "score_cite_norm" in df.columns
    if not cite_ok:
        df["score_cite_norm"] = 0.0

    for _, g in df[df["_in_topn"]].groupby("chapter_id"):
        llm_raw = pd.to_numeric(g["score_llm_rerank_v1"], errors="coerce").fillna(0.0)
        labels = df.loc[g.index, "llm_label"].fillna("").astype(str)
        conf = pd.to_numeric(df.loc[g.index, "llm_confidence"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        label_rank = labels.map({"exclude": 0, "maybe": 1, "include": 2}).fillna(0).astype(int).to_numpy(dtype=int)
        label_rank = np.where(conf >= float(STAGEC3_CONFIDENCE_MIN), label_rank, 0)
        llm_raw = llm_raw.where(label_rank > 0, 0.0)
        llm_n = _minmax_series(llm_raw).to_numpy(dtype=float)

        stagec = pd.to_numeric(g["score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(g["score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

        top = pd.DataFrame({
            "ix": g.index.to_list(),
            "doc_id": df.loc[g.index, id_col].fillna("").astype(str).to_list(),
            "label_rank": label_rank,
            "llm_n": llm_n,
            "stagec": stagec,
            "cite": cite,
        })
        top = top.sort_values(["label_rank", "llm_n", "stagec", "cite", "doc_id"], ascending=[False, False, False, False, True], kind="mergesort")
        boosted = top[top["label_rank"] > 0].copy()
        for r, ix in enumerate(boosted["ix"].tolist()):
            df.loc[ix, "score_stageC3_topn_final"] = 2.0 - (r * 1e-6)

    if not cite_ok:
        df = df.drop(columns=["score_cite_norm"], errors="ignore")

    print("\nStage C.3 totals:")
    print_kv({
        'seconds': f"{float(totals.get('seconds',0.0)):.1f}",
        'requests': _fmt_int(totals.get('requests')),
        'cached_files': _fmt_int(totals.get('cached_files')),
        'input_tokens': _fmt_int(totals.get('input_tokens')),
        'cached_input_tokens': _fmt_int(totals.get('cached_input_tokens')),
        'output_tokens': _fmt_int(totals.get('output_tokens')),
        'cost_usd': _fmt_usd(totals.get('cost_usd')),
    })

    try:
        rows = [ {'chapter_id': k, 'topn_used': str(v)} for k, v in (totals.get('topn_used_by_chapter') or {}).items() ]
        if rows:
            print("\nTop-N used by chapter:")
            print_table(rows, columns=['chapter_id','topn_used'], max_rows=200)

            # Per-chapter label distribution inside the scored shortlist
            try:
                rows2 = []
                for chapter_id, gg in df[df.get('_in_topn', False)].groupby('chapter_id'):
                    lab = gg.get('llm_label', '').fillna('').astype(str)
                    conf = pd.to_numeric(gg.get('llm_confidence', 0), errors='coerce').fillna(0.0)
                    rows2.append({
                        'chapter_id': str(chapter_id),
                        'topn_used': str((totals.get('topn_used_by_chapter') or {}).get(str(chapter_id), '')),
                        'include': str(int((lab == 'include').sum())),
                        'maybe': str(int((lab == 'maybe').sum())),
                        'exclude': str(int((lab == 'exclude').sum())),
                        'blank': str(int((lab == '').sum())),
                        'avg_conf': f"{float(conf.mean() if len(conf) else 0.0):.1f}",
                    })

                if rows2:
                    print("\nPer-chapter label breakdown (in_topn):")
                    print_table(rows2, columns=['chapter_id','topn_used','include','maybe','exclude','blank','avg_conf'], max_rows=200)
            except Exception:
                pass

    except Exception:
        pass

    return df, totals

print("Stage C.3 finalized params:")
print("- STAGEC3_TOPN:", STAGEC3_TOPN)
print("- STAGEC3_MODEL:", STAGEC3_MODEL)
print("- STAGEC3_PROMPT_VERSION:", STAGEC3_PROMPT_VERSION)


Stage C.3 finalized params:
- STAGEC3_TOPN: 50
- STAGEC3_MODEL: gpt-5-nano
- STAGEC3_PROMPT_VERSION: v2


## Stage D (finalized): MMR TF-IDF selection (diverse top-20)

Purpose: reduce redundancy in the final list while preserving (and in our benchmark improving) relevance.

Final settings (from fully adjudicated benchmark):
- `topm=100`, `lambda=0.6`, `k_select=20`
- pool ordered by `score_stageC3_topn_final`
- selection uses MMR on TF-IDF similarity, then ranks selected items by `score_stageC3_signal_v1`.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

STAGED_ENABLED = True
STAGED_K_SELECT = 20
STAGED_TOPM = 100
STAGED_LAMBDA = 0.6
STAGED_TFIDF_MAX_FEATURES = 20_000


def _build_text(title: str, abstract: str) -> str:
    t = str(title or "").strip()
    a = str(abstract or "").strip()
    return (t + ". " + a).strip()


def _minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def add_stagec3_signal_v1(
    df_in: pd.DataFrame,
    *,
    topn: int = STAGEC3_TOPN,
    stagec_col: str = "score_stageC_final",
    llm_col: str = "score_llm_rerank_v1",
    out_col: str = "score_stageC3_signal_v1",
    authority_col: str = "score_cite_norm",
    authority_gamma: float = 0.08,
) -> pd.DataFrame:
    """Adds a relevance signal for Stage D.

    - For items inside Stage C top-N: `1 + minmax(llm_score) + authority_gamma*minmax(authority)` (label/confidence gated)
    - For the tail: `score_stageC_final`
    """
    df = df_in.copy()
    df[out_col] = pd.to_numeric(df[stagec_col], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        if "_in_topn" in g.columns:
            idx = g[g["_in_topn"].fillna(False)].index
            if len(idx) == 0:
                idx = g.sort_values(stagec_col, ascending=False, kind="mergesort").head(int(topn)).index
        else:
            idx = g.sort_values(stagec_col, ascending=False, kind="mergesort").head(int(topn)).index
        llm_raw = pd.to_numeric(df.loc[idx, llm_col], errors="coerce").fillna(0.0)
        gate = pd.Series(True, index=idx)
        if "llm_label" in df.columns:
            labels = df.loc[idx, "llm_label"].fillna("").astype(str)
            gate = gate & ~labels.eq("exclude")
        if "llm_confidence" in df.columns:
            conf = pd.to_numeric(df.loc[idx, "llm_confidence"], errors="coerce").fillna(0.0)
            conf_min = float(globals().get("STAGEC3_CONFIDENCE_MIN", 0) or 0)
            if conf_min > 0:
                gate = gate & (conf >= conf_min)

        valid_idx = idx[gate.to_numpy(dtype=bool)]
        if len(valid_idx):
            llm_n_valid = _minmax_series(llm_raw.loc[valid_idx])
            signal = 1.0 + llm_n_valid
            if float(authority_gamma) and (authority_col in df.columns):
                auth = pd.to_numeric(df.loc[valid_idx, authority_col], errors="coerce").fillna(0.0)
                auth_n = _minmax_series(auth)
                signal = signal + float(authority_gamma) * auth_n
            df.loc[valid_idx, out_col] = signal
    return df


def mmr_select(relevance: np.ndarray, sim: np.ndarray, k: int, lam: float) -> List[int]:
    n = int(len(relevance))
    if n == 0:
        return []
    k = int(min(k, n))

    chosen: List[int] = []
    remaining = list(range(n))

    first = int(np.argmax(relevance))
    chosen.append(first)
    remaining.remove(first)

    eps = 1e-12
    while len(chosen) < k and remaining:
        best_i = None
        best_score = -1e18
        for i in remaining:
            max_sim = max(float(sim[i, j]) for j in chosen) if chosen else 0.0
            s = float(lam) * float(relevance[i]) - (1.0 - float(lam)) * float(max_sim)
            if (s > best_score + eps) or (abs(s - best_score) <= eps and (best_i is None or int(i) < int(best_i))):
                best_score = s
                best_i = int(i)
        chosen.append(int(best_i))
        remaining.remove(int(best_i))
    return chosen


def add_stageD_mmr_tfidf_v2(
    df_in: pd.DataFrame,
    *,
    baseline_col: str = "score_stageC3_topn_final",
    relevance_col: str = "score_stageC3_signal_v1",
    title_col: str = "title",
    abstract_col: str = "abstract",
    id_col: str = "merge_key",
    topm: int = STAGED_TOPM,
    k_select: int = STAGED_K_SELECT,
    lam: float = STAGED_LAMBDA,
    out_col: str = "score_stageD_final",
) -> pd.DataFrame:
    """Stage D: select a diverse top-K using TF-IDF MMR.

    Output behavior:
    - Selected docs get scores `3.0 - r*1e-6` (strict ordering).
    - All other docs keep `baseline_col` scores.
    """
    df = df_in.copy()
    required = ["chapter_id", baseline_col, relevance_col, title_col, abstract_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for Stage D: {missing}")

    df[out_col] = pd.to_numeric(df[baseline_col], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        g_sorted = g.sort_values([baseline_col, id_col], ascending=[False, True], kind="mergesort")
        pool = g_sorted.copy()
        # Prefer selecting only from the docs that Stage C.3 actually scored (avoids unlabeled tail docs).
        if "_in_topn" in pool.columns:
            pool = pool[pool["_in_topn"].fillna(False)].copy()
        if 'llm_label' in pool.columns:
            pool = pool[pool['llm_label'].fillna('').astype(str).ne('')].copy()
            pool = pool[~pool['llm_label'].fillna('').astype(str).eq('exclude')].copy()
        if int(topm) > 0 and len(pool) > int(topm):
            pool = pool.sort_values([baseline_col, id_col], ascending=[False, True], kind="mergesort").head(int(topm)).copy()
        if len(pool) == 0:
            print(f"[StageD] Warning: empty pool for {cid}; skipping Stage D for this chapter.")
            continue
        if len(pool) < int(k_select):
            print(f"[StageD] Warning: pool<{k_select} for {cid}: pool={len(pool)}. Consider increasing Stage C.3 topn/topn_max.")
        texts = [_build_text(t, a) for t, a in zip(pool[title_col].fillna(""), pool[abstract_col].fillna(""))]
        tfidf = TfidfVectorizer(max_features=int(STAGED_TFIDF_MAX_FEATURES))
        X = tfidf.fit_transform(texts)
        S = cosine_similarity(X)

        rel = pd.to_numeric(pool[relevance_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        chosen_local = mmr_select(rel, S, k=int(k_select), lam=float(lam))
        chosen_idx = pool.iloc[chosen_local].index.tolist()

        chosen_sorted = (
            pool.loc[chosen_idx]
            .sort_values([relevance_col, baseline_col, id_col], ascending=[False, False, True], kind="mergesort")
            .index.tolist()
        )
        for r, ix in enumerate(chosen_sorted):
            df.loc[ix, out_col] = 3.0 - (r * 1e-6)

    return df


print("Stage D finalized params:")
print("- enabled:", STAGED_ENABLED)
print("- k_select:", STAGED_K_SELECT)
print("- topm:", STAGED_TOPM)
print("- lambda:", STAGED_LAMBDA)


print_section("Stage D config")
print_kv({
    "enabled": STAGED_ENABLED,
    "k_select": STAGED_K_SELECT,
    "topm": STAGED_TOPM,
    "lambda": STAGED_LAMBDA,
    "tfidf_max_features": STAGED_TFIDF_MAX_FEATURES,
})


Stage D finalized params:
- enabled: True
- k_select: 20
- topm: 100
- lambda: 0.6

Stage D config
enabled            : True
k_select           : 20
topm               : 100
lambda             : 0.6
tfidf_max_features : 20000


## Final run (production)

Runs Stage C.3 (LLM rerank within top‑N) and Stage D (MMR TF‑IDF) and writes outputs to `final_pipeline/results/`.
After you `Run All`, paste the printed JSON summary back here.


In [8]:

# -----------------------------
# Final run: Stage C.3 + Stage D + exports
# -----------------------------

import json
import hashlib
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display  # type: ignore
except Exception:  # pragma: no cover
    def display(x):  # type: ignore
        print(x)


def _latest(path_glob):
    cands = sorted(path_glob, key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0] if cands else None


# Load Stage C pool if needed
if "stageC_all" not in globals() or not isinstance(stageC_all, pd.DataFrame) or stageC_all.empty:
    p = _latest(list(STAGEC_DIR.glob("*_stageC_all_pool_scored.csv")))
    if p is None:
        raise FileNotFoundError("No Stage C combined CSV found under final_pipeline/stageC/. Run the Stage C cell first.")
    stageC_all = pd.read_csv(p)
    print("Loaded Stage C combined:", p, "| rows=", len(stageC_all))
else:
    print("Using in-memory stageC_all | rows=", len(stageC_all))


# Stage C.3 (API calls; cached to disk)
stageC3_df, stageC3_totals = await stagec3_rerank_topn(
    stageC_all,
    blueprints_by_chapter_id=blueprints,
    min_non_exclude=int(STAGED_K_SELECT),
)


# Stage D (offline)
final_score_col = "score_stageC3_topn_final"
if STAGED_ENABLED:
    stageC3_df = add_stagec3_signal_v1(stageC3_df)
    stageC3_df = add_stageD_mmr_tfidf_v2(stageC3_df)
    final_score_col = "score_stageD_final"


# Save full scored output
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
full_scored_csv = RESULTS_DIR / f"{RUN_ID}_full_scored.csv"
stageC3_df.to_csv(full_scored_csv, index=False, encoding="utf-8")
print("Saved full scored CSV:", full_scored_csv)

print_section("OpenAI cost summary")
parts = []
total_cost = 0.0
for name, d in [("stageB_blueprints", globals().get("bp_totals")), ("embeddings", globals().get("embed_totals")), ("stageC3_rerank", globals().get("stageC3_totals"))]:
    if isinstance(d, dict):
        c = float(d.get("cost_usd", 0.0) or 0.0)
        total_cost += c
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>if parts:
    print_table(parts, columns=["stage","requests","in_tok","cached_in","out_tok","cost_usd"], max_rows=20)
print("Total OpenAI cost (estimated):", _fmt_usd(total_cost))


# -----------------------------
# Query attribution (which retrieval queries produced the winners)
# -----------------------------
PRINT_QUERY_ATTRIBUTION = True
QUERY_ATTRIBUTION_MAX_ROWS = 120


def _split_queries_field(x: object) -> list[str]:
    s = '' if x is None else str(x)
    parts = [p.strip() for p in s.split(';')]
    parts = [p for p in parts if p]
    # dedupe (case-insensitive), preserve order
    out: list[str] = []
    seen: set[str] = set()
    for q in parts:
        k = q.lower()
        if k in seen:
            continue
        seen.add(k)
        out.append(q)
    return out


def _query_maps_for_chapter(chapter_id: str) -> tuple[dict[str, dict], dict[str, str]]:
    qmeta_by_id: dict[str, dict] = {}
    qid_by_query_text: dict[str, str] = {}
    try:
        plan = query_plans_by_chapter.get(chapter_id) if isinstance(query_plans_by_chapter, dict) else None
        if plan is None:
            return qmeta_by_id, qid_by_query_text
        specs = plan.queries if hasattr(plan, "queries") else (plan.get("queries") or [])
        for q in specs:
            qd = q.model_dump() if hasattr(q, "model_dump") else dict(q)
            qid = str(qd.get("id") or qd.get("query_id") or "").strip()
            qtxt = str(qd.get("query") or qd.get("query_string") or "").strip()
            if not qid:
                continue
            qmeta_by_id[qid] = qd
            if qtxt:
                qid_by_query_text[qtxt.lower()] = qid
    except Exception:
        pass
    return qmeta_by_id, qid_by_query_text


def _row_query_ids(row: pd.Series, *, qid_by_query_text: dict[str, str]) -> list[str]:
    qids = _split_queries_field(row.get("query_ids", ""))
    if qids:
        return qids
    out: list[str] = []
    for q in _split_queries_field(row.get("queries", "")):
        qid = qid_by_query_text.get(str(q).lower())
        out.append(qid or q)
    return _split_queries_field("; ".join(out))


def _add_query_attribution_cols(df_in: pd.DataFrame, *, qmeta_by_id: dict[str, dict], qid_by_query_text: dict[str, str]) -> pd.DataFrame:
    df = df_in.copy()
    if 'queries' not in df.columns:
        df['queries'] = ''
    if 'query_ids' not in df.columns:
        df['query_ids'] = ''

    qid_lists = [ _row_query_ids(row, qid_by_query_text=qid_by_query_text) for _, row in df.iterrows() ]
    df['query_ids_norm'] = ["; ".join(xs) for xs in qid_lists]
    df['query_hit_count'] = [int(len(xs)) for xs in qid_lists]

    def kinds(xs: list[str]) -> str:
        vals = []
        for qid in xs:
            qd = qmeta_by_id.get(qid) or {}
            k = str(qd.get("kind") or "").strip()
            if k:
                vals.append(k)
        return "; ".join(sorted(set(vals)))

    def svcs(xs: list[str]) -> str:
        vals = []
        for qid in xs:
            qd = qmeta_by_id.get(qid) or {}
            s = str(qd.get("service") or "").strip()
            if s:
                vals.append(s)
        return "; ".join(sorted(set(vals)))

    def langs(xs: list[str]) -> str:
        vals = []
        for qid in xs:
            qd = qmeta_by_id.get(qid) or {}
            s = str(qd.get("language") or "").strip()
            if s:
                vals.append(s)
        return "; ".join(sorted(set(vals)))

    df['query_kinds_hit'] = [kinds(xs) for xs in qid_lists]
    df['query_services_hit'] = [svcs(xs) for xs in qid_lists]
    df['query_languages_hit'] = [langs(xs) for xs in qid_lists]

    def origin(xs: list[str]) -> str:
        if not xs:
            return "missing"
        services = sorted(set([str((qmeta_by_id.get(qid) or {}).get("service") or "") for qid in xs if qid in qmeta_by_id]))
        if not services:
            return "unknown"
        if len(services) == 1:
            return services[0]
        return "multi_service"

    df['query_origin'] = [origin(xs) for xs in qid_lists]
    return df


def _query_productivity_rows(stage_df: pd.DataFrame, top_df: pd.DataFrame, *, qmeta_by_id: dict[str, dict], qid_by_query_text: dict[str, str]) -> list[dict]:
    if len(top_df) == 0 or len(stage_df) == 0:
        return []

    # Build doc -> query_ids and query_id -> docs maps.
    doc_to_qids: dict[str, list[str]] = {}
    qid_to_docs: dict[str, set[str]] = {}
    merge_col = "merge_key" if "merge_key" in stage_df.columns else None
    if merge_col is None:
        stage_df = stage_df.reset_index().rename(columns={"index": "__row_idx"})
        merge_col = "__row_idx"
    for _, row in stage_df.iterrows():
        doc_id = str(row.get(merge_col))
        qids = _row_query_ids(row, qid_by_query_text=qid_by_query_text)
        doc_to_qids[doc_id] = qids
        for qid in qids:
            qid_to_docs.setdefault(qid, set()).add(doc_id)

    # top-30 contribution per query
    top_doc_ids = [str(x) for x in top_df.get(merge_col, pd.Series([], dtype=object)).tolist()]
    final_contrib: dict[str, int] = {}
    for doc_id in top_doc_ids:
        for qid in doc_to_qids.get(doc_id, []):
            final_contrib[qid] = final_contrib.get(qid, 0) + 1

    # unique hits (docs only retrieved by one query)
    unique_hits: dict[str, int] = {}
    for doc_id, qids in doc_to_qids.items():
        uq = [q for q in qids if q]
        if len(uq) == 1:
            q = uq[0]
            unique_hits[q] = unique_hits.get(q, 0) + 1

    # llm stats
    llm_label_map = {}
    llm_score_map = {}
    if "llm_label" in stage_df.columns:
        llm_label_map = dict(zip(stage_df[merge_col].astype(str), stage_df["llm_label"].fillna("").astype(str)))
    if "score_llm_rerank_v1" in stage_df.columns:
        llm_score_map = dict(zip(stage_df[merge_col].astype(str), pd.to_numeric(stage_df["score_llm_rerank_v1"], errors="coerce")))

    rows: list[dict] = []
    for qid, docs in qid_to_docs.items():
        docs_list = list(docs)
        labels = [llm_label_map.get(d, "") for d in docs_list if d in llm_label_map]
        label_nonblank = [x for x in labels if x]
        include_maybe = sum(1 for x in label_nonblank if x in ("include", "maybe"))
        include_rate = (float(include_maybe) / float(len(label_nonblank))) if label_nonblank else 0.0
        scores = [float(llm_score_map[d]) for d in docs_list if d in llm_score_map and pd.notna(llm_score_map[d])]
        mean_llm = float(sum(scores) / len(scores)) if scores else float("nan")
        qd = qmeta_by_id.get(qid) or {}
        rows.append({
            "query_id": qid,
            "service": str(qd.get("service") or ""),
            "kind": str(qd.get("kind") or ""),
            "language": str(qd.get("language") or ""),
            "cap": int(qd.get("cap", 0) or 0),
            "use_no_year": bool(qd.get("use_no_year", False)),
            "query": str(qd.get("query") or qd.get("query_string") or qid),
            "hits_top30": int(final_contrib.get(qid, 0)),
            "final_selection_contrib": int(final_contrib.get(qid, 0)),
            "hits_stageC3": int(len(docs_list)),
            "unique_hits": int(unique_hits.get(qid, 0)),
            "include_rate": include_rate,
            "mean_llm_score": mean_llm,
        })

    rows.sort(key=lambda r: (-int(r.get("hits_top30", 0)), -int(r.get("unique_hits", 0)), str(r.get("query_id", ""))))
    return rows


# Save per-chapter top30
top30_paths = {}
cols_show = [
    "chapter_id",
    "title",
    "year",
    "venue",
    "doi_norm",
    "sources",
    "queries",
    "query_ids",
    "query_kinds",
    "query_services",
    "query_languages",
    "query_ids_norm",
    "query_kinds_hit",
    "query_services_hit",
    "query_languages_hit",
    "query_origin",
    "query_hit_count",
    "score_stageC_final",
    "score_llm_rerank_v1",
    "llm_label",
    "llm_confidence",
    "llm_notes",
    final_score_col,
    "merge_key",
]

for cid, g in stageC3_df.groupby("chapter_id"):
    out_csv = RESULTS_DIR / f"{RUN_ID}_{cid}_top30.csv"
    top = g.sort_values(final_score_col, ascending=False, kind="mergesort").copy()
    if "llm_label" in top.columns:
        top = top[~top["llm_label"].fillna("").astype(str).eq("exclude")].copy()
    top = top.head(30).copy()

    # Query attribution: which Stage A queries retrieved these items?
    qmeta_by_id, qid_by_query_text = _query_maps_for_chapter(str(cid))
    top = _add_query_attribution_cols(top, qmeta_by_id=qmeta_by_id, qid_by_query_text=qid_by_query_text)
    keep = [c for c in cols_show if c in top.columns]
    top[keep].to_csv(out_csv, index=False, encoding="utf-8")
    top30_paths[str(cid)] = str(out_csv)
    print(f"[{cid}] saved top30: {out_csv}")
    display(top[keep].head(10))


    if bool(PRINT_QUERY_ATTRIBUTION):
        print_section(f"Query attribution [{cid}] (top-30)")

        # Origin quota by service footprint
        try:
            vc = top.get('query_origin', pd.Series([], dtype=str)).fillna('missing').astype(str).value_counts().to_dict()
            rows_o = []
            for k, v in vc.items():
                rows_o.append({'origin': k, 'count': int(v), 'share': f"{100.0*float(v)/float(len(top) or 1):.1f}%"})
            print("Origin quota:")
            print_table(rows_o, columns=['origin','count','share'], max_rows=20)
        except Exception:
            pass

        # Productivity per query
        stage_all_ch = stageC3_df[stageC3_df["chapter_id"].astype(str).eq(str(cid))].copy()
        rows_q = _query_productivity_rows(stage_all_ch, top, qmeta_by_id=qmeta_by_id, qid_by_query_text=qid_by_query_text)
        if rows_q:
            out_q_csv = RESULTS_DIR / f"{RUN_ID}_{cid}_query_productivity_top30.csv"
            try:
                pd.DataFrame(rows_q).to_csv(out_q_csv, index=False, encoding="utf-8")
                print("Saved query productivity CSV:", out_q_csv)
            except Exception:
                pass
        if rows_q:
            print("\nQuery productivity (top-30):")
            rows_q2 = [
                {
                    **r,
                    'hits_top30': _fmt_int(r.get('hits_top30', 0)),
                    'hits_stageC3': _fmt_int(r.get('hits_stageC3', 0)),
                    'unique_hits': _fmt_int(r.get('unique_hits', 0)),
                    'include_rate': _fmt_pct(r.get('include_rate', 0.0)),
                    'mean_llm_score': (f"{float(r.get('mean_llm_score')):.1f}" if pd.notna(r.get('mean_llm_score')) else ""),
                }
                for r in rows_q
            ]
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen>

# Save global top30 across all chapters
global_top30_csv = RESULTS_DIR / f"{RUN_ID}_top30_all_chapters.csv"
gall = stageC3_df.sort_values(final_score_col, ascending=False, kind="mergesort").copy()
if "llm_label" in gall.columns:
    gall = gall[~gall["llm_label"].fillna("").astype(str).eq("exclude")].copy()
g30 = gall.head(30).copy()
keep_global = [c for c in cols_show if c in g30.columns]
g30[keep_global].to_csv(global_top30_csv, index=False, encoding="utf-8")
print("\nSaved global top30:", global_top30_csv)
display(g30[keep_global].head(30))

# Final plots (debug)
print_section("Plots")
plt = get_plt()
if plt and bool(PLOTS_ENABLED):
    try:
        import numpy as np

        # Global top-30 final score by rank
        if 'g30' in globals() and isinstance(g30, pd.DataFrame) and (final_score_col in g30.columns):
            s = pd.to_numeric(g30[final_score_col], errors='coerce').fillna(0.0).to_numpy(dtype=float)
            fig, ax = plt.subplots(1, 1, figsize=(10, 3.2))
            ax.plot(range(1, len(s) + 1), s, marker='o', linewidth=1.5)
            ax.set_title('Global top-30 final score by rank')
            ax.set_xlabel('rank')
            ax.set_ylabel(final_score_col)
            ax.grid(True, alpha=0.25)
            plt.tight_layout()
            plt.show()

        # Chapter composition in global top-30
        if 'g30' in globals() and isinstance(g30, pd.DataFrame) and ('chapter_id' in g30.columns):
            vc = g30['chapter_id'].fillna('').astype(str).value_counts()
            if len(vc):
                fig, ax = plt.subplots(1, 1, figsize=(10, max(2.6, 0.5 * len(vc) + 1.2)))
                ax.barh([short_label(x, 34) for x in vc.index.tolist()], vc.values.tolist(), color='#4C78A8')
                ax.set_title('Global top-30: results per chapter')
                ax.set_xlabel('count')
                plt.tight_layout()
                plt.show()

        # LLM label composition in per-chapter top-30
        if ('llm_label' in stageC3_df.columns) and ('chapter_id' in stageC3_df.columns):
            top_by_ch = stageC3_df.sort_values(final_score_col, ascending=False, kind='mergesort').groupby('chapter_id').head(30).copy()
            top_by_ch['_llm_label'] = top_by_ch['llm_label'].fillna('').astype(str)
            top_by_ch.loc[top_by_ch['_llm_label'].eq(''), '_llm_label'] = '(blank)'

            counts = top_by_ch.pivot_table(index='chapter_id', columns='_llm_label', values=final_score_col, aggfunc='size', fill_value=0)
            col_order = ['include', 'maybe', 'exclude', '(blank)']
            cols = [c for c in col_order if c in counts.columns] + [c for c in counts.columns if c not in col_order]
            counts = counts[cols]

            fig, ax = plt.subplots(1, 1, figsize=(10, max(3.0, 0.55 * len(counts) + 1.6)))
            left = np.zeros(len(counts), dtype=float)
            colors = {'include': '#54A24B', 'maybe': '#F58518', 'exclude': '#E45756', '(blank)': '#B8B0AC'}
            ylabels = [short_label(x, 34) for x in counts.index.tolist()]

            for col in cols:
                vals = counts[col].to_numpy(dtype=float)
                ax.barh(ylabels, vals, left=left, label=col, color=colors.get(col))
                left = left + vals

            ax.set_title('LLM labels in top-30 per chapter')
            ax.set_xlabel('count')
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()

        # Authority vs final score in global top-30
        if 'g30' in globals() and isinstance(g30, pd.DataFrame) and ('score_cite_norm' in g30.columns) and (final_score_col in g30.columns):
            x = pd.to_numeric(g30['score_cite_norm'], errors='coerce').fillna(0.0)
            y = pd.to_numeric(g30[final_score_col], errors='coerce').fillna(0.0)
            fig, ax = plt.subplots(1, 1, figsize=(6.2, 4.6))
            ax.scatter(x, y, s=40, alpha=0.8, color='#4C78A8')
            ax.set_title('Authority vs final score (global top-30)')
            ax.set_xlabel('score_cite_norm')
            ax.set_ylabel(final_score_col)
            ax.grid(True, alpha=0.25)
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print('[plots] Final plots skipped:', e)


# Build a concise run summary for copy/paste
stagea_paths = {}
for ch in CHAPTERS:
    cid = ch["chapter_id"]
    bp = blueprints[cid]
    # Recompute the Stage A signature (must match Stage A cell logic)
    try:
        plan = query_plans_by_chapter.get(cid) if isinstance(query_plans_by_chapter, dict) else None
        if plan is None:
            raise KeyError(f"Missing query plan for chapter {cid}")
        plan_dump = plan.model_dump() if hasattr(plan, "model_dump") else plan
        concept_terms = openalex_concept_terms_for_chapter(bp)
    except Exception:
        plan_dump = {}
        concept_terms = []

    # NOTE: This signature must match Stage A cell logic exactly (paths depend on it).
    plan_specs = []
    plan_q_count = 0
    plan_oa_count = 0
    plan_s2_count = 0
    plan_gap_count = 0
    plan_no_year = 0
    if plan is not None:
        try:
            specs0 = plan.queries if hasattr(plan, "queries") else (plan.get("queries") or [])
        except Exception:
            specs0 = []
        plan_specs = [q.model_dump() if hasattr(q, "model_dump") else dict(q) for q in (specs0 or [])]
        plan_q_count = len(plan_specs)
        plan_oa_count = sum(1 for q in plan_specs if str(q.get("service", "")).strip() == "openalex")
        plan_s2_count = sum(1 for q in plan_specs if str(q.get("service", "")).strip() == "semanticscholar")
        plan_gap_count = 0
        plan_no_year = sum(1 for q in plan_specs if bool(q.get("use_no_year", False)))

    try:
        search_key = _openalex_search_filter_key()  # defined in Stage A cell
    except Exception:
        use_no_stem = bool(globals().get("OA_USE_NO_STEM"))
        search_key = globals().get("OA_TITLE_ABSTRACT_FILTER_KEY_NO_STEM" if use_no_stem else "OA_TITLE_ABSTRACT_FILTER_KEY")

    sig_obj = {
        "query_plan": plan.model_dump() if (plan is not None and hasattr(plan, "model_dump")) else (plan_dump if isinstance(plan_dump, dict) else plan_specs),
        "query_plan_counts": {"total": plan_q_count, "openalex": plan_oa_count, "s2": plan_s2_count, "no_year": plan_no_year},
        "OA_MAX_WORKS_PER_QUERY": globals().get("OA_MAX_WORKS_PER_QUERY"),
        "OA_REQUIRE_ABSTRACT": globals().get("OA_REQUIRE_ABSTRACT"),
        "OA_FILTER_LANGUAGES": globals().get("OA_FILTER_LANGUAGES"),
        "OA_FILTER_DATES": globals().get("OA_FILTER_DATES"),
        "OA_SEARCH_KEY": search_key,
        "OA_AUTHORITY_SORT": globals().get("OA_AUTHORITY_SORT"),
        "OA_CONCEPT_FILTER_ENABLED": globals().get("OA_CONCEPT_FILTER_ENABLED"),
        "OA_CONCEPT_FILTER_FOR_OPENALEX": globals().get("OA_CONCEPT_FILTER_FOR_OPENALEX"),
        "OA_CONCEPT_TERMS": concept_terms,
        "OA_CONCEPT_MAX_IDS": globals().get("OA_CONCEPT_MAX_IDS"),
        "OA_CONCEPT_TERMS_MAX": globals().get("OA_CONCEPT_TERMS_MAX"),
        "OA_CONCEPTS_PER_TERM": globals().get("OA_CONCEPTS_PER_TERM"),
        "OA_CONCEPT_LEVEL_MAX": globals().get("OA_CONCEPT_LEVEL_MAX"),
        "OA_CONCEPT_MIN_WORKS": globals().get("OA_CONCEPT_MIN_WORKS"),
        "OA_CONCEPT_FILTER_SHARE": globals().get("OA_CONCEPT_FILTER_SHARE"),
        "OA_CONCEPT_FILTER_ALWAYS_KINDS": sorted(list(globals().get("OA_CONCEPT_FILTER_ALWAYS_KINDS") or [])),
        "S2_REQUIRE_ABSTRACT_AFTER_BATCH": globals().get("S2_REQUIRE_ABSTRACT_AFTER_BATCH"),
        "S2_ABSTRACT_MIN_CHARS": globals().get("S2_ABSTRACT_MIN_CHARS"),
        "S2_FETCH_ABSTRACTS_VIA_BATCH": globals().get("S2_FETCH_ABSTRACTS_VIA_BATCH"),
        "S2_BATCH_SIZE": globals().get("S2_BATCH_SIZE"),
        "S2_LIMIT": globals().get("S2_LIMIT"),
        "S2_MAX_PAGES_PER_QUERY": globals().get("S2_MAX_PAGES_PER_QUERY"),
        "S2_USE_BULK_SEARCH": False,
    }
    sig = hashlib.sha1(json.dumps(sig_obj, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:12]
    stagea_paths[cid] = str((STAGEA_DIR / cid / sig / "stageA_combined.csv").resolve())

summary = {
    "run_id": RUN_ID,
    "chapters": [c["chapter_id"] for c in CHAPTERS],
    "stageB": {"variant": STAGEB_FINAL_VARIANT, "model": BLUEPRINT_MODEL},
    "stageA": {"paths": stagea_paths},
    "stageC": {
        "weights": {
            "w_embed_max": STAGEC_W_EMBED_MAX,
            "w_embed": STAGEC_W_EMBED,
            "cite_weight": STAGEC_CITE_WEIGHT,
        },
        "embed_totals": globals().get("embed_totals", {}),
    },
    "stageC3": {"topn": STAGEC3_TOPN, "model": STAGEC3_MODEL, "totals": stageC3_totals},
    "stageD": {"enabled": STAGED_ENABLED, "topm": STAGED_TOPM, "k_select": STAGED_K_SELECT, "lambda": STAGED_LAMBDA},
    "outputs": {
        "full_scored_csv": str(full_scored_csv),
        "top30_csvs": top30_paths,
        "global_top30_csv": str(globals().get("global_top30_csv", "")),
        "final_score_col": final_score_col,
    },
}

summary_path = RESULTS_DIR / f"{RUN_ID}_run_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")


# -----------------------------
# QA summary (sanity checks)
# -----------------------------
print('\nQA summary (top-30):')
for cid, g in stageC3_df.groupby('chapter_id'):
    top = g.sort_values(final_score_col, ascending=False, kind='mergesort').head(30)
    if 'llm_label' in top.columns:
        counts = top['llm_label'].fillna('').astype(str).value_counts().to_dict()
        print(f'- {cid}: llm_label counts in top30 = {counts}')
        if counts.get('exclude', 0) > 0:
            print(f'  WARNING: excludes present in top30 for {cid}')
        if counts.get('', 0) > 0:
            print(f'  WARNING: blank llm_label present in top30 for {cid} (likely unscored tail)')


Using in-memory stageC_all | rows= 283

Stage C.3 rerank
model           : gpt-5-nano
topn            : 50
topn_max        : 250
min_non_exclude : 20
concurrency     : 8
force_rerank    : False
[StageC3] Round 1: scoring 50 docs | model=gpt-5-nano | concurrency=8


Stage C.3 rerank (gpt-5-nano):   0%|          | 0/50 [00:00<?, ?it/s]


Stage C.3 totals:
seconds             : 128.6
requests            : 50
cached_files        : 0
input_tokens        : 45,761
cached_input_tokens : 0
output_tokens       : 58,145
cost_usd            : $0.0255

Top-N used by chapter:
chapter_id           | topn_used
---------------------+----------
fall_of_rome_economy | 50       

Per-chapter label breakdown (in_topn):
chapter_id           | topn_used | include | maybe | exclude | blank | avg_conf
---------------------+-----------+---------+-------+---------+-------+---------
fall_of_rome_economy | 50        | 13      | 18    | 19      | 0     | 60.6    
Saved full scored CSV: <projektverzeichnis>\sources_workspace\final_pipeline\results\20260210_131329_full_scored.csv

OpenAI cost summary
stage             | requests | in_tok | cached_in | out_tok | cost_usd
------------------+----------+--------+-----------+---------+---------
stageB_blueprints | 4        | 15,463 | 0         | 22,774  | $0.0494 
embeddings        | 6        | 58,900 

,chapter_id,title,year,venue,doi_norm,sources,queries,query_ids,query_kinds,query_services,...,query_languages_hit,query_origin,query_hit_count,score_stageC_final,score_llm_rerank_v1,llm_label,llm_confidence,llm_notes,score_stageD_final,merge_key
67,fall_of_rome_economy,Coins and coin use at the late Roman village o...,2013.0,Relicta Archeologie Monumenten- en Landschapso...,10.55465/smpy7861,openalex,Late Antique North Africa AND Western Roman Em...,Q011,anchor,openalex,...,en,openalex,1,0.594641,88.0,include,78.0,"Strong fit: coin hoards, monetary circulation,...",3.000000,doi:10.55465/smpy7861
81,fall_of_rome_economy,Mints not Mines: a macroscale investigation of...,2023.0,Internet Archaeology,10.11141/ia.61.10,semantic_scholar,Late Roman West Western Roman Empire Roman Imp...,Q051,anchor,semanticscholar,...,en,semanticscholar,1,0.693689,88.0,include,85.0,"Strong alignment with MUST_COVER: coinage, deb...",2.999999,doi:10.11141/ia.61.10
118,fall_of_rome_economy,Imperial Monetary Policy and Social Reaction i...,2018.0,Journal des Économistes et des Études Humaines,10.1515/jeeh-2017-0002,semantic_scholar,Western Roman coin hoards coin hoard chronolog...,Q088,proxy,semanticscholar,...,en,semanticscholar,1,0.706957,88.0,include,85.0,Strong fit: directly addresses Western Roman m...,2.999998,doi:10.1515/jeeh-2017-0002
189,fall_of_rome_economy,Wars within the Frontiers: Archaeologies of Re...,2013.0,None,10.1163/9789004252585_029,openalex,Italy Late Antiquity AND Western Roman Empire ...,Q009; Q011,anchor,openalex,...,en,openalex,2,0.533885,82.0,include,78.0,Strong alignment with Western Roman Empire eco...,2.999997,doi:10.1163/9789004252585_029
20,fall_of_rome_economy,Imperial Transportation and Communication from...,2016.0,UWSpace (University of Waterloo),None,openalex,Western Roman Empire AND Late Antiquity 250-60...,Q006,anchor,openalex,...,en,openalex,1,0.575305,78.0,include,78.0,Fits Western Roman Empire fiscal infrastructur...,2.999996,ty:imperial transportation and communication f...
79,fall_of_rome_economy,At the Edge of Empire: Iron Age and Early Roma...,2012.0,Leicester Research Archive (University of Leic...,None,openalex,Gaul rural economy AND provincial inscriptions...,Q009; Q031,anchor; facet,openalex,...,en,openalex,2,0.696898,78.0,include,85.0,Strong fit for monetary and exchange proxies (...,2.999995,ty:at the edge of empire iron age and early ro...
110,fall_of_rome_economy,TYPICAL FEATURES OF A ROMAN IMPERIAL DENARIUS ...,2020.0,None,10.14795/j.v7i1_si.491,semantic_scholar,Western Roman coin hoards coin hoard chronolog...,Q088,proxy,semanticscholar,...,en,semanticscholar,1,0.598889,78.0,include,70.0,Provides coin hoard evidence and monetary prox...,2.999994,doi:10.14795/j.v7i1_si.491
125,fall_of_rome_economy,Comment on Mints not Mines: a macroscale inves...,2023.0,Internet Archaeology,10.11141/ia.61.10.comment,openalex,coin hoard chronology AND silver content AND m...,Q040,proxy,openalex,...,en,openalex,1,0.613982,78.0,include,72.0,Strong fit: discusses Roman coinage data and m...,2.999993,doi:10.11141/ia.61.10.comment
137,fall_of_rome_economy,A gold coin of the Roman emperor Anthemius (46...,2021.0,Vjesnik Arheološkog muzeja u Zagrebu,10.52064/vamz.54.1.4,openalex,Late Antique North Africa AND Western Roman Em...,Q011,anchor,openalex,...,en,openalex,1,0.643917,78.0,include,85.0,Strong fit: discusses Western Empire coinage (...,2.999992,doi:10.52064/vamz.54.1.4
218,fall_of_rome_economy,The end of the regular coin supply in theCroat...,2021.0,Vjesnik Arheološkog muzeja u Zagrebu,10.52064/vamz.54.1.2,openalex,Late Roman West AND Western Roman Empire AND N...,Q005,anchor,openalex,...,en,openalex,1,0.672126,78.0,include,85.0,Numismatic study of western Danube limes coin ...,2.999991,doi:10.52064/vamz.54.1.2



Query attribution [fall_of_rome_economy] (top-30)
Origin quota:
origin          | count | share
----------------+-------+------
semanticscholar | 16    | 53.3%
openalex        | 14    | 46.7%
Saved query productivity CSV: <projektverzeichnis>\sources_workspace\final_pipeline\results\20260210_131329_fall_of_rome_economy_query_productivity_top30.csv

Query productivity (top-30):
<Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase geladen><Prompt entfernt: wird zur Laufzeit aus Firebase 

,chapter_id,title,year,venue,doi_norm,sources,queries,query_ids,query_kinds,query_services,query_languages,score_stageC_final,score_llm_rerank_v1,llm_label,llm_confidence,llm_notes,score_stageD_final,merge_key
67,fall_of_rome_economy,Coins and coin use at the late Roman village o...,2013.0,Relicta Archeologie Monumenten- en Landschapso...,10.55465/smpy7861,openalex,Late Antique North Africa AND Western Roman Em...,Q011,anchor,openalex,en,0.594641,88.0,include,78.0,"Strong fit: coin hoards, monetary circulation,...",3.000000,doi:10.55465/smpy7861
81,fall_of_rome_economy,Mints not Mines: a macroscale investigation of...,2023.0,Internet Archaeology,10.11141/ia.61.10,semantic_scholar,Late Roman West Western Roman Empire Roman Imp...,Q051,anchor,semanticscholar,en,0.693689,88.0,include,85.0,"Strong alignment with MUST_COVER: coinage, deb...",2.999999,doi:10.11141/ia.61.10
118,fall_of_rome_economy,Imperial Monetary Policy and Social Reaction i...,2018.0,Journal des Économistes et des Études Humaines,10.1515/jeeh-2017-0002,semantic_scholar,Western Roman coin hoards coin hoard chronolog...,Q088,proxy,semanticscholar,en,0.706957,88.0,include,85.0,Strong fit: directly addresses Western Roman m...,2.999998,doi:10.1515/jeeh-2017-0002
189,fall_of_rome_economy,Wars within the Frontiers: Archaeologies of Re...,2013.0,None,10.1163/9789004252585_029,openalex,Italy Late Antiquity AND Western Roman Empire ...,Q009; Q011,anchor,openalex,en,0.533885,82.0,include,78.0,Strong alignment with Western Roman Empire eco...,2.999997,doi:10.1163/9789004252585_029
20,fall_of_rome_economy,Imperial Transportation and Communication from...,2016.0,UWSpace (University of Waterloo),None,openalex,Western Roman Empire AND Late Antiquity 250-60...,Q006,anchor,openalex,en,0.575305,78.0,include,78.0,Fits Western Roman Empire fiscal infrastructur...,2.999996,ty:imperial transportation and communication f...
79,fall_of_rome_economy,At the Edge of Empire: Iron Age and Early Roma...,2012.0,Leicester Research Archive (University of Leic...,None,openalex,Gaul rural economy AND provincial inscriptions...,Q009; Q031,anchor; facet,openalex,en,0.696898,78.0,include,85.0,Strong fit for monetary and exchange proxies (...,2.999995,ty:at the edge of empire iron age and early ro...
110,fall_of_rome_economy,TYPICAL FEATURES OF A ROMAN IMPERIAL DENARIUS ...,2020.0,None,10.14795/j.v7i1_si.491,semantic_scholar,Western Roman coin hoards coin hoard chronolog...,Q088,proxy,semanticscholar,en,0.598889,78.0,include,70.0,Provides coin hoard evidence and monetary prox...,2.999994,doi:10.14795/j.v7i1_si.491
125,fall_of_rome_economy,Comment on Mints not Mines: a macroscale inves...,2023.0,Internet Archaeology,10.11141/ia.61.10.comment,openalex,coin hoard chronology AND silver content AND m...,Q040,proxy,openalex,en,0.613982,78.0,include,72.0,Strong fit: discusses Roman coinage data and m...,2.999993,doi:10.11141/ia.61.10.comment
137,fall_of_rome_economy,A gold coin of the Roman emperor Anthemius (46...,2021.0,Vjesnik Arheološkog muzeja u Zagrebu,10.52064/vamz.54.1.4,openalex,Late Antique North Africa AND Western Roman Em...,Q011,anchor,openalex,en,0.643917,78.0,include,85.0,Strong fit: discusses Western Empire coinage (...,2.999992,doi:10.52064/vamz.54.1.4
218,fall_of_rome_economy,The end of the regular coin supply in theCroat...,2021.0,Vjesnik Arheološkog muzeja u Zagrebu,10.52064/vamz.54.1.2,openalex,Late Roman West AND Western Roman Empire AND N...,Q005,anchor,openalex,en,0.672126,78.0,include,85.0,Numismatic study of western Danube limes coin ...,2.999991,doi:10.52064/vamz.54.1.2



Plots



QA summary (top-30):
- fall_of_rome_economy: llm_label counts in top30 = {'maybe': 14, 'include': 12, 'exclude': 4}
